[Markdown Cell]

# Phase 7 / 05 D Prefix

Use Kaggle **T4 x2**, Internet ON. Attach completed P7_00 prepared data and the matching own Phase 1
foundation, tokenizer and metadata/manifest. For exact continuation attach COMPLETE output of this
SAME Phase 7 task; weights/source/cache journals are required, not just an executed .ipynb.
All required methods start from own Phase 1. A Phase 6 parent is accepted as an automatic teacher
source by notebook 07, never as a substitute for the student's Phase 1 initialization.

[Markdown Cell]

## Score-improvement run: inputs and what gets trained
Attach COMPLETE P7S_00 output, original P7_00 prepared data, and this method's COMPLETE previous
P7 output. The saved own encoder/caches/checkpoints are reused automatically when its ID matches.
With no previous task, attach the matching own Phase 1 foundation/tokenizer; the original required
adaptation trains first. Notebook 07 also supports an existing P7 teacher or supported Phase 6 parent.

This update keeps the already adapted neural weights frozen for a second-stage ranking enhancement.
The shared auxiliary ranker was retrained in P7S_00 on ALL official training pairs. Here each method's
real neural scores are combined with its ranking evidence. Every submitted rule has nonzero neural
contribution. This is not described as a new round of LoRMA/IA3/prompt optimization.
More epochs alone are not the default: your A/C validation already deteriorated before early stopping.

In [1]:
# [Code Cell]
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

requirements = {'numpy': 'numpy>=1.26,<3', 'scipy': 'scipy>=1.10,<2',
    'tokenizers': 'tokenizers>=0.21,<1', 'huggingface_hub': 'huggingface_hub>=0.30,<2',
    'ijson': 'ijson>=3.3,<4', 'tqdm': 'tqdm>=4.66,<5'}
missing = [value for name, value in requirements.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Dependencies ready. Kaggle PyTorch is preserved. Start a fresh session before Run All.')

os.environ.setdefault('OMP_NUM_THREADS', '4')
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.5,<2'])


Dependencies ready. Kaggle PyTorch is preserved. Start a fresh session before Run All.


[Markdown Cell]

## 1. Paths and exact experiment identity

In [2]:
# [Code Cell]
INPUT_ROOT = '/kaggle/input'
EXPERIMENT_ID = 'P7_05_D_Prefix'
FAMILY, METHOD = 'd', 'prefix'
RESUME_RUN_DIR = None
OUTPUT_ROOT = '/kaggle/working/cs781_phase7_score/' + EXPERIMENT_ID
PREPARED_DIR = None
PARENT_RUN_DIR = None  # exact own Phase 1 foundation root, if ambiguous
TOKENIZER_PATH = None
METADATA_PATH = None
FEATURE_CACHE_DIR = None  # optional completed SAME frozen-prefix cache
TEACHER_RUN_DIR = None  # 07: ANY complete supported Phase 6/7 task root, or P7 teacher export
REQUIRE_EXISTING_CHECKPOINT = False
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

# Execution controls: the original query batch, negatives, optimizer and LR plan stay fixed.
T4_CACHE_BATCH = 64
T4_MAX_MICROBATCH = 128
T4_INFERENCE_BATCH = 128
T4_MEMORY_FRACTION = .65  # reserve capacity for optimizer states/prefetched inputs
T4_TUNING_SECONDS = 30
T4_PREFETCH = True

SCORE_DATA_ROOT = None  # root of complete P7S_00 output; auto-discovered if unique


[Markdown Cell]

## 2. Training and efficiency settings

In [3]:
# [Code Cell]
MAX_EPOCHS = 12
TARGET_TOTAL_EPOCHS = None  # absolute E+1 to extend after E completed epochs
TRAINING_MINUTES = 35  # shared launch deadline, including automatic teacher/cache work
CACHE_MINUTES = 35
CHUNK_TOKENS = 256
TOKENS_PER_CHUNK = 4
MEMORY_SLOTS = 128
CACHE_CHUNK_BATCH = 16  # safely halves on frozen-prefix CUDA OOM; retains every token
TRAIN_LAST_UNITS = 2  # C unit = whole Transformer–GRU stage
QUERY_BATCH_PER_RANK = 4
ENCODE_BATCH_SIZE = 16
LEARNING_RATE = 0.0002
HEAD_LEARNING_RATE = 0.0005
MODE = 'train'


[Markdown Cell]

### What learns, and why caching is valid
Every source token is encoded by genuinely frozen own Phase 1 bottom blocks. Each chunk contributes
pooled contextual features, including its last partial chunk. A bounded memory then feeds the own
adapted top blocks. This representation compression can lose details; it never drops source text.
Only frozen features are reused. Adapted layers, prompts, prefixes and the retrieval head execute
with autograd on every optimizer step. C retains FP32 GRUs; D preserves sparse local/dilated/global routes.

A LoRMA learns $(I+alpha BA/r)W_0$. A partial unfreezes its final own blocks. C IA3 rescales attention
K/V and SwiGLU intermediate activations. D prefixes are learned per-layer K/V entries, not input prompt
tokens. B prompt tuning learns decoder input embeddings conditioned on its own frozen encoder memory.
The compact B KD student reuses/prunes its own Phase 1 decoder blocks and learns teacher score distributions.

Normalized query/case vectors use cosine similarity. Multi-positive contrastive training averages
the log-probability of every reachable gold candidate; all other selected nonpositive cases compete
as hard/random/in-batch negatives. Known same-text aliases of positives are not false negatives.
Every train query is visited once in a complete epoch. All split candidate pools remain complete
for indexing and evaluation; minibatch negative selection is a training objective, not a dataset cutoff.
KD adds $lambda T^2 KL(p_{teacher,T} || p_{student,T})$ over aligned candidate IDs, retaining gold supervision.

Each required learned method keeps a positive neural contribution when fusing with BM25. Select
checkpoints and fusion on the tune part of official Dev; freeze them before confirmation/test.
Private test F1 is unmeasured. FP16 AMP, NCCL spawn, full-state 500-update/epoch checkpoints and a
shared ten-hour launch budget cover automatic teacher/cache/training work. Pauses remain recoverable.

[Markdown Cell]

### Materialize complete importable modules
The following small source cells write the reviewed runtime. No remote notebook code is executed.

In [4]:
# [Code Cell]
module_parts = {}


[Markdown Cell]

`__init__.py` — Reviewed source, configuration and ownership verification.

In [5]:
# [Code Cell]
module_parts.setdefault('__init__.py', []).append(r'''"""Own-foundation efficient prior-case retrieval, without external pretrained weights."""
VERSION = 'cs781-phase7-pcr-v1'
''')


[Markdown Cell]

`adaptation.py` — Multiplicative LoRMA, partial layers, actual IA3 K/V/FFN scales, true K/V prefixes and a compact B.

In [6]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''"""Actual LoRMA, partial, prompt, IA3, K/V-prefix and compact student adaptations."""
import math
import types
import torch
from torch import nn
from torch.nn import functional as F


class LoRMA(nn.Module):
    """Frozen W0 plus multiplicative output update: (I + alpha BA/r) W0 x."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.a = nn.Parameter(torch.empty(rank, base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))

    def forward(self, value):
        hidden = F.linear(value, self.base.weight)
        result = hidden + F.linear(F.linear(hidden, self.a), self.b) * self.scale
        return result if self.base.bias is None else result + self.base.bias


class IA3QKV(nn.Module):
    """Q remains unchanged; learned vectors rescale attention K and V channels."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        width = base.out_features // 3
        self.key_scale = nn.Parameter(torch.ones(width))
        self.value_scale = nn.Parameter(torch.ones(width))

    def forward(self, value):
        query, key, val = self.base(value).chunk(3, dim=-1)
        return torch.cat((query, key * self.key_scale, val * self.value_scale), dim=-1)


class IA3FFN(nn.Module):
    """Scale intermediate SwiGLU activations before the frozen down projection."""
''')


In [7]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, value):
        return self.base(value * self.scale)


def prefix_attend(self, query, kv, positions, key_valid=None):
    """Real sparse routes plus always-visible learned per-head prefix K/V.

    Prefixes are attention cache entries, not extra input tokens. They do not
    shift the real tokens' rotary positions or replace local/dilated/global edges.
    Top-layer document memories are bounded, so this exact mask remains small.
    """
    import importlib
    route = importlib.import_module(type(self).__module__).sparse_key_indices
    length = kv[0].shape[2]
    indices, legal = route(positions, length, self.local_window, self.dilation, self.dilated_links, self.global_stride)
    counts = torch.zeros(len(positions), length, dtype=torch.int32, device=query.device)
    counts.scatter_add_(1, indices, legal.to(torch.int32))
    allowed = (counts > 0)[None, None].expand(query.shape[0], 1, -1, -1)
    if key_valid is not None:
        allowed = allowed & key_valid[:, None, None, :]
    prefix = self.prefix_key.shape[1]
    allowed = torch.cat((torch.ones(query.shape[0], 1, len(positions), prefix,
                        dtype=torch.bool, device=query.device), allowed), dim=-1)
    key = torch.cat((self.prefix_key[None].expand(query.shape[0], -1, -1, -1).to(kv[0].dtype), kv[0]), dim=2)
    value = torch.cat((self.prefix_value[None].expand(query.shape[0], -1, -1, -1).to(kv[1].dtype), kv[1]), dim=2)
    output = F.scaled_dot_product_attention(query, key, value, attn_mask=allowed,
        dropout_p=self.dropout if self.training else 0., is_causal=False)
    return self.out(output.transpose(1, 2).contiguous().flatten(2))


def adapt(backbone, cfg):
    backbone.requires_grad_(False)
    changed = []
    if cfg.family == 'b':
        if cfg.method == 'kd':
''')


In [8]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''            # Keep first/last own decoder blocks; strictly load Phase 1 BEFORE
            # pruning. No random replacement decoder or external student weights.
            count = len(backbone.decoder)
            keep = max(1, count // 2)
            width = backbone.shared_embedding.embedding_dim
            head_estimate = 2 * width * cfg.embedding_dim + width * (cfg.prompt_tokens + 10) + 1
            original = list(backbone.decoder)
            # Compact as far as possible while respecting the course's strict
            # 50M lower boundary on the complete own student architecture.
            full_count = sum(value.numel() for value in backbone.parameters())
            while keep < count:
                trial = sorted(set(round(i * (count - 1) / max(1, keep - 1)) for i in range(keep)))
                removed = sum(sum(value.numel() for value in original[index].parameters()) for index in range(count) if index not in trial)
                if full_count - removed + head_estimate > 50_000_000:
                    break
                keep += 1
            retained = sorted(set(round(i * (count - 1) / max(1, keep - 1)) for i in range(keep)))
            backbone.decoder = nn.ModuleList(backbone.decoder[i] for i in retained)
            for layer in backbone.decoder:
                layer.requires_grad_(True)
            backbone.decoder_norm.requires_grad_(True)
            changed.append('own decoder blocks retained: ' + str(retained))
        return changed
    modules = backbone.stages if cfg.family == 'c' else backbone.layers
    if cfg.train_last_units < 1 or not len(modules):
        raise ValueError('TRAIN_LAST_UNITS and the own-model unit count must be positive')
    # Slices naturally clamp to a shallow foundation's actual depth. If every
    # block is selected, its own embedding table remains the frozen cache prefix.
    selected = list(modules)[-cfg.train_last_units:]
    if cfg.method == 'partial':
        for module in selected:
            module.requires_grad_(True)
        backbone.final_norm.requires_grad_(True)
        return ['last own units and final norm']
    for index, module in enumerate(selected):
        for name, layer in list(module.named_modules()):
            replacement = None
            if isinstance(layer, nn.Linear):
                if cfg.method == 'lorma':
                    replacement = LoRMA(layer, cfg.rank, cfg.alpha)
''')


In [9]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''                elif cfg.method == 'ia3' and name.endswith('attention.qkv'):
                    replacement = IA3QKV(layer)
                elif cfg.method == 'ia3' and name.endswith('ffn.down'):
                    replacement = IA3FFN(layer)
            if replacement is not None:
                parent, _, child = name.rpartition('.')
                setattr(module.get_submodule(parent) if parent else module, child, replacement)
                changed.append(f'{index}:{name}')
            if cfg.method == 'prefix' and type(layer).__name__ == 'SparseCausalSelfAttention':
                layer.prefix_key = nn.Parameter(torch.randn(layer.heads, cfg.prefix_tokens, layer.head_dim) * .02)
                layer.prefix_value = nn.Parameter(torch.randn(layer.heads, cfg.prefix_tokens, layer.head_dim) * .02)
                layer.attend = types.MethodType(prefix_attend, layer)
                changed.append(f'{index}:{name}:KV-prefix')
    if not changed:
        raise ValueError('No intended adaptation modules were found')
    return changed
''')


[Markdown Cell]

`batches.py` — All queries/positives, hard and random negatives, false-negative masks and KD distributions.

In [10]:
# [Code Cell]
module_parts.setdefault('batches.py', []).append(r'''"""Exhaustive query plans, all gold positives, and split-safe hard/in-batch negatives."""
import hashlib
from pathlib import Path
import numpy as np
import torch
from torch.nn import functional as F
from .io_utils import read_json
from .lexical import scores_for, stable_order


def epoch_order(size, seed, epoch, slots):
    values = np.random.default_rng(seed + epoch).permutation(size).tolist()
    values.extend([-1] * ((-len(values)) % slots))
    return np.asarray(values, np.int64).reshape(-1, slots)


class RetrievalBatches:
    """Keep every original training query and every reachable positive candidate."""
    def __init__(self, job, cfg, store):
        self.cfg, self.store = cfg, store
        references = store.catalog['references']['train']
        self.query_ids = references['queries']['ids']
        self.query_docs = references['queries']['document_indices']
        self.candidate_ids = references['candidates']['ids']
        self.candidate_docs = references['candidates']['document_indices']
        self.by_id = {key: index for index, key in enumerate(self.candidate_ids)}
        self.truth = read_json(Path(job['prepared_dir']) / 'train_truth.json')
        self.positives = [[self.by_id[key] for key in self.truth[query] if key in self.by_id] for query in self.query_ids]
        self.lexical = scores_for(job['lexical'], 'train')
        if job['lexical']['splits']['train']['query_ids'] != self.query_ids or job['lexical']['splits']['train']['candidate_ids'] != self.candidate_ids:
            raise ValueError('Training lexical and feature-cache ID order differs')
        self.teacher = None
        if job.get('teacher'):
            value = job['teacher']
            if value['query_ids'] != self.query_ids or value['candidate_ids'] != self.candidate_ids:
                raise ValueError('Teacher and student candidate/query IDs are misaligned')
            self.teacher = np.memmap(value['scores_path'], np.float32, 'r', shape=(len(self.query_ids), len(self.candidate_ids)))

    def __len__(self):
        return len(self.query_ids)
''')


In [11]:
# [Code Cell]
module_parts.setdefault('batches.py', []).append(r'''
    def build(self, query_indices, epoch, step, rank, device):
        chosen = set()
        for query in query_indices:
            if query < 0:
                continue
            positives = set(self.positives[query])
            chosen.update(positives)
            alias_docs = {self.candidate_docs[index] for index in positives}
            negatives = [int(index) for index in stable_order(self.lexical[query])
                         if int(index) not in positives and self.candidate_docs[int(index)] not in alias_docs]
            chosen.update(negatives[:self.cfg.hard_negatives])
            if self.teacher is not None:
                teacher_negatives = [int(index) for index in stable_order(self.teacher[query]) if int(index) not in positives]
                chosen.update(teacher_negatives[:self.cfg.hard_negatives])
            rng = np.random.default_rng(self.cfg.seed + epoch * 1_000_003 + query * 101 + 17)
            randoms = rng.permutation(len(self.candidate_ids))
            taken = 0
            for index in randoms:
                if int(index) not in positives and self.candidate_docs[int(index)] not in alias_docs:
                    chosen.add(int(index))
                    taken += 1
                    if taken == self.cfg.random_negatives:
                        break
        if not chosen:
            chosen.add(0)  # zero-weight DDP tail still constructs a complete graph
        candidates = sorted(chosen)
        documents = [self.query_docs[max(0, query)] for query in query_indices] + [self.candidate_docs[index] for index in candidates]
        roles = [0] * len(query_indices) + [1] * len(candidates)
        inputs = self.store.batch(documents, roles, device)
        positives = torch.zeros(len(query_indices), len(candidates), dtype=torch.bool, device=device)
        denominator = torch.ones_like(positives)
        weights = torch.tensor([query >= 0 for query in query_indices], dtype=torch.float32, device=device)
        for row, query in enumerate(query_indices):
            if query < 0:
                continue
            gold = set(self.positives[query])
            aliases = {self.candidate_docs[index] for index in gold}
            positives[row] = torch.tensor([index in gold for index in candidates], device=device)
            denominator[row] = torch.tensor([index in gold or self.candidate_docs[index] not in aliases for index in candidates], device=device)
''')


In [12]:
# [Code Cell]
module_parts.setdefault('batches.py', []).append(r'''        teacher = None
        if self.teacher is not None:
            teacher = torch.from_numpy(np.asarray(self.teacher[[max(0, q) for q in query_indices]][:, candidates]).copy()).to(device)
        return inputs, positives, denominator, weights, teacher


def relevance_loss(embeddings, query_count, positive, allowed, weights, cfg, teacher=None):
    queries, cases = embeddings[:query_count], embeddings[query_count:]
    with torch.autocast(device_type=queries.device.type, enabled=False):
        cosine = queries.float() @ cases.float().T
    logits = (cosine / cfg.temperature).masked_fill(~allowed, -1e4)
    counts = positive.sum(-1)
    log_probabilities = F.log_softmax(logits, dim=-1)
    positive_loss = -(log_probabilities * positive).sum(-1) / counts.clamp_min(1)
    # Empty/unreachable gold sets remain audited, visited, and supervised as
    # all-negative queries over this pool. No fake positive or query is invented.
    negative_loss = .1 * F.softplus(logits).mean(-1)
    per_query = torch.where(counts > 0, positive_loss, negative_loss)
    if teacher is not None:
        scale = cfg.temperature * cfg.kd_temperature
        probability = F.softmax((teacher.float() / scale).masked_fill(~allowed, -1e4), dim=-1)
        student = F.log_softmax((cosine / scale).masked_fill(~allowed, -1e4), dim=-1)
        kd = F.kl_div(student, probability, reduction='none').sum(-1) * cfg.kd_temperature ** 2
        per_query = (1 - cfg.kd_weight) * per_query + cfg.kd_weight * kd
    return (per_query * weights).sum(), cosine
''')


[Markdown Cell]

`compare.py` — Six-technique audit and optional tune-validated rank ensemble.

In [13]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''"""Audit all six real trained techniques and build a tune-validated rank ensemble."""
from pathlib import Path
import shutil
import zipfile
import numpy as np
from .io_utils import atomic_json, read_json, sha256_file
from .parents import source_hashes
from .preparation import verify_prepared
from .lexical import scores_for, stable_order
from .ranking import inverse_rank, metrics, package
from .state import Checkpoints

REQUIRED = {('a', 'lorma'), ('a', 'partial'), ('b', 'prompt'), ('c', 'ia3'), ('d', 'prefix'), ('b', 'kd')}


def task_outputs(input_root, prepared, task_roots=None):
    data = verify_prepared(prepared)['identity']
    roots = [Path(value) for value in task_roots] if task_roots else sorted({file.parent for file in Path(input_root).rglob('final_report.json')
        if read_json(file).get('version') == 'pcr-output-v1'})
    accepted, rejected = [], []
    for root in roots:
        try:
            job, report = read_json(root / 'job.json'), read_json(root / 'final_report.json')
            if (job['version'] != 'cs781-phase7-pcr-v1' or job['prepared_identity'] != data
                    or report['status'] != 'submission_verified' or read_json(root / 'training_summary.json')['status'] != 'complete'):
                raise ValueError('Not a completed matching Phase 7 model output')
            if source_hashes(root / 'code/pcr') != job['runtime_source_sha256'] or source_hashes(root / 'parent_source' / ('model_' + job['config']['family'])) != job['parent_contract']['source_sha256']:
                raise ValueError('Saved own model/runtime source snapshot changed')
            manager = Checkpoints(root, job['fingerprint'], 0)
            if report['checkpoint_sha256'] not in {receipt['sha256'] for _, receipt in manager.candidates()}:
                raise ValueError('Ranking checkpoint is missing or differs from its receipt')
            archive = root / Path(report['zip_path']).name
            if sha256_file(root / 'prediction.json') != report['prediction_sha256'] or sha256_file(archive) != report['zip_sha256']:
                raise ValueError('Actual submission files changed')
            from .official_evaluator import validate
            from .io_utils import records
            test_ids = [row['id'] for row in records(Path(prepared) / 'test_queries.jsonl')]
            test_pool = read_json(Path(prepared) / 'test_candidate_ids.json')
            if validate(read_json(root / 'prediction.json'), {key: [] for key in test_ids}, set(test_pool)):
                raise ValueError('Actual prediction file violates official query/pool/top-seven rules')
''')


In [14]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''            with zipfile.ZipFile(archive) as zipped:
                if zipped.testzip() is not None or zipped.namelist() != ['prediction.json'] or zipped.read('prediction.json') != (root / 'prediction.json').read_bytes():
                    raise ValueError('Invalid submission ZIP layout or bytes')
            if not 0 < report.get('learned_model_contribution', 0) <= 1:
                raise ValueError('Required learned-method outputs must use their own neural scores')
            directory = root / 'rankings' / Path(report['ranking_directory']).name
            identity = read_json(directory / 'ranking_identity.json')
            if identity['checkpoint_sha256'] != report['checkpoint_sha256'] or identity['data_identity'] != data:
                raise ValueError('Ranking cache belongs to another model/data version')
            for split in ('dev', 'test'):
                receipt = read_json(directory / (split + '_scores.json'))
                if sha256_file(directory / (split + '_scores.bin')) != receipt['sha256']:
                    raise ValueError('Saved full-pool scores are corrupted')
                expected_candidates = read_json(Path(prepared) / (split + '_candidate_ids.json'))
                if receipt['candidate_ids'] != expected_candidates:
                    raise ValueError('Candidate pool ID order differs')
            accepted.append({'root': root, 'job': job, 'report': report, 'directory': directory})
        except (ValueError, KeyError, OSError) as error:
            rejected.append({'root': str(root), 'reason': str(error)})
    return accepted, rejected


def compare(input_root, prepared, output, task_roots=None, ensemble=False):
    root = Path(output)
    root.mkdir(parents=True, exist_ok=True)
    tasks, rejected = task_outputs(input_root, prepared, task_roots)
    if not tasks:
        atomic_json(root / 'input_report.json', {'rejected': rejected})
        raise FileNotFoundError('Attach COMPLETE completed P7 outputs with actual checkpoints and full-pool ranking files; see input_report.json')
    selection = read_json(Path(prepared) / 'selection_split.json')
    truth = read_json(Path(prepared) / 'dev_truth.json')
    seen, methods = {}, []
    for task in tasks:
        report = task['report']
        method = (task['job']['config']['family'], task['job']['config']['method'])
        score = report['fusion']['tune']['micro_f1_at7']
        if method not in seen or score > seen[method]['report']['fusion']['tune']['micro_f1_at7']:
            seen[method] = task
    for method, task in sorted(seen.items()):
        destination = root / Path(task['report']['zip_path']).name
''')


In [15]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''        shutil.copyfile(task['root'] / destination.name, destination)
        methods.append({'family': method[0], 'method': method[1], 'experiment_id': task['job']['config']['experiment_id'],
            'tune': task['report']['fusion']['tune'], 'confirmation': task['report']['fusion']['confirmation'],
            'official_dev': task['report']['fusion']['official_dev'], 'zip_path': str(destination),
            'parameters': read_json(task['root'] / 'model_report.json')})
    winner = max(tasks, key=lambda task: task['report']['fusion']['tune']['micro_f1_at7'])
    best = root / 'best_single_submission.zip'
    shutil.copyfile(winner['root'] / Path(winner['report']['zip_path']).name, best)
    result = {'version': 'pcr-comparison-v1', 'status': 'verified', 'methods': methods,
        'missing_required_methods': [list(value) for value in sorted(REQUIRED - set(seen))],
        'six_distinct_experiments_complete': REQUIRED <= set(seen), 'rejected': rejected,
        'best_single': {'experiment_id': winner['job']['config']['experiment_id'], 'zip_path': str(best),
                        'tune': winner['report']['fusion']['tune']}, 'private_test_score': None}
    if ensemble:
        chosen = list(seen.values())
        info = read_json(chosen[0]['directory'] / 'dev_scores.json')
        test_info = read_json(chosen[0]['directory'] / 'test_scores.json')
        # Each method contributes its frozen, independently tune-selected neural
        # + lexical ranking. Forward selection is then fit ONLY on tune queries.
        matrices = {'dev': [], 'test': []}
        for task in chosen:
            rules = task['report']['fusion']['rule']
            for split in ('dev', 'test'):
                receipt = read_json(task['directory'] / (split + '_scores.json'))
                if receipt['query_ids'] != (info if split == 'dev' else test_info)['query_ids']:
                    raise ValueError('Ensemble query ID order differs')
                dense = np.memmap(task['directory'] / (split + '_scores.bin'), np.float32, 'r', shape=tuple(receipt['shape']))
                lexical = scores_for(task['job']['lexical'], split) if Path(task['job']['lexical']['directory']).is_dir() else None
                if lexical is None:
                    # Rebind the lexical read-only input after Kaggle remounts.
                    from .lexical import lexical_cache
                    candidates = [file.parent for file in Path(input_root).rglob('lexical_manifest.json')
                                  if read_json(file).get('prepared_identity') == task['job']['prepared_identity']]
                    lexical = scores_for(lexical_cache(prepared, candidates[0] if candidates else root / 'lexical'), split)
                fused = np.empty_like(dense)
                for row in range(len(dense)):
                    alpha, constant = rules['neural_weight'], rules['rank_constant']
                    values = dense[row] if alpha == 1 else alpha / (constant + inverse_rank(dense[row])) + (1 - alpha) / (constant + inverse_rank(lexical[row]))
                    fused[row] = 1 / (60 + inverse_rank(values))
                matrices[split].append(fused)
''')


In [16]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''        def predict(values, description):
            return {query: [description['candidate_ids'][int(index)] for index in stable_order(values[row])[:7]]
                    for row, query in enumerate(description['query_ids'])}
        first = max(range(len(chosen)), key=lambda index: chosen[index]['report']['fusion']['tune']['micro_f1_at7'])
        weights = np.zeros(len(chosen), np.float32)
        weights[first] = 1.
        combined = matrices['dev'][first].copy()
        best_score = metrics(truth, predict(combined, info), info['candidate_ids'], selection['tune_ids'])['micro_f1_at7']
        trials = []
        for _ in range(2):
            upgrade = None
            for index in range(len(chosen)):
                for added_weight in (.15, .3, .5):
                    candidate = (1 - added_weight) * combined + added_weight * matrices['dev'][index]
                    evaluated = metrics(truth, predict(candidate, info), info['candidate_ids'], selection['tune_ids'])
                    trials.append({'added_model': index, 'weight': added_weight, 'tune': evaluated})
                    if evaluated['micro_f1_at7'] > best_score and (upgrade is None or evaluated['micro_f1_at7'] > upgrade[0]):
                        upgrade = (evaluated['micro_f1_at7'], index, added_weight, candidate)
            if upgrade is None:
                break
            best_score, index, alpha, combined = upgrade
            weights *= 1 - alpha
            weights[index] += alpha
        # Freeze weights before confirmation/test; no private labels exist here.
        dev_prediction = predict(combined, info)
        test = sum(float(weight) * matrix for weight, matrix in zip(weights, matrices['test']))
        receipt = package(root / 'ensemble', predict(test, test_info), test_info['query_ids'], test_info['candidate_ids'], 'P7_Validated_Ensemble')
        result['ensemble'] = {'frozen_weights': weights.tolist(), 'model_IDs': [task['job']['config']['experiment_id'] for task in chosen],
            'tune': metrics(truth, dev_prediction, info['candidate_ids'], selection['tune_ids']),
            'confirmation': metrics(truth, dev_prediction, info['candidate_ids'], selection['confirm_ids']),
            'official_dev': metrics(truth, dev_prediction, info['candidate_ids']), 'trials': trials, **receipt}
    atomic_json(root / 'final_comparison.json', result)
    return result


def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-root', required=True)
    parser.add_argument('--prepared', required=True)
''')


In [17]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''    parser.add_argument('--output', required=True)
    parser.add_argument('--task-roots')
    parser.add_argument('--ensemble', action='store_true')
    args = parser.parse_args()
    roots = read_json(args.task_roots) if args.task_roots else None
    print(compare(args.input_root, args.prepared, args.output, roots, args.ensemble), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`config.py` — Reviewed source, configuration and ownership verification.

In [18]:
# [Code Cell]
module_parts.setdefault('config.py', []).append(r'''"""Immutable learning/cache choices with separate portable paths and run budgets."""
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class PCRConfig:
    experiment_id: str = 'P7_01_A_LoRMA'
    family: str = 'a'
    method: str = 'lorma'
    run_dir: str = '/kaggle/working/cs781_phase7/P7_01_A_LoRMA'
    input_root: str = '/kaggle/input'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    feature_cache_dir: str | None = None
    teacher_run_dir: str | None = None
    parent_checkpoint: str = 'best'
    world_size: int = 2
    seed: int = 781
    max_epochs: int = 12
    target_total_epochs: int | None = None
    minimum_epochs: int = 3
    patience: int = 3
    max_run_seconds: int = 36000
    cache_run_seconds: int = 36000
    min_free_bytes: int = 1 << 30
    chunk_tokens: int = 256
    tokens_per_chunk: int = 4
    memory_slots: int = 128
    cache_chunk_batch: int = 16
    train_last_units: int = 2
    rank: int = 16
    alpha: float = 16.
    prompt_tokens: int = 32
    prefix_tokens: int = 16
    embedding_dim: int = 256
    query_batch_per_rank: int = 4
    encode_batch_size: int = 16
''')


In [19]:
# [Code Cell]
module_parts.setdefault('config.py', []).append(r'''    hard_negatives: int = 12
    random_negatives: int = 4
    learning_rate: float = .0002
    head_learning_rate: float = .0005
    weight_decay: float = .01
    temperature: float = .07
    kd_temperature: float = 2.
    kd_weight: float = .5
    warmup_fraction: float = .08
    min_lr_ratio: float = .1
    max_grad_norm: float = 1.
    checkpoint_encoder: bool = True
    checkpoint_every_steps: int = 500
    log_every_steps: int = 5
    max_consecutive_overflows: int = 8

    def validate(self):
        choices = {'a': ('lorma', 'partial'), 'b': ('prompt', 'kd'), 'c': ('ia3',), 'd': ('prefix',)}
        if self.family not in choices or self.method not in choices[self.family]:
            raise ValueError('Select A LoRMA/partial, B prompt/KD, C IA3 or D prefix')
        if self.world_size != 2 or self.checkpoint_every_steps != 500:
            raise ValueError('Use two NCCL workers and checkpoint every 500 updates')
        positive = ('max_epochs', 'max_run_seconds', 'cache_run_seconds', 'chunk_tokens', 'tokens_per_chunk',
                    'memory_slots', 'cache_chunk_batch', 'train_last_units', 'rank', 'embedding_dim',
                    'query_batch_per_rank', 'encode_batch_size', 'hard_negatives', 'random_negatives',
                    'prompt_tokens', 'prefix_tokens')
        if any(getattr(self, name) < 1 for name in positive):
            raise ValueError('Budgets, batch sizes and representation dimensions must be positive')
        if not 0 <= self.kd_weight <= 1 or min(self.temperature, self.kd_temperature) <= 0:
            raise ValueError('Invalid contrastive/KD controls')
        if min(self.learning_rate, self.head_learning_rate) <= 0:
            raise ValueError('Learning rates must be positive')
        if self.target_total_epochs is not None and self.target_total_epochs < 1:
            raise ValueError('TARGET_TOTAL_EPOCHS is the absolute positive epoch target')

    def choices(self):
        transport = {'run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path',
            'metadata_path', 'resume_run_dir', 'feature_cache_dir', 'teacher_run_dir', 'max_run_seconds',
            'cache_run_seconds', 'cache_chunk_batch', 'min_free_bytes', 'target_total_epochs'}
        return {key: value for key, value in asdict(self).items() if key not in transport}
''')


[Markdown Cell]

`distillation.py` — Automatic own teacher construction from P7 or supported P6 saved output; no separate export required.

In [20]:
# [Code Cell]
module_parts.setdefault('distillation.py', []).append(r'''"""One-notebook KD: accept own P7/P6 outputs, bootstrap retrieval teacher and export scores."""
from dataclasses import asdict, replace
from pathlib import Path
import json
import shutil
import torch
from .config import PCRConfig
from .io_utils import atomic_json, identity, publish_copy, read_json, records, sha256_file
from .parents import parent_class, source_hashes
from .state import Checkpoints


def validate_teacher(root, prepared_identity, query_ids, candidate_ids):
    root = Path(root)
    receipt = read_json(root / 'teacher_export_receipt.json')
    file = root / 'teacher_scores.bin'
    expected_shape = [len(query_ids), len(candidate_ids)]
    if (receipt.get('version') != 'pcr-teacher-v1' or receipt.get('status') != 'complete'
            or receipt.get('teacher_is_own_phase7_retriever') is not True
            or receipt.get('test_labels_used') is not False or receipt.get('only_training_query_targets') is not True
            or receipt['prepared_identity'] != prepared_identity or receipt['query_ids'] != query_ids
            or receipt['candidate_ids'] != candidate_ids or receipt['shape'] != expected_shape
            or file.stat().st_size != len(query_ids) * len(candidate_ids) * 4
            or sha256_file(file) != receipt['sha256']):
        raise ValueError('Teacher data/IDs/full-pool score bytes are incompatible or incomplete')
    return {**receipt, 'scores_path': str(file)}


def teacher_snapshot(job, cfg):
    train_ids = [row['id'] for row in records(Path(job['prepared_dir']) / 'train_queries.jsonl')]
    candidate_ids = read_json(Path(job['prepared_dir']) / 'train_candidate_ids.json')
    roots = [Path(cfg.run_dir), Path(cfg.run_dir) / 'auto_teacher', *[Path(value) for value in job['resume_roots']]]
    roots += [Path(value) / 'auto_teacher' for value in job['resume_roots']]
    roots += [Path(cfg.teacher_run_dir)] if cfg.teacher_run_dir else [file.parent for file in Path(cfg.input_root).rglob('teacher_export_receipt.json')]
    for root in roots:
        if not (root / 'teacher_export_receipt.json').is_file():
            continue
        if read_json(root / 'teacher_export_receipt.json').get('version') != 'pcr-teacher-v1':
            continue  # Phase 6 summary-target receipts are not PCR score exports
        try:
''')


In [21]:
# [Code Cell]
module_parts.setdefault('distillation.py', []).append(r'''            value = validate_teacher(root, job['prepared_identity'], train_ids, candidate_ids)
        except (ValueError, OSError, KeyError):
            if cfg.teacher_run_dir and root == Path(cfg.teacher_run_dir):
                raise
            continue
        target = Path(cfg.run_dir)
        for name in ('teacher_export_receipt.json', 'teacher_scores.bin'):
            publish_copy(root / name, target / name)
        return {**value, 'scores_path': str(target / 'teacher_scores.bin')}
    return None


def recovered_phase1(phase6_root, destination, student_job):
    """Recover untouched Phase 1 base tensors from a verified frozen-PEFT P6 run.

    Summarization heads, learned low-rank factors, adapters and prompts are NOT
    used as retrieval targets or as student initialization. A Phase 7 retrieval
    teacher is trained internally from this recovered own Phase 1 starting point.
    P6 KD changed B's base weights, so its original B is taken from the mandatory
    matching student Phase 1 attachment instead of claiming those weights frozen.
    """
    from .p6_contract import APPROVED_PHASE6_SOURCES
    root, target = Path(phase6_root), Path(destination)
    old = read_json(root / 'job.json')
    if old.get('version') != 'cs781-phase6-summ-v1':
        raise ValueError('This parent is not an own Phase 6 output')
    if old['runtime_source_sha256'] != APPROVED_PHASE6_SOURCES or source_hashes(root / 'code/summ') != old['runtime_source_sha256']:
        raise ValueError('Phase 6 runtime differs from the reviewed own frozen-PEFT implementation')
    manager = Checkpoints(root, old['fingerprint'], 0)
    latest = manager.latest()
    if latest is None:
        raise FileNotFoundError('Attach full saved Phase 6 output with actual checkpoints, receipts, tokenizer and source; .ipynb alone has no weights')
    payload = manager.load(latest)
    if not payload['cursor']['complete']:
        raise ValueError('The attached Phase 6 parent is paused; finish/save it or attach a completed parent')
    family, method = old['config']['family'], old['config']['method']
    if family == 'b' and (method == 'kd' or old['parent']['checkpoint_sha256'] == student_job['parent']['checkpoint_sha256']):
        return {'parent': student_job['parent'], 'contract': student_job['parent_contract'],
                'source': student_job['parent_source_dir'], 'tokenizer': student_job['tokenizer_path'],
                'metadata': str(next(path for path in (Path(student_job['config']['run_dir']) / 'tokenizer').iterdir() if path.name != 'tokenizer.json')),
''')


In [22]:
# [Code Cell]
module_parts.setdefault('distillation.py', []).append(r'''                'family': 'b', 'provenance': 'verified own B Phase 1 attachment; updated P6 KD task weights are not treated as a frozen original base'}
    supported = {'a': ('lorma', 'lora'), 'b': ('prompt',), 'c': ('houlsby',), 'd': ('pfeiffer',)}
    if method not in supported.get(family, ()):
        raise ValueError('This Phase 6 adaptation does not prove a frozen recoverable Phase 1 base')
    source = root / 'parent_source' / ('model_' + family)
    Config, Model = parent_class(str(source), family, old['parent_contract'])
    with torch.device('meta'):
        probe = Model(Config(**old['parent']['config']))
    expected = probe.state_dict()
    recovered = {}
    for name, value in payload['model'].items():
        if not name.startswith('backbone.'):
            continue
        key = name[len('backbone.'):].replace('.base.', '.')
        if key in expected:
            if key in recovered or value.shape != expected[key].shape or value.dtype != torch.float32:
                raise ValueError('Ambiguous/incompatible frozen base tensor: ' + key)
            recovered[key] = value
    if set(recovered) != set(expected):
        raise ValueError('Phase 6 output is missing original frozen foundation tensors')
    target.mkdir(parents=True, exist_ok=True)
    checkpoint = target / 'recovered_phase1_weights.pt'
    torch.save(recovered, checkpoint)
    parent = dict(old['parent'])
    parent.update(checkpoint=str(checkpoint), checkpoint_sha256=sha256_file(checkpoint),
        checkpoint_bytes=checkpoint.stat().st_size, checkpoint_mtime_ns=checkpoint.stat().st_mtime_ns,
        format='native_state_dict')
    receipt = {'phase6_checkpoint_sha256': latest[1]['sha256'], 'family': family, 'method': method,
        'original_phase1_receipt_sha256': old['parent']['checkpoint_sha256'], 'recovered_weights_sha256': parent['checkpoint_sha256'],
        'phase6_adaptation_weights_discarded': True, 'retrieval_teacher_will_start_from_own_phase1': True}
    atomic_json(target / 'phase6_parent_recovery.json', receipt)
    return {'parent': parent, 'contract': old['parent_contract'], 'source': str(source),
            'tokenizer': str(root / 'tokenizer/tokenizer.json'),
            'metadata': str(next(path for path in (root / 'tokenizer').iterdir() if path.name != 'tokenizer.json')),
            'family': family, 'provenance': receipt}


def auto_teacher(job, cfg, catalog, run_component):
    """Accept a score export, a complete P7 teacher, any supported P6 parent, or own B.

''')


In [23]:
# [Code Cell]
module_parts.setdefault('distillation.py', []).append(r'''    All missing preparation/export work runs inside notebook 07. The supplied
    student always starts from its own Phase 1 B; teacher weights are never copied
    into the student. Pauses preserve the internal teacher's full output.
    """
    snapshot = teacher_snapshot(job, cfg)
    if snapshot:
        return snapshot
    root = Path(cfg.run_dir)
    bootstrap = root / 'auto_teacher'
    explicit = Path(cfg.teacher_run_dir) if cfg.teacher_run_dir else None
    p7 = [explicit] if explicit and (explicit / 'job.json').is_file() and read_json(explicit / 'job.json').get('version') == 'cs781-phase7-pcr-v1' else []
    if not explicit:
        p7 = [file.parent for file in Path(cfg.input_root).rglob('job.json')
              if read_json(file).get('version') == 'cs781-phase7-pcr-v1' and
              read_json(file).get('prepared_identity') == job['prepared_identity'] and
              read_json(file).get('config', {}).get('method') != 'kd' and (file.parent / 'training_summary.json').is_file() and
              read_json(file.parent / 'training_summary.json').get('status') == 'complete']
    if p7:
        # Teacher quality is judged on retrieval development data, not its old
        # summarization score. Complete parent outputs can export scores directly.
        selected = max(p7, key=lambda path: read_json(path / 'training_summary.json').get('best_score', -1))
        old = read_json(selected / 'job.json')
        if old['config']['method'] == 'kd':
            raise ValueError('Select a larger non-KD Phase 7 teacher, or attach any supported Phase 6 A/B/C/D model for automatic teacher training')
        teacher_cfg = PCRConfig(**{**old['config'], 'run_dir': str(bootstrap), 'resume_run_dir': str(selected),
            'input_root': cfg.input_root, 'prepared_dir': cfg.prepared_dir, 'target_total_epochs': None})
        teacher_job = run_component(teacher_cfg, catalog, teacher_only=True, export_only=True)
        if teacher_job is None:
            return None
        return teacher_snapshot(job, cfg)
    if (bootstrap / 'job.json').is_file():
        old = read_json(bootstrap / 'job.json')
        values = {**old['config'], 'run_dir': str(bootstrap), 'resume_run_dir': str(bootstrap),
                  'input_root': cfg.input_root, 'prepared_dir': cfg.prepared_dir}
        teacher_cfg = PCRConfig(**values)
        run_component(teacher_cfg, catalog, teacher_only=True)
        return teacher_snapshot(job, cfg)
    p6 = [explicit] if explicit and (explicit / 'job.json').is_file() and read_json(explicit / 'job.json').get('version') == 'cs781-phase6-summ-v1' else []
    if not explicit:
        p6 = [file.parent for file in Path(cfg.input_root).rglob('job.json') if read_json(file).get('version') == 'cs781-phase6-summ-v1'
''')


In [24]:
# [Code Cell]
module_parts.setdefault('distillation.py', []).append(r'''              and (file.parent / 'training_summary.json').is_file()
              and read_json(file.parent / 'training_summary.json').get('status') == 'complete']
    if explicit and not p6:
        raise FileNotFoundError('TEACHER_RUN_DIR must be a complete own Phase 6/7 task root or a P7 teacher score export, containing job.json/actual weights')
    if p6:
        selected = max(p6, key=lambda path: read_json(path / 'job.json').get('parent_parameter_count', 0))
        override = recovered_phase1(selected, root / 'recovered_teacher_foundation', job)
    else:
        override = {'parent': job['parent'], 'contract': job['parent_contract'], 'source': job['parent_source_dir'],
            'tokenizer': job['tokenizer_path'], 'metadata': str(next(path for path in (root / 'tokenizer').iterdir() if path.name != 'tokenizer.json')),
            'family': 'b', 'provenance': 'Own full B Phase 1 -> internally trained prompt teacher -> compact B KD student'}
    family = override['family']
    method = {'a': 'partial', 'b': 'prompt', 'c': 'ia3', 'd': 'prefix'}[family]
    teacher_cfg = replace(cfg, experiment_id='P7_Auto_Teacher_' + family.upper(), family=family, method=method,
        run_dir=str(bootstrap), resume_run_dir=None, teacher_run_dir=None, target_total_epochs=None,
        train_last_units=1 if family == 'c' else 2, max_epochs=6, minimum_epochs=2, patience=2,
        learning_rate=.001 if family in ('b', 'c') else .0002, kd_weight=0.,
        embedding_dim=max(512, cfg.embedding_dim) if family == 'b' else cfg.embedding_dim)
    atomic_json(root / 'auto_teacher_plan.json', {'source': override['provenance'], 'teacher_config': asdict(teacher_cfg),
        'student_starts_from_original_own_phase1_B': True, 'phase6_summarizer_used_as_retrieval_teacher_without_training': False})
    run_component(teacher_cfg, catalog, parent_override=override, teacher_only=True)
    return teacher_snapshot(job, cfg)
''')


[Markdown Cell]

`export.py` — Two-GPU score export and dev-selected checkpoint/ranking rule.

In [25]:
# [Code Cell]
module_parts.setdefault('export.py', []).append(r'''"""Distributed full-pool scoring and dev-selected official submission artifacts."""
import argparse
from datetime import timedelta
from pathlib import Path
import socket
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from .config import PCRConfig
from .features import FeatureStore
from .model import build_model
from .state import Checkpoints
from .io_utils import atomic_json, identity, read_json, sha256_file
from .ranking import embed, dense_scores, select_fusion, predictions, package
from .lexical import scores_for


def score_worker(rank, job, checkpoint, root, splits, port):
    cfg = PCRConfig(**job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank, world_size=2, timeout=timedelta(hours=2))
    try:
        model = build_model(job, cfg, device, load_parent=False)
        payload = torch.load(checkpoint, map_location='cpu', weights_only=True, mmap=True)
        model.load_state_dict(payload['model'], strict=True)
        del payload
        model.eval()
        store = FeatureStore(job['features'])
        root = Path(root)
        for split in splits:
            receipt = root / (split + '_scores.json')
            score_file = root / (split + '_scores.bin')
            complete = [bool(receipt.is_file() and score_file.is_file() and sha256_file(score_file) == read_json(receipt)['sha256']) if rank == 0 else None]
            dist.broadcast_object_list(complete, src=0)
            if complete[0]:
                continue
            candidates = embed(model, store, split, 'candidates', cfg, device, rank, cfg.world_size)
''')


In [26]:
# [Code Cell]
module_parts.setdefault('export.py', []).append(r'''            queries = embed(model, store, split, 'queries', cfg, device, rank, cfg.world_size)
            if rank == 0:
                reference = store.catalog['references'][split]
                target = root / (split + '_scores.bin')
                matrix = dense_scores(queries, candidates, target)
                shape = list(matrix.shape)
                del matrix
                atomic_json(receipt, {'sha256': sha256_file(target), 'shape': shape,
                    'query_ids': reference['queries']['ids'], 'candidate_ids': reference['candidates']['ids']})
            del queries, candidates
            dist.barrier()
    finally:
        dist.destroy_process_group()


def score_checkpoint(job, candidate, splits):
    root = Path(job['config']['run_dir']) / 'rankings' / candidate[1]['sha256'][:16]
    root.mkdir(parents=True, exist_ok=True)
    body = {'checkpoint_sha256': candidate[1]['sha256'], 'data_identity': job['prepared_identity'],
            'feature_identity': job['features']['identity'], 'fingerprint': job['fingerprint']}
    signature = root / 'ranking_identity.json'
    if signature.is_file() and read_json(signature) != body:
        raise ValueError('Saved ranking scores belong to another checkpoint/data/feature cache')
    atomic_json(signature, body)
    with socket.socket() as sock:
        sock.bind(('127.0.0.1', 0))
        port = sock.getsockname()[1]
    mp.spawn(score_worker, args=(job, str(candidate[0]), str(root), splits, port), nprocs=2, join=True)
    return root


def export_task(job_path, teacher_only=False):
    job = read_json(job_path)
    cfg = PCRConfig(**job['config'])
    root = Path(cfg.run_dir)
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    latest, best = manager.latest(), manager.best()
    if not latest or not best or not manager.load(latest)['cursor']['complete']:
        raise ValueError('Finish/resume training before claiming a complete retrieval submission or teacher')
    manager.adopt(best, best=True)
''')


In [27]:
# [Code Cell]
module_parts.setdefault('export.py', []).append(r'''    manager.adopt(latest)
    if teacher_only:
        directory = score_checkpoint(job, best, ('train',))
        values = read_json(directory / 'train_scores.json')
        target = root / 'teacher_scores.bin'
        from .io_utils import publish_copy
        publish_copy(directory / 'train_scores.bin', target)
        report = {'version': 'pcr-teacher-v1', 'status': 'complete', 'prepared_identity': job['prepared_identity'],
            'teacher_is_own_phase7_retriever': True, 'test_labels_used': False, 'only_training_query_targets': True,
            'sha256': sha256_file(target), 'teacher_checkpoint_sha256': best[1]['sha256'],
            'teacher_parameters': read_json(root / 'model_report.json')['model_unique_parameters'], **values}
        atomic_json(root / 'teacher_export_receipt.json', report)
        return report
    truth = read_json(Path(job['prepared_dir']) / 'dev_truth.json')
    selection = read_json(Path(job['prepared_dir']) / 'selection_split.json')
    candidates, seen = [], set()
    for name, candidate in (('best', best), ('latest', latest)):
        if candidate[1]['sha256'] in seen:
            continue
        seen.add(candidate[1]['sha256'])
        directory = score_checkpoint(job, candidate, ('dev',))
        info = read_json(directory / 'dev_scores.json')
        dense = np.memmap(directory / 'dev_scores.bin', np.float32, 'r', shape=tuple(info['shape']))
        evaluated = select_fusion(dense, scores_for(job['lexical'], 'dev'), info['query_ids'], info['candidate_ids'], truth, selection)
        candidates.append({'selection': name, 'checkpoint_sha256': candidate[1]['sha256'], 'directory': str(directory),
                           'fusion': evaluated, 'checkpoint': candidate})
    winner = max(candidates, key=lambda row: (row['fusion']['tune']['micro_f1_at7'], row['selection'] == 'best'))
    # The checkpoint and fusion rule are frozen before test scoring.
    atomic_json(root / 'frozen_ranking_rule.json', {key: value for key, value in winner.items() if key != 'checkpoint'})
    dev_info = read_json(Path(winner['directory']) / 'dev_scores.json')
    dev_dense = np.memmap(Path(winner['directory']) / 'dev_scores.bin', np.float32, 'r', shape=tuple(dev_info['shape']))
    dev_rows = predictions(dev_dense, scores_for(job['lexical'], 'dev'), dev_info['query_ids'], dev_info['candidate_ids'], **winner['fusion']['rule'])
    atomic_json(root / 'dev_prediction.json', dev_rows)
    directory = score_checkpoint(job, winner['checkpoint'], ('test',))
    info = read_json(directory / 'test_scores.json')
    dense = np.memmap(directory / 'test_scores.bin', np.float32, 'r', shape=tuple(info['shape']))
    rules = winner['fusion']['rule']
    rows = predictions(dense, scores_for(job['lexical'], 'test'), info['query_ids'], info['candidate_ids'], **rules)
    receipt = package(root, rows, info['query_ids'], info['candidate_ids'], cfg.experiment_id)
    report = {'version': 'pcr-output-v1', 'status': 'submission_verified', 'experiment_id': cfg.experiment_id,
''')


In [28]:
# [Code Cell]
module_parts.setdefault('export.py', []).append(r'''        'family': cfg.family, 'method': cfg.method, 'prepared_identity': job['prepared_identity'],
        'checkpoint_sha256': winner['checkpoint_sha256'], 'ranking_directory': str(directory),
        'checkpoint_selection': winner['selection'], 'fusion': winner['fusion'], 'test_score': None,
        'dev_prediction_sha256': sha256_file(root / 'dev_prediction.json'),
        'test_labels_used': False, 'learned_model_contribution': rules['neural_weight'], **receipt}
    atomic_json(root / 'final_report.json', report)
    atomic_json(root / 'submission_receipt.json', report)
    return report


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    parser.add_argument('--teacher-only', action='store_true')
    args = parser.parse_args()
    print(export_task(args.job, args.teacher_only), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`features.py` — All-source frozen prefix encoding, ragged FP16 caches and per-document fault recovery.

In [29]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''"""Resumable frozen-prefix caches; never cache a trainable top-layer output."""
import argparse
from datetime import timedelta
import hashlib
import json
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn import functional as F
from tokenizers import Tokenizer
import tokenizers
from tqdm.auto import tqdm
from .config import PCRConfig
from .io_utils import atomic_json, ensure_space, identity, read_json, records, sha256_file
from .model import build_model


def catalog(prepared, root):
    """Exact-text de-duplication across all IDs/splits only saves frozen computation."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    signature = read_json(Path(prepared) / 'prepared_manifest.json')['identity']
    saved = root / 'document_catalog.json'
    if saved.is_file():
        value = read_json(saved)
        if value['prepared_identity'] != signature or sha256_file(root / 'documents.jsonl') != value['documents_sha256']:
            raise ValueError('Frozen document catalog changed')
        return value
    seen, references = {}, {}
    path = root / 'documents.jsonl'
    with path.open('w', encoding='utf-8', newline='\n') as stream:
        for split in ('train', 'dev', 'test'):
            references[split] = {}
            for kind in ('queries', 'candidates'):
''')


In [30]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''                values, ids = [], []
                for row in records(Path(prepared) / (split + '_' + kind + '.jsonl')):
                    digest = hashlib.sha256(row['text'].encode()).hexdigest()
                    if digest not in seen:
                        seen[digest] = len(seen)
                        stream.write(json.dumps({'index': seen[digest], 'text': row['text'], 'sha256': digest}, ensure_ascii=False) + '\n')
                    values.append(seen[digest])
                    ids.append(row['id'])
                references[split][kind] = {'document_indices': values, 'ids': ids}
    value = {'prepared_identity': signature, 'unique_documents': len(seen), 'references': references,
             'documents_sha256': sha256_file(path)}
    atomic_json(saved, value)
    return value


def cache_identity(job, cfg):
    return identity({'version': 'pcr-frozen-prefix-v1', 'foundation': job['parent']['checkpoint_sha256'],
        'tokenizer': job['parent']['tokenizer_sha256'], 'tokenizers': tokenizers.__version__, 'family': cfg.family,
        'prefix_cut': 'full-own-B-encoder' if cfg.family == 'b' else max(0,
            job['parent']['config']['gru_layers' if cfg.family == 'c' else 'n_layers'] - cfg.train_last_units),
        'chunk': cfg.chunk_tokens, 'pooled_per_chunk': cfg.tokens_per_chunk, 'slots': cfg.memory_slots,
        'prepared': job['prepared_identity'], 'torch': str(torch.__version__),
        'frozen_execution_source': {name: sha256_file(Path(__file__).parent / name)
                                    for name in ('features.py', 'model.py', 'adaptation.py')},
        'foundation_source': job['parent_contract']['source_sha256'], 'policy': 'all-tokens-fp16-ragged-memory-v1'})


def verify_cache(root, expected):
    root = Path(root)
    value = read_json(root / 'feature_manifest.json')
    if value.get('version') != 'pcr-features-v1' or value.get('status') != 'complete' or value['identity'] != expected:
        raise ValueError('Feature cache is incomplete or from another frozen prefix/data/tokenizer')
    for name, digest in value['files'].items():
        if sha256_file(root / name) != digest:
            raise ValueError('Frozen feature cache checksum failed: ' + name)
    expected_index = {}
    for rank in range(2):
        offset = 0
        for number, item in enumerate(records(root / f'rank_{rank}.jsonl')):
            if item['index'] != rank + 2 * number or item['offset'] != offset or item['length'] < 1:
''')


In [31]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''                raise ValueError('Frozen cache journal has an invalid document sequence/offset')
            offset += item['length']
            expected_index[str(item['index'])] = {'rank': rank, **item}
        if (root / f'rank_{rank}.bin').stat().st_size != offset * value['width'] * 2:
            raise ValueError('Frozen cache feature bytes do not match the committed row lengths')
    if value['index'] != expected_index or value['catalog'] != read_json(root / 'document_catalog.json') or len(expected_index) != value['catalog']['unique_documents']:
        raise ValueError('Frozen cache index/catalog differs from its committed source records')
    return {**value, 'directory': str(root)}


def select_cache(job, cfg):
    expected = cache_identity(job, cfg)
    if cfg.feature_cache_dir:
        return verify_cache(cfg.feature_cache_dir, expected)
    paths = [Path(cfg.feature_cache_dir)] if cfg.feature_cache_dir else sorted({file.parent for file in Path(cfg.input_root).rglob('feature_manifest.json')})
    local = Path(cfg.run_dir) / 'features'
    paths += sorted({file.parent for file in Path(cfg.run_dir).rglob('feature_manifest.json')})
    if (local / 'feature_manifest.json').is_file():
        paths.insert(0, local)
    for root in paths:
        value = read_json(root / 'feature_manifest.json')
        if value.get('identity') == expected:
            return verify_cache(root, expected)
    if cfg.feature_cache_dir:
        raise FileNotFoundError('FEATURE_CACHE_DIR does not contain a matching completed cache')
    # Recover an interrupted cache from the same saved task, without treating
    # uncommitted bytes as completed documents.
    for old in job.get('resume_roots', []):
        source = Path(old) / 'features'
        if (source / 'cache_job.json').is_file() and read_json(source / 'cache_job.json')['identity'] == expected:
            if source.resolve() != local.resolve() and not (local / 'cache_job.json').is_file():
                import shutil
                needed = sum(file.stat().st_size for file in source.iterdir() if file.is_file())
                local.mkdir(parents=True, exist_ok=True)
                ensure_space(local, needed, cfg.min_free_bytes)
                for file in source.iterdir():
                    if file.is_file():
                        shutil.copyfile(file, local / file.name)
            break
    local.mkdir(parents=True, exist_ok=True)
''')


In [32]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''    cat = catalog(job['prepared_dir'], local)
    width = job['parent']['config']['d_model']
    estimate = cat['unique_documents'] * cfg.memory_slots * width * 2
    # Reserve room for recovery states as well as the feature cache. Inputs live
    # on Kaggle's read-only mounts and are not copied if already complete.
    ensure_space(local, max(0, estimate - sum(p.stat().st_size for p in local.glob('rank_*.bin'))) +
                 job['parent_parameter_count'] * 12, cfg.min_free_bytes)
    cache_job = {'identity': expected, 'config': job['config'], 'model_job': job,
                 'catalog': cat, 'directory': str(local), 'width': width}
    atomic_json(local / 'cache_job.json', cache_job)
    return {'status': 'needs_build', 'directory': str(local), 'identity': expected}


def recover_journal(path, binary, width, rank):
    """Discard an interrupted trailing row; retain every committed document."""
    values, byte_offset, end = [], 0, 0
    if path.is_file():
        with path.open('rb') as stream:
            while True:
                line = stream.readline()
                if not line:
                    break
                if not line.endswith(b'\n'):
                    break
                row = json.loads(line)
                if row['index'] != rank + 2 * len(values) or row['offset'] != end or row['length'] < 1:
                    raise ValueError('Invalid frozen-cache journal order/offset')
                if values and row['index'] <= values[-1]['index']:
                    raise ValueError('Duplicate/out-of-order cached document')
                end += row['length']
                values.append(row)
                byte_offset = stream.tell()
        with path.open('r+b') as stream:
            stream.truncate(byte_offset)
    binary.touch(exist_ok=True)
    if binary.stat().st_size < end * width * 2:
        raise ValueError('Committed frozen features are truncated; attach an intact saved output')
    with binary.open('r+b') as stream:
        stream.truncate(end * width * 2)
    return values, end
''')


In [33]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''

@torch.inference_mode()
def document_features(model, text, tokenizer, cfg, device):
    tokens = tokenizer.encode(text, add_special_tokens=False).ids + [2]
    chunks = [tokens[first:first + cfg.chunk_tokens] for first in range(0, len(tokens), cfg.chunk_tokens)]
    values, first, batch_size = [], 0, cfg.cache_chunk_batch
    while first < len(chunks):
        parts = chunks[first:first + batch_size]
        ids = torch.zeros(len(parts), cfg.chunk_tokens, dtype=torch.long)
        valid = torch.zeros_like(ids, dtype=torch.bool)
        for row, part in enumerate(parts):
            ids[row, :len(part)] = torch.tensor(part)
            valid[row, :len(part)] = True
        try:
            hidden = None
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                hidden = model.frozen_prefix(ids.to(device), valid.to(device), all(len(part) == cfg.chunk_tokens for part in parts))
            row = 0
            while row < len(parts):
                length, stop = len(parts[row]), row + 1
                while stop < len(parts) and len(parts[stop]) == length:
                    stop += 1
                pooled = F.adaptive_avg_pool1d(hidden[row:stop, :length].float().transpose(1, 2), min(length, cfg.tokens_per_chunk))
                values.append(pooled.transpose(1, 2).reshape(-1, hidden.shape[-1]).cpu())
                row = stop
            del hidden
            first += len(parts)
        except torch.cuda.OutOfMemoryError:
            hidden = None
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
    memory = torch.cat(values)
    memory = F.adaptive_avg_pool1d(memory.T[None], min(len(memory), cfg.memory_slots))[0].T
    if not torch.isfinite(memory).all():
        raise FloatingPointError('Nonfinite frozen-prefix representation')
    return memory.half().numpy(), len(tokens)

''')


In [34]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''
def cache_worker(rank, cache_job, port, event):
    cfg = PCRConfig(**cache_job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', world_size=2, rank=rank, timeout=timedelta(hours=2))
    root, width = Path(cache_job['directory']), cache_job['width']
    try:
        model = build_model(cache_job['model_job'], cfg, device)
        model.eval()
        tokenizer = Tokenizer.from_file(cache_job['model_job']['tokenizer_path'])
        journal, binary = root / f'rank_{rank}.jsonl', root / f'rank_{rank}.bin'
        previous, offset = recover_journal(journal, binary, width, rank)
        done = {row['index'] for row in previous}
        total = (cache_job['catalog']['unique_documents'] + 1 - rank) // 2
        progress = tqdm(total=total, initial=len(done), disable=rank != 0, desc='Frozen prefix cache, GPU 0', unit='cases')
        started = time.monotonic()
        with binary.open('ab') as output, journal.open('a', encoding='utf-8', newline='\n') as log:
            for row in records(root / 'documents.jsonl'):
                if row['index'] % 2 != rank or row['index'] in done:
                    continue
                deadline = float(os.environ.get('CS781_P7_DEADLINE', 'inf'))
                if event.is_set() or time.time() >= deadline or time.monotonic() - started >= cfg.cache_run_seconds:
                    break
                document_started = time.monotonic()
                memory, tokens = document_features(model, row['text'], tokenizer, cfg, device)
                output.write(memory.tobytes())
                output.flush()
                os.fsync(output.fileno())
                item = {'index': row['index'], 'offset': offset, 'length': len(memory), 'source_tokens': tokens,
                        'content_sha256': row['sha256'], 'encode_seconds': time.monotonic() - document_started}
                log.write(json.dumps(item) + '\n')
                log.flush()
                os.fsync(log.fileno())
                offset += len(memory)
                done.add(row['index'])
                progress.update()
                progress.set_postfix(source_tokens=tokens, committed=len(done))
        progress.close()
''')


In [35]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''        status = torch.tensor(int(len(done) == total), device=device)
        dist.all_reduce(status, op=dist.ReduceOp.MIN)
        if status.item():
            dist.barrier()
            if rank == 0:
                index = {}
                tokens, encode_seconds = 0, 0.
                for source_rank in range(2):
                    for item in records(root / f'rank_{source_rank}.jsonl'):
                        index[str(item['index'])] = {'rank': source_rank, **item}
                        tokens += item['source_tokens']
                        encode_seconds += item.get('encode_seconds', 0.)
                if len(index) != cache_job['catalog']['unique_documents']:
                    raise ValueError('Cache does not cover every unique query/candidate document')
                files = {name: sha256_file(root / name) for name in ('rank_0.bin', 'rank_1.bin', 'rank_0.jsonl', 'rank_1.jsonl', 'document_catalog.json', 'documents.jsonl')}
                manifest = {'version': 'pcr-features-v1', 'status': 'complete', 'identity': cache_job['identity'],
                    'width': width, 'index': index, 'catalog': cache_job['catalog'], 'files': files,
                    'all_source_tokens_encoded': True, 'unique_document_source_tokens_with_EOS': tokens,
                    'sum_document_encode_seconds_both_GPUs': encode_seconds,
                    'final_cache_session_wall_seconds': time.monotonic() - started,
                    'frozen_outputs_only': True}
                atomic_json(root / 'feature_manifest.json', manifest)
        elif rank == 0:
            atomic_json(root / 'cache_pause_report.json', {'status': 'paused_recoverable', 'next_action': 'Save complete output; resume the SAME notebook. Completed case rows are retained.'})
    finally:
        dist.destroy_process_group()


class FeatureStore:
    """Read-only ragged FP16 memories indexed by exact original document identity."""
    def __init__(self, manifest):
        self.manifest = manifest
        self.catalog = manifest['catalog']
        root, width = Path(manifest['directory']), manifest['width']
        self.arrays = {}
        for rank in range(2):
            file = root / f'rank_{rank}.bin'
            self.arrays[rank] = np.memmap(file, np.float16, 'r').reshape(-1, width) if file.stat().st_size else np.empty((0, width), np.float16)

    def batch(self, indices, roles, device):
''')


In [36]:
# [Code Cell]
module_parts.setdefault('features.py', []).append(r'''        rows = [self.manifest['index'][str(int(index))] for index in indices]
        length = max(item['length'] for item in rows)
        values = torch.zeros(len(rows), length, self.manifest['width'], dtype=torch.float16)
        valid = torch.zeros(len(rows), length, dtype=torch.bool)
        for row, item in enumerate(rows):
            value = self.arrays[item['rank']][item['offset']:item['offset'] + item['length']]
            if len(value) != item['length']:
                raise ValueError('Frozen feature row is truncated')
            values[row, :len(value)] = torch.from_numpy(value.copy())
            valid[row, :len(value)] = True
        role = torch.as_tensor(roles, dtype=torch.long)
        def move(value):
            return value.pin_memory().to(device, non_blocking=True) if device.type == 'cuda' else value.to(device)
        return {'memory': move(values), 'valid': move(valid), 'role': move(role)}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--cache-job', required=True)
    args = parser.parse_args()
    job = read_json(args.cache_job)
    context = mp.get_context('spawn')
    event = context.Event()
    signal.signal(signal.SIGINT, lambda *_: event.set())
    signal.signal(signal.SIGTERM, lambda *_: event.set())
    with socket.socket() as sock:
        sock.bind(('127.0.0.1', 0))
        port = sock.getsockname()[1]
    mp.spawn(cache_worker, args=(job, port, event), nprocs=2, join=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`io_utils.py` — Reviewed source, configuration and ownership verification.

In [37]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()
''')


In [38]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''
def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)

def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
''')


In [39]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''    os.replace(temporary, destination)
''')


[Markdown Cell]

`jobs.py` — Reviewed source, configuration and ownership verification.

In [40]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''"""Verify own foundations, prepared data, immutable source and exact recovery settings."""
from dataclasses import asdict
from pathlib import Path
import shutil
import torch
import tokenizers
from . import VERSION
from .config import PCRConfig
from .io_utils import atomic_json, ensure_space, identity, publish_copy, read_json, sha256_file
from .parents import parent_class, source_hashes, tokenizer_inputs, verify_parent
from .preparation import prepared_input
from .lexical import lexical_cache
from .state import Checkpoints


def hardware():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select Kaggle T4 x2: two CUDA devices and the NCCL backend are required')
    for rank in range(2):
        if 'T4' not in torch.cuda.get_device_name(rank):
            raise RuntimeError('Expected two T4 GPUs; found ' + torch.cuda.get_device_name(rank))
        print('GPU', rank, torch.cuda.get_device_name(rank), flush=True)


def resume_inputs(cfg):
    local = Path(cfg.run_dir) / 'job.json'
    paths = [local] if local.is_file() else [Path(cfg.resume_run_dir) / 'job.json'] if cfg.resume_run_dir else sorted(Path(cfg.input_root).rglob('job.json'))
    choices = []
    for file in paths:
        value = read_json(file)
        if value.get('version') != VERSION or value.get('config', {}).get('experiment_id') != cfg.experiment_id:
            if cfg.resume_run_dir:
                raise ValueError('RESUME_RUN_DIR must be this SAME Phase 7 experiment, not a foundation or Phase 6 task')
            continue
        if value['runtime_source_sha256'] != source_hashes(Path(__file__).parent) or source_hashes(file.parent / 'code/pcr') != value['runtime_source_sha256']:
            raise ValueError('Exact recovery requires its original Phase 7 notebook/runtime')
        if PCRConfig(**value['config']).choices() != cfg.choices() or value['torch'] != str(torch.__version__):
            raise ValueError('Saved learning choices or PyTorch build differ; use its exact settings/build')
        choices.append((file.parent, value))
    if len({value['fingerprint'] for _, value in choices}) > 1:
''')


In [41]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''        raise ValueError('Conflicting saved versions share this ID; set RESUME_RUN_DIR')
    return choices


def select_lexical(prepared, cfg):
    data = read_json(Path(prepared) / 'prepared_manifest.json')['identity']
    local = Path(cfg.run_dir) / 'lexical'
    paths = [local] if (local / 'lexical_manifest.json').is_file() else []
    paths += [file.parent for file in Path(cfg.input_root).rglob('lexical_manifest.json')
              if read_json(file).get('prepared_identity') == data]
    return lexical_cache(prepared, paths[0] if paths else local)


def prepare_job(cfg, catalog, teacher=None, parent_override=None):
    cfg.validate()
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    snapshot = root / 'code/pcr'
    snapshot.mkdir(parents=True, exist_ok=True)
    for file in Path(__file__).parent.glob('*.py'):
        target = snapshot / file.name
        if file.resolve() != target.resolve():
            if (root / 'job.json').is_file() and target.is_file() and target.read_bytes() != file.read_bytes():
                raise ValueError('Saved Phase 7 source changed; use its exact notebook')
            shutil.copyfile(file, target)
    prepared, manifest = prepared_input(cfg)
    previous = resume_inputs(cfg)
    prior = previous[0] if previous else None
    source = root / 'parent_source' / ('model_' + cfg.family)
    source.mkdir(parents=True, exist_ok=True)
    token_dir = root / 'tokenizer'
    token_dir.mkdir(exist_ok=True)
    if prior:
        old_root, old = prior
        parent, contract = dict(old['parent']), old['parent_contract']
        origin = old_root / 'parent_source' / ('model_' + cfg.family)
        tokenizer = old_root / 'tokenizer/tokenizer.json'
        metadata = next(file for file in (old_root / 'tokenizer').iterdir() if file.name != 'tokenizer.json')
        seed = old_root / 'foundation_seed.pt'
        if seed.is_file():
''')


In [42]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''            parent['checkpoint'] = str(seed)
            parent['checkpoint_mtime_ns'] = seed.stat().st_mtime_ns
    elif parent_override:
        parent, contract = dict(parent_override['parent']), parent_override['contract']
        origin, tokenizer, metadata = map(Path, (parent_override['source'], parent_override['tokenizer'], parent_override['metadata']))
    else:
        contract = catalog[cfg.family]
        parent_root, parent = verify_parent(cfg, contract)
        contract = parent.get('native_contract', contract)
        tokenizer, metadata, hashes = tokenizer_inputs(cfg, contract)
        parent.update(hashes)
        origin = parent_root / parent['native_contract']['source_subdir'] if parent.get('native_contract') else parent_root / 'code' / ('model_' + cfg.family)
    for file in origin.glob('*.py'):
        destination = source / file.name
        if file.resolve() != destination.resolve():
            shutil.copyfile(file, destination)
    if source_hashes(source) != contract['source_sha256']:
        raise ValueError('Foundation source differs from its verified contract')
    for file in (tokenizer, metadata):
        if file.resolve() != (token_dir / file.name).resolve():
            shutil.copyfile(file, token_dir / file.name)
    if sha256_file(token_dir / 'tokenizer.json') != parent['tokenizer_sha256']:
        raise ValueError('Tokenizer does not match this actual own foundation')
    # Preserve an initial foundation snapshot so a timeout during frozen-cache
    # creation can resume using the complete task output, even before training.
    seed = root / 'foundation_seed.pt'
    if not seed.is_file():
        incoming = Path(parent['checkpoint'])
        if not incoming.is_file():
            candidates = [file for file in Path(cfg.input_root).rglob(incoming.name)
                          if file.stat().st_size == parent['checkpoint_bytes']]
            incoming = next((file for file in candidates if sha256_file(file) == parent['checkpoint_sha256']), None)
            if incoming is None:
                raise FileNotFoundError('Attach the matching Phase 1 foundation or the complete prior task output with foundation_seed.pt')
        ensure_space(root, incoming.stat().st_size, cfg.min_free_bytes)
        publish_copy(incoming, seed)
    if sha256_file(seed) != parent['checkpoint_sha256']:
        raise ValueError('Initial own foundation snapshot is corrupted')
    parent.update(checkpoint=str(seed), checkpoint_bytes=seed.stat().st_size, checkpoint_mtime_ns=seed.stat().st_mtime_ns)
    Config, Model = parent_class(str(source), cfg.family, contract)
''')


In [43]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''    with torch.device('meta'):
        probe = Model(Config(**parent['config']))
    count = sum(value.numel() for value in probe.parameters())
    del probe
    assert 50_000_000 < count < 300_000_000
    lexical = select_lexical(prepared, cfg)
    fingerprint = identity({'version': VERSION, 'choices': cfg.choices(), 'data': manifest['identity'],
        'foundation': parent['checkpoint_sha256'], 'tokenizer': parent['tokenizer_sha256'],
        'sources': source_hashes(Path(__file__).parent), 'torch': str(torch.__version__),
        'tokenizers': tokenizers.__version__,
        'teacher': teacher and teacher['sha256']})
    if prior and fingerprint != old['fingerprint']:
        raise ValueError('Recovery fingerprint changed; do not mix another run/teacher/optimizer')
    target = cfg.target_total_epochs or (old['target_epochs'] if prior else cfg.max_epochs)
    job = {'version': VERSION, 'config': asdict(cfg), 'prepared_dir': str(prepared), 'prepared_identity': manifest['identity'],
        'parent': parent, 'parent_contract': contract, 'parent_source_dir': str(source), 'parent_parameter_count': count,
        'tokenizer_path': str(token_dir / 'tokenizer.json'), 'fingerprint': fingerprint, 'lexical': lexical,
        'runtime_source_sha256': source_hashes(Path(__file__).parent), 'torch': str(torch.__version__),
        'resume_roots': list(dict.fromkeys([str(path) for path, _ in previous] +
            [value for _, old_job in previous for value in old_job.get('resume_roots', []) if Path(value).is_dir()])),
        'teacher': teacher, 'target_epochs': target}
    manager = Checkpoints(root, fingerprint, cfg.min_free_bytes, job['resume_roots'])
    latest = manager.latest()
    if prior and not latest and (prior[1].get('resume_checkpoint') or (Path(prior[0]) / 'training_summary.json').is_file() or any((Path(prior[0]) / 'checkpoints').glob('*.pt.json'))):
        raise FileNotFoundError('A previously trained task is missing its full checkpoints; attach COMPLETE outputs')
    if latest:
        best = manager.best()
        if best:
            manager.adopt(best, best=True)
        latest = manager.adopt(latest)
    if latest and target < latest[1]['epoch']:
        raise ValueError('Target epochs are below completed epochs')
    job['resume_checkpoint'] = str(latest[0]) if latest else None
    job['extension_requested'] = bool(latest and cfg.target_total_epochs is not None and target > latest[1]['epoch'])
    atomic_json(root / 'job.json', job)
    atomic_json(root / 'task_config.json', asdict(cfg))
    return job


def sanity(job):
''')


In [44]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''    from .features import FeatureStore
    from .model import build_model
    cfg = PCRConfig(**job['config'])
    device = torch.device('cuda', 0)
    model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        payload = torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)
        model.load_state_dict(payload['model'], strict=True)
    store = FeatureStore(job['features'])
    reference = store.catalog['references']['train']
    inputs = store.batch([reference['queries']['document_indices'][0], reference['candidates']['document_indices'][0]], [0, 1], device)
    model.train()
    with torch.autocast('cuda', dtype=torch.float16):
        embeddings = model(**inputs)
        loss = (embeddings[0] * embeddings[1]).sum()
    loss.backward()
    if not torch.isfinite(loss) or any(value.grad is None or not torch.isfinite(value.grad).all() for value in model.parameters() if value.requires_grad):
        raise FloatingPointError('Full-size own model shape/gradient sanity failed')
    atomic_json(Path(cfg.run_dir) / 'model_report.json', {**model.parameter_report, 'finite_loss_and_all_active_gradients': True})
    del model, loss, embeddings, inputs
    torch.cuda.empty_cache()
''')


[Markdown Cell]

`lexical.py` — Sparse full-pool BM25 from provided text, with deterministic ties.

In [45]:
# [Code Cell]
module_parts.setdefault('lexical.py', []).append(r'''"""Full-pool BM25 from official text only; deterministic sparse CPU indexing."""
from collections import Counter
from pathlib import Path
import math
import re
import numpy as np
from scipy import sparse
from tqdm.auto import tqdm
from .io_utils import atomic_json, identity, read_json, records, sha256_file


def words(text):
    return re.findall(r'\w+', text.casefold(), flags=re.UNICODE)


def stable_order(scores):
    return np.argsort(-np.asarray(scores), kind='stable')


def lexical_cache(prepared, destination):
    prepared, root = Path(prepared), Path(destination)
    root.mkdir(parents=True, exist_ok=True)
    data = read_json(prepared / 'prepared_manifest.json')['identity']
    signature = identity({'version': 'pcr-bm25-v1', 'data': data, 'k1': 1.2, 'b': .75, 'qtf': 'log1p'})
    manifest_path = root / 'lexical_manifest.json'
    if manifest_path.is_file():
        manifest = read_json(manifest_path)
        if manifest['identity'] != signature or any(sha256_file(root / name) != value for name, value in manifest['files'].items()):
            raise ValueError('Lexical cache identity or file checksum changed')
        return {**manifest, 'directory': str(root)}
    files, splits = {}, {}
    for split in ('train', 'dev', 'test'):
        rows = list(records(prepared / (split + '_candidates.jsonl')))
        vocabulary, indptr, indices, values, lengths = {}, [0], [], [], []
        for row in tqdm(rows, desc='BM25 ' + split, unit='cases'):
            counts = Counter(words(row['text']))
            lengths.append(sum(counts.values()))
            for term, count in counts.items():
                column = vocabulary.setdefault(term, len(vocabulary))
                indices.append(column)
''')


In [46]:
# [Code Cell]
module_parts.setdefault('lexical.py', []).append(r'''                values.append(count)
            indptr.append(len(indices))
        matrix = sparse.csr_matrix((np.asarray(values, np.float32), np.asarray(indices, np.int32), np.asarray(indptr, np.int64)),
                                   shape=(len(rows), len(vocabulary)))
        frequency = np.bincount(matrix.indices, minlength=len(vocabulary))
        idf = np.log1p((len(rows) - frequency + .5) / (frequency + .5)).astype(np.float32)
        length = np.asarray(lengths, np.float32)
        denominator = 1.2 * (1 - .75 + .75 * length / max(float(length.mean()), 1.))
        matrix.data = matrix.data * 2.2 / (matrix.data + np.repeat(denominator, np.diff(matrix.indptr))) * idf[matrix.indices]
        queries = list(records(prepared / (split + '_queries.jsonl')))
        path = root / (split + '_bm25.bin')
        scores = np.memmap(path, np.float32, 'w+', shape=(len(queries), len(rows)))
        for first in tqdm(range(0, len(queries), 64), desc='Score BM25 ' + split, unit='blocks'):
            blocks = []
            for query in queries[first:first + 64]:
                counter = Counter(words(query['text']))
                columns = [vocabulary[term] for term in counter if term in vocabulary]
                counts = [math.log1p(counter[term]) for term in counter if term in vocabulary]
                blocks.append(sparse.csr_matrix((np.asarray(counts, np.float32), ([0] * len(columns), columns)), shape=(1, len(vocabulary))))
            query_matrix = sparse.vstack(blocks, format='csr')
            scores[first:first + len(blocks)] = (query_matrix @ matrix.T).toarray()
        scores.flush()
        del scores
        files[path.name] = sha256_file(path)
        splits[split] = {'query_ids': [row['id'] for row in queries], 'candidate_ids': [row['id'] for row in rows],
                         'shape': [len(queries), len(rows)]}
    manifest = {'version': 'pcr-bm25-v1', 'identity': signature, 'prepared_identity': data, 'files': files, 'splits': splits}
    atomic_json(manifest_path, manifest)
    return {**manifest, 'directory': str(root)}


def scores_for(manifest, split):
    return np.memmap(Path(manifest['directory']) / (split + '_bm25.bin'), np.float32, 'r', shape=tuple(manifest['splits'][split]['shape']))
''')


[Markdown Cell]

`model.py` — Own frozen foundation prefix -> adapted own top -> role-aware normalized retrieval vectors.

In [47]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''"""Frozen whole-case prefix cache -> own trainable top -> normalized retrieval vectors."""
from dataclasses import replace
import importlib
import types
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from .adaptation import adapt
from .parents import parent_class


def optimize_sparse(backbone):
    """Same sparse legal-key sets; avoid gathered K/V duplication for T <= 1024."""
    for attention in backbone.modules():
        if type(attention).__name__ != 'SparseCausalSelfAttention' or hasattr(attention, '_pcr_original_attend'):
            continue
        attention._pcr_original_attend = attention.attend
        attention._pcr_mask_cache = {}
        route = importlib.import_module(type(attention).__module__).sparse_key_indices
        def attend(self, query, kv, positions, key_valid=None, route_function=route):
            length = kv[0].shape[2]
            if length > 1024 or query.shape[2] != length:
                return self._pcr_original_attend(query, kv, positions, key_valid)
            signature = (query.device, length)
            mask = self._pcr_mask_cache.get(signature)
            if mask is None:
                indices, valid = route_function(torch.arange(length, device=query.device), length,
                    self.local_window, self.dilation, self.dilated_links, self.global_stride)
                counts = torch.zeros(length, length, dtype=torch.int32, device=query.device)
                counts.scatter_add_(1, indices, valid.to(torch.int32))
                mask = (counts > 0)[None, None]
                if len(self._pcr_mask_cache) >= 2:
                    self._pcr_mask_cache.clear()
                self._pcr_mask_cache[signature] = mask
            valid = mask if key_valid is None else mask & key_valid[:, None, None, :]
            value = F.scaled_dot_product_attention(query, kv[0], kv[1], attn_mask=valid,
                dropout_p=self.dropout if self.training else 0., is_causal=False)
            return self.out(value.transpose(1, 2).contiguous().flatten(2))
''')


In [48]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''        attention.attend = types.MethodType(attend, attention)


class CaseRetriever(nn.Module):
    """memory[B,M,D], valid[B,M], role[B] -> unit vectors[B,E].

    Own Phase 1 frozen bottom blocks encode every token once. Only their pooled
    outputs are cached. All adapted top blocks and learned heads execute again
    with gradients on each optimizer update; no stale trainable features are used.
    """
    def __init__(self, backbone, cfg, base_cfg):
        super().__init__()
        self.backbone, self.cfg, self.base_cfg = backbone, cfg, base_cfg
        self.changed = adapt(backbone, cfg)
        width = base_cfg.d_model
        self.position = nn.Linear(3, width, bias=False)
        self.role = nn.Embedding(2, width)
        nn.init.zeros_(self.position.weight)
        nn.init.zeros_(self.role.weight)
        self.pool_score = nn.Linear(width, 1)
        nn.init.zeros_(self.pool_score.weight)
        nn.init.zeros_(self.pool_score.bias)
        self.query_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, cfg.embedding_dim, bias=False))
        self.case_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, cfg.embedding_dim, bias=False))
        self.case_projection.load_state_dict(self.query_projection.state_dict())
        if cfg.family == 'b':
            # Decoder input prompts are actual trainable embeddings; encoder
            # outputs stay frozen and can therefore be cached exactly.
            self.query_prompts = nn.Parameter(torch.randn(cfg.prompt_tokens, width) * .02)

    def units(self):
        return self.backbone.stages if self.cfg.family == 'c' else self.backbone.layers

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()  # frozen prefix never receives stochastic dropout
        if mode:
            if self.cfg.family == 'b':
                self.backbone.decoder.train()
            else:
''')


In [49]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''                for module in list(self.units())[-self.cfg.train_last_units:]:
                    module.train()
        return self

    @torch.no_grad()
    def frozen_prefix(self, ids, mask, full):
        """Full source chunks[B,T] -> frozen contextual token features[B,T,D]."""
        if self.cfg.family == 'b':
            return self.backbone.encode(ids, mask)
        hidden = self.backbone.token_embedding(ids)
        if self.cfg.family == 'd':
            allowed = None if full else mask
        else:
            allowed = None
            if not full:
                causal = torch.ones(ids.shape[1], ids.shape[1], dtype=torch.bool, device=ids.device).tril()
                allowed = causal[None, None] & mask[:, None, None, :]
        for module in list(self.units())[:-self.cfg.train_last_units]:
            hidden = module(hidden, allowed)
        return hidden

    def _encode(self, memory, valid, role):
        length = memory.shape[1]
        position = torch.arange(length, device=memory.device)[None].float() / (valid.sum(-1) - 1).clamp_min(1)[:, None]
        positions = torch.stack((position, 1 - position, torch.ones_like(position)), dim=-1)
        hidden = memory.to(self.position.weight.dtype) + self.position(positions) + self.role(role)[:, None]
        if self.cfg.family == 'b':
            batch = len(memory)
            prompts = self.query_prompts[None].expand(batch, -1, -1)
            prompts = self.backbone.dropout(prompts + self.backbone.target_positions(torch.arange(prompts.shape[1], device=memory.device))[None])
            causal = torch.ones(prompts.shape[1], prompts.shape[1], dtype=torch.bool, device=memory.device).tril()[None, None]
            cross = valid[:, None, None, :]
            for layer in self.backbone.decoder:
                prompts = layer(prompts, hidden, causal, cross)
            hidden = self.backbone.decoder_norm(prompts)
            valid = torch.ones(hidden.shape[:2], dtype=torch.bool, device=memory.device)
        else:
            if self.cfg.family == 'd':
                allowed = valid
            else:
''')


In [50]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''                causal = torch.ones(length, length, dtype=torch.bool, device=memory.device).tril()
                allowed = causal[None, None] & valid[:, None, None, :]
            for module in list(self.units())[-self.cfg.train_last_units:]:
                hidden = module(hidden, allowed)
            hidden = self.backbone.final_norm(hidden)
        scores = self.pool_score(hidden).float().squeeze(-1).masked_fill(~valid, -1e4)
        weights = scores.softmax(-1).to(hidden.dtype)
        pooled = (hidden * weights[:, :, None]).sum(1)
        query = self.query_projection(pooled).float()
        case = self.case_projection(pooled).float()
        # Both projection graphs participate even for a padding-only role batch.
        projected = torch.where((role == 0)[:, None], query, case)
        return F.normalize(projected, p=2, dim=-1, eps=1e-6)

    def forward(self, memory, valid, role):
        values = []
        for first in range(0, len(memory), self.cfg.encode_batch_size):
            stop = first + self.cfg.encode_batch_size
            args = (memory[first:stop], valid[first:stop], role[first:stop])
            values.append(checkpoint(self._encode, *args, use_reentrant=False)
                          if self.training and self.cfg.checkpoint_encoder else self._encode(*args))
        return torch.cat(values)


def build_model(job, cfg, device, load_parent=True, enforce_boundary=True):
    Config, Model = parent_class(job['parent_source_dir'], cfg.family, job['parent_contract'])
    changes = {'activation_checkpointing': False}
    if cfg.family == 'c':
        changes['recurrent_checkpointing'] = False
    if cfg.family == 'd':
        changes['sparse_attention_checkpointing'] = False
    base_cfg = replace(Config(**job['parent']['config']), **changes)
    with torch.device('meta'):
        backbone = Model(base_cfg)
    if load_parent:
        file = Path(job['parent']['checkpoint'])
        if file.stat().st_size != job['parent']['checkpoint_bytes'] or file.stat().st_mtime_ns != job['parent']['checkpoint_mtime_ns']:
            raise ValueError('Foundation checkpoint changed after verification')
        payload = torch.load(file, map_location='cpu', weights_only=True, mmap=True)
        state = payload if job['parent'].get('format') == 'native_state_dict' else payload['model']
''')


In [51]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''        backbone.load_state_dict(state, strict=True, assign=True)
        del payload
    else:
        backbone._apply(lambda value: torch.empty_like(value, device=device) if value.is_meta else value.to(device))
    foundation = sum(value.numel() for value in backbone.parameters())
    if cfg.family == 'd':
        optimize_sparse(backbone)
    actual = CaseRetriever(backbone, cfg, base_cfg).to(device)
    total = sum(value.numel() for value in actual.parameters())
    active = sum(value.numel() for value in actual.parameters() if value.requires_grad)
    if enforce_boundary:
        assert 50_000_000 < foundation < 300_000_000
        assert 50_000_000 < total < 300_000_000 and 0 < active <= total
    capacity = base_cfg.max_source_tokens if cfg.family == 'b' else base_cfg.max_seq_tokens
    if cfg.chunk_tokens > capacity or cfg.memory_slots > capacity:
        raise ValueError('Source chunk or document memory exceeds own foundation context capacity')
    if cfg.family == 'b' and cfg.prompt_tokens > base_cfg.max_decoder_tokens:
        raise ValueError('Prompt length exceeds own B decoder position capacity')
    actual.parameter_report = {'foundation_unique_parameters': foundation, 'model_unique_parameters': total,
        'active_trainable_parameters': active, 'family': cfg.family, 'method': cfg.method,
        'own_foundation_reused': True, 'external_pretrained_weights': False, 'adapted': actual.changed,
        'student_decoder_pruned': cfg.method == 'kd' and len(backbone.decoder) < base_cfg.decoder_layers}
    return actual
''')


[Markdown Cell]

`native_contracts.py` — Reviewed source, configuration and ownership verification.

In [52]:
# [Code Cell]
module_parts.setdefault('native_contracts.py', []).append(r'''"""Reviewed own-foundation portable structural fingerprints; Python 3.10-3.13."""
APPROVED_NATIVE_SOURCES = [{'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '586cd48bdc420d8fed5db7b074ceba38491b8f091f8f28d702ed74983c000b63',
                      'model.py': '836d92b81250ca5d0e1a84c3df8181c1129e437bfea75c4f8653d2ebd3bf545b'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'bce55e18e1388720b701933df11a420783e56fe826a6780cfedce2551f468910',
                      'model.py': '43d74a2894b9bf74eee67f044a344557c39776a270491cd03a80e0440c44f1f4'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_40h/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '6b6759c5c8375120cd82882ba4a2c48ef7ab4160895379c5ec29144f29551a52',
                      'model.py': '8af20d2199c02557c5f4c7b7f3ddf08a8dd3f0e3941bcb75bcda21761bc6e88e'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_AutoResume_v3/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'b83d3fee697e9eee3861de2d5d0e386a5b07a3397e3b80073aa58c1bdd1b0150',
                      'loss.py': 'd803e39be8f8998383b5627e711dda1a427f95cf358af7a0a9a58519cb38e92c',
                      'model.py': '9a90daad6e2cb3544d94ad14fbaa2df5453fd15250e7a7aa2ee45c51a4cc1025',
                      'model_b.py': '80f26dd23889221032a5444b37457aae0419536b7c498de328dbd675d9f9fb15',
                      'model_c.py': '0e1f50425fdcde72c90745888212fe305e898dcd844c246bab6a669d44316158',
                      'model_d.py': '8e768f3e73f78cf3e69347eae5972956fcfd10005e6849f564fef3be265f5b7f'},
  'package': 'foundation_upgrade',
  'profile': 'Foundation_Upgrade/Models_BCD/runtime/foundation_upgrade',
  'schema': 'bcd-upgraded-foundation-v1'}]
''')


[Markdown Cell]

`native_parents.py` — Reviewed source, configuration and ownership verification.

In [53]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''"""Load reviewed own foundation exports directly, without fabricating old receipts."""
import importlib
import importlib.util
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file, identity
from .native_contracts import APPROVED_NATIVE_SOURCES
from .source_identity import ALGORITHM, source_fingerprint


def sources(directory: Path) -> dict:
    return {p.name: sha256_file(p) for p in sorted(directory.glob('*.py'))}


def validate_source(directory: Path, family: str, contract: dict) -> None:
    """Check exact saved bytes AND a reviewed core implementation before import."""
    actual = sources(directory)
    if actual != contract['source_sha256']:
        raise ValueError('Native foundation source snapshot changed or is incomplete')
    expected_schema = 'a-upgraded-foundation-v1' if family == 'a' else 'bcd-upgraded-foundation-v1'
    if contract.get('schema') != expected_schema or contract.get('family') != family:
        raise ValueError('Native foundation family/schema mismatch')
    mismatches = []
    for approved in APPROVED_NATIVE_SOURCES:
        if approved['schema'] != expected_schema:
            continue
        try:
            ast_hashes = {name: source_fingerprint((directory / name).read_text(encoding='utf8'))
                          for name in approved['core_ast_sha256']}
            if ast_hashes == approved['core_ast_sha256']:
                return
            mismatches.append({'profile': approved.get('profile', approved['package']),
                               'different_modules': [n for n in ast_hashes if ast_hashes[n] != approved['core_ast_sha256'][n]],
                               'actual_core_identity': ast_hashes})
        except (OSError, SyntaxError, UnicodeError):
            continue
    raise ValueError('Unrecognized native model implementation after portable structural comparison. '
                     f'Algorithm={ALGORITHM}; differences={mismatches}. Preserve source files for review; do not bypass checks.')
''')


In [54]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''

def native_class(directory: str, family: str, contract: dict):
    directory = Path(directory)
    validate_source(directory, family, contract)
    package = 'verified_native_' + family + '_' + identity(contract['source_sha256'])[:16]
    if package not in sys.modules:
        spec = importlib.util.spec_from_file_location(package, directory / '__init__.py',
                                                       submodule_search_locations=[str(directory)])
        module = importlib.util.module_from_spec(spec)
        sys.modules[package] = module
        spec.loader.exec_module(module)
    configuration = importlib.import_module(package + '.config')
    implementation = importlib.import_module(package + '.model')
    return configuration.TrainConfig, getattr(implementation, contract['class_name'])


def verify_export(root: Path, cfg) -> tuple[Path, dict, dict]:
    """Verify actual portable weights and tokenizer; no LM binaries or old metadata required."""
    manifest_path = root / 'foundation_manifest.json'
    manifest = read_json(manifest_path)
    schema = manifest.get('schema')
    family = 'a' if schema == 'a-upgraded-foundation-v1' else manifest.get('family')
    if family != cfg.family or schema not in ('a-upgraded-foundation-v1', 'bcd-upgraded-foundation-v1'):
        raise ValueError('Export belongs to another model family or unsupported schema')
    if (manifest.get('status') != 'exported' or manifest.get('training_run_status') != 'complete'
            or manifest.get('full_epoch_completed') is not True or manifest.get('completed_full_epochs', 0) < 1):
        raise ValueError('Finish 03 and run 04 with a complete full epoch before using this foundation')
    if manifest.get('selected_snapshot_has_new_updates') is not True or manifest.get('best_snapshot_step', 0) < 1:
        raise ValueError('This export selected the initial baseline; no upgraded weight updates were selected')
    weights, config, token = root / 'model_weights.pt', root / 'model_config.json', root / 'tokenizer.json'
    if sha256_file(weights) != manifest['model_weights_sha256'] or sha256_file(token) != manifest['tokenizer_sha256']:
        raise ValueError('Native export weights/tokenizer SHA-256 mismatch')
    if manifest.get('model_config_sha256') and sha256_file(config) != manifest['model_config_sha256']:
        raise ValueError('Native export model configuration SHA-256 mismatch')
    values = read_json(config)
    if family != 'a' and values.get('family') != family:
        raise ValueError('Model configuration belongs to another family')
    package = 'a_upgrade' if family == 'a' else 'foundation_upgrade'
    directory = root / 'code' / package
''')


In [55]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''    code_hashes = sources(directory)
    if manifest.get('source_sha256') and manifest['source_sha256'] != code_hashes:
        raise ValueError('Source files differ from the native export manifest')
    contract = {'kind': 'native_foundation_export', 'family': family, 'schema': schema,
                'source_sha256': code_hashes, 'class_name': 'LegalCausalDecoder' if family == 'a' else 'build_model',
                'data_fingerprint': identity({'export_manifest': sha256_file(manifest_path),
                                              'tokenizer': manifest['tokenizer_sha256']}),
                'export_root': str(root), 'source_subdir': 'code/' + package}
    Config, Model = native_class(str(directory), family, contract)
    if values.get('vocab_size') != 32768:
        raise ValueError('Unexpected native vocabulary size')
    if family == 'a' and any(values.get(k) != v for k, v in
                             {'d_model': 768, 'n_heads': 12, 'n_layers': 12, 'd_ff': 2048,
                              'rms_epsilon': 1e-6, 'rope_base': 10000.0, 'max_seq_tokens': 2048}.items()):
        raise ValueError('Model A export differs from the reviewed 110M architecture')
    tokenizer = Tokenizer.from_file(str(token))
    if tokenizer.get_vocab_size() != values['vocab_size'] or [tokenizer.token_to_id(t) for t in
            ('<pad>', '<s>', '</s>', '<unk>', '<sep>', '<mask>')] != list(range(6)):
        raise ValueError('Tokenizer vocabulary/special IDs differ')
    if cfg.tokenizer_path and sha256_file(Path(cfg.tokenizer_path)) != manifest['tokenizer_sha256']:
        raise ValueError('TOKENIZER_PATH differs from the selected native foundation tokenizer')
    with torch.device('meta'):
        model = Model(Config(**values))
    state = torch.load(weights, map_location='cpu', weights_only=True, mmap=True)
    if not isinstance(state, dict) or not all(isinstance(t, torch.Tensor) for t in state.values()):
        raise ValueError('model_weights.pt must be the raw 04 state dictionary. Do not rename a 03 best.pt recovery payload.')
    for name, tensor in state.items():
        if tensor.is_floating_point() and (tensor.dtype != torch.float32 or
                not all(bool(torch.isfinite(chunk).all()) for chunk in tensor.reshape(-1).split(1_000_000))):
            raise ValueError('Nonfinite/non-FP32 native master weight: ' + name)
    model.load_state_dict(state, strict=True, assign=True)
    count = sum(p.numel() for p in model.parameters())
    if not 50_000_000 < count < 300_000_000 or count != manifest['unique_parameters']:
        raise ValueError('Native model parameter count differs from manifest/boundary')
    del model, state
    result = {'family': family, 'config': values, 'format': 'native_state_dict',
              'completed_epochs': manifest['completed_full_epochs'], 'planned_epochs': values['epochs'],
              'checkpoint': str(weights), 'checkpoint_sha256': manifest['model_weights_sha256'],
              'checkpoint_bytes': weights.stat().st_size, 'checkpoint_mtime_ns': weights.stat().st_mtime_ns,
              'checkpoint_epoch': manifest['best_snapshot_epoch'], 'checkpoint_step': manifest['best_snapshot_step'],
''')


In [56]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''              'selection': '04 exported development-best snapshot', 'data_fingerprint': contract['data_fingerprint'],
              'source_sha256': code_hashes, 'tokenizer_sha256': manifest['tokenizer_sha256'],
              'metadata_sha256': sha256_file(manifest_path), 'tokenizer_receipt_name': 'foundation_manifest.json',
              'source_checkpoint_sha256': manifest['source_checkpoint_sha256'],
              'foundation_manifest_sha256': sha256_file(manifest_path), 'native_contract': contract,
              'original_fingerprint': None, 'original_torch': None, 'checkpoint_warnings': []}
    return root, result, contract


def select_export(cfg):
    """Prefer a verified native export, otherwise let the original verified loader run."""
    base = Path(cfg.parent_run_dir) if cfg.parent_run_dir else Path(cfg.input_root)
    paths = [base / 'foundation_manifest.json'] if (base / 'foundation_manifest.json').is_file() else sorted(base.rglob('foundation_manifest.json'))
    accepted, diagnostics, relevant = [], [], False
    for path in paths:
        value = read_json(path)
        family = 'a' if value.get('schema') == 'a-upgraded-foundation-v1' else value.get('family')
        if family != cfg.family:
            continue
        relevant = True
        row = {'run_dir': str(path.parent), 'format': 'native_foundation_export', 'accepted': False}
        try:
            root, parent, contract = verify_export(path.parent, cfg)
            row.update(accepted=True, checkpoint_sha256=parent['checkpoint_sha256'])
            accepted.append((root, parent))
        except (OSError, ValueError, KeyError, TypeError, RuntimeError, SyntaxError, AssertionError) as error:
            row['reason'] = f'{type(error).__name__}: {error}'
        diagnostics.append(row)
    if not relevant:
        return None
    atomic_json(Path(cfg.run_dir) / 'parent_input_report.json', {'native_candidates': diagnostics,
                'source_identity_algorithm': ALGORITHM, 'loader_python': sys.version.split()[0]})
    print({'native_parent_candidates': diagnostics}, flush=True)
    distinct = {(p['checkpoint_sha256'], identity(p['source_sha256']), p['tokenizer_sha256']): (r, p) for r, p in accepted}
    if len(distinct) != 1:
        raise ValueError(f'Found {len(distinct)} distinct valid upgraded {cfg.family.upper()} exports. '
                         'Attach the COMPLETE 04 output with actual model_weights.pt and code, '
                         'or set PARENT_RUN_DIR to its exact foundation folder. No fallback to old weights was attempted. '
                         'See parent_input_report.json.')
    return next(iter(distinct.values()))
''')


[Markdown Cell]

`official_evaluator.py` — Unmodified organizer-provided local evaluator.

In [57]:
# [Code Cell]
module_parts.setdefault('official_evaluator.py', []).append(r'''#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc
''')


In [58]:
# [Code Cell]
module_parts.setdefault('official_evaluator.py', []).append(r'''

def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)
''')


In [59]:
# [Code Cell]
module_parts.setdefault('official_evaluator.py', []).append(r'''

def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")
    p.add_argument("--prediction",type=Path,required=True)
''')


In [60]:
# [Code Cell]
module_parts.setdefault('official_evaluator.py', []).append(r'''    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")
    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")
    return p.parse_args()


def main():
    a=parse_args()
    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)
    errs=validate(pred,gold,candidates)
    if errs:
        print("LOCAL VALIDATION FAILED")
        for i,e in enumerate(errs,1): print(f"  {i}. {e}")
        print(f"Total validation errors: {len(errs)}")
        raise SystemExit(1)
    result=score(gold,pred)
    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}
    print("Validation passed.")
    print(f"Micro-F1@7: {scores['primary_metric_1']:.10f}")
    print(f"Micro-Precision@7: {scores['secondary_metric_1']:.10f}")
    print(f"Micro-Recall@7: {scores['secondary_metric_2']:.10f}")
    print(f"TP={result['tp']} FP={result['fp']} FN={result['fn']}")
    if a.output:
        a.output.parent.mkdir(parents=True,exist_ok=True)
        with a.output.open('w',encoding='utf-8') as f: json.dump(scores,f,indent=2); f.write('\n')
        print(f"Wrote {a.output}")

if __name__=='__main__': main()
''')


[Markdown Cell]

`p6_contract.py` — Reviewed source, configuration and ownership verification.

In [61]:
# [Code Cell]
module_parts.setdefault('p6_contract.py', []).append(r'''APPROVED_PHASE6_SOURCES = {'__init__.py': 'c058812345f3788951feab67dd3a5d92996eedb96390900a013170238004049f', 'adaptation.py': 'd2c048dc8e024e49a90f2abf83bcbbdd8c9a2fbf7678d2b8eacf3308b844d718', 'compare.py': 'cb7b8304a1836007355455fc61d17cbd937e03576d7ae9be35e091acb9d95121', 'config.py': 'd9c8b3b2ad89f25e8755b8c6228e05cbcbce01fb3365da851ed845e489775dce', 'data.py': 'bbfb5a9cb0eb56a0a2a73d5870d3850200f41e16f49571094f97752f33ba514d', 'evaluation.py': '3a38aa9eab4c9372a1e3638c1fae5d7ffc84d39d9277b5bf5f40e3acc65a1649', 'experiments.py': '2e068ad8dde6fbd6ed0e8883dd2c5daa80b2c72a0d2044939b9c523f0d773950', 'generation.py': '7ddb2b6e10af8e911bf5b515f01da8cd1fcbd2e391a37a2805d75ab53185814e', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'jobs.py': 'e771c0de302c9a1c6c892f6077c534871262e4872befa719039882e95d568d80', 'model.py': 'dcb3c551618754e2196bedb3271335e29ffb4332779d568c4cd1ea81f148ce6f', 'native_contracts.py': 'f76cae1b17467620d65bcdd85d0aba6d15b5bf8cf7f8e4a8b30580b7951e368c', 'native_parents.py': '1c77968d9f566a65e4508d3cb590fe08fba058c7490de2e13baf3165095170ca', 'parents.py': '48ea34b6c454d31337d9b38c08556816b14af3ad48cd02430bedc51be0f83087', 'predict.py': 'c417b8690f4aa4325d03c74aaecc2e17eb2417460aabbd443da7880385c6cc2e', 'preparation.py': 'c17f209343cbbe0bca5cf093a602f0e6e66361518e891f3561f7f19825bf7d51', 'schema.py': 'edca4639fc4f0afe29bf63b3990c7ebdc9da9898ead67f347066261393fe1b95', 'screen.py': 'f643d5e00ba7fcf5c12f1a44fdb72b016f9aea41055ca95d2ccfc11fa1752aa6', 'source_identity.py': 'bff746c41ac050d03b1b02aee9d6e709c334697ad9ff6e69af4edacd0923690c', 'state.py': '7d2d65b3686fb26d27efae65b4b478fc39c59c7e311ebe2a87da7ac5c261223d', 'train.py': '94d1c3eb0c2a307daaffa5c9b261545b024fc7550f5b72d081b55c8f7c264ec2', 'workflow.py': '4d3d108abe6e2c006d8e47ea9fac909131d414a97e8c83711f3ab3410ca5d577'}
''')


[Markdown Cell]

`parents.py` — Reviewed source, configuration and ownership verification.

In [62]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
''')


In [63]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
''')


In [64]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    from .native_parents import select_export
    native = select_export(cfg)
    if native is not None:
        return native
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json. '
                         'For upgraded models attach the complete 04 foundation export with model_weights.pt, '
                         'model_config.json, tokenizer.json, foundation_manifest.json and code; a 03 recovery run is not that export.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
''')


In [65]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting Phase 7.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    if contract.get('kind') == 'native_foundation_export':
        root = Path(contract['export_root'])
        manifest = read_json(root / 'foundation_manifest.json')
        token, receipt = root / 'tokenizer.json', root / 'foundation_manifest.json'
        if sha256_file(token) != manifest['tokenizer_sha256']:
''')


In [66]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''            raise ValueError('Native tokenizer changed after foundation verification')
        return token, receipt, {'tokenizer_sha256': manifest['tokenizer_sha256'],
                                'metadata_sha256': sha256_file(receipt),
                                'tokenizer_receipt_name': 'foundation_manifest.json'}
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    if contract.get('kind') == 'native_foundation_export':
        from .native_parents import native_class
        return native_class(directory, family, contract)
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
''')


In [67]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

`preparation.py` — Official query/candidate files, original IDs, complete gold sets and split-pool audit.

In [68]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''"""Streaming official PCR parsing, split-specific pools and strict auditable artifacts."""
from pathlib import Path
import hashlib
import json
import re
import shutil
import sqlite3
import unicodedata
import zipfile
from .io_utils import atomic_json, identity, read_json, sha256_file, records

FILES = {f'{split}_{kind}': f'Phase_7_PCR/{folder}/{split}{"_candidates" if kind == "candidates" else ""}.json'
         for split, folder in (('train', 'Train'), ('dev', 'Dev'), ('test', 'Test'))
         for kind in ('queries', 'candidates')}


def flatten(value):
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return '\n'.join(flatten(part) for part in value)
    if isinstance(value, dict) and 'text' in value:
        return flatten(value['text'])
    raise ValueError('Case text must be a string or ordered sentence list; set FIELD_MAP for another field')


def identifier(value):
    if isinstance(value, int) and not isinstance(value, bool):
        value = str(value)
    if not isinstance(value, str) or not value.strip():
        raise ValueError('A nonempty original case ID is required')
    return value  # keep original strings and leading zeroes exactly


def stream_rows(path, query, labelled, layout=None, fields=None):
    """Map-of-cases or array; optional explicit ijson prefix supports wrappers."""
    import ijson
    fields = fields or {}
''')


In [69]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''    with Path(path).open('rb') as stream:
        head = stream.read(4096).lstrip(b'\xef\xbb\xbf \r\n\t')[:1]
    mode, prefix = (layout or {}).get('mode'), (layout or {}).get('prefix', '')
    mode = mode or ('array' if head == b'[' else 'mapping')
    with Path(path).open('rb') as stream:
        if stream.read(3) != b'\xef\xbb\xbf':
            stream.seek(0)
        array_prefix = prefix if prefix.endswith('.item') or prefix == 'item' else (prefix + '.item').lstrip('.')
        iterator = ijson.items(stream, array_prefix) if mode == 'array' else ijson.kvitems(stream, prefix)
        for number, entry in enumerate(iterator):
            outer, row = (None, entry) if mode == 'array' else entry
            if isinstance(row, (str, list)) and not query:
                row = {'text': row}
            if not isinstance(row, dict):
                raise ValueError(f'{path}: record {number} must be a case object; set SOURCE_LAYOUTS')
            own_id = row.get(fields.get('id', 'id'), row.get('case_id', outer))
            case_id = identifier(own_id)
            if outer is not None:
                outer_id = identifier(outer)
                same_number = outer_id.isdigit() and case_id.isdigit() and len(outer_id) < 30 and len(case_id) < 30 and int(outer_id) == int(case_id)
                if outer_id != case_id and not same_number:
                    raise ValueError('Conflicting map key/inner case ID: ' + str(outer))
                case_id = outer_id  # mapping keys define the official ID namespace
            key = fields.get('text', 'text')
            if key not in row:
                raise ValueError('Missing text field for ' + case_id + '; configure FIELD_MAP')
            result = {'id': case_id, 'text': flatten(row[key])}
            if query and labelled:
                key = fields.get('labels', 'relevant_candidates')
                labels = row.get(key)
                if not isinstance(labels, list):
                    raise ValueError('Labelled query needs relevant_candidates list: ' + case_id)
                # Same blank-label behavior as the provided official evaluator.
                clean, seen = [], set()
                for value in labels:
                    if not isinstance(value, str) or not value.strip():
                        continue
                    if value in seen:
                        raise ValueError('Duplicate gold candidate ID for ' + case_id + ': ' + value)
                    clean.append(value)
''')


In [70]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''                    seen.add(value)
                result['relevant_candidates'] = clean
                result['blank_or_nonstring_labels_ignored'] = len(labels) - len(clean)
            # Test annotations are never selected into the prepared representation.
            yield result


def download(repository, revision, output, token, path_map=None):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        commit = api.dataset_info(repository, revision=revision).sha
        available = set(api.list_repo_files(repository, repo_type='dataset', revision=commit))
    except Exception as error:
        raise RuntimeError('Enable Kaggle Internet, HF_TOKEN/HF Secret and official repository access') from error
    selected = {key: (path_map or {}).get(key, value) for key, value in FILES.items()}
    if set(selected.values()) - available:
        raise ValueError('Official Phase 7 paths changed; configure HF_PATH_MAP from the official tree')
    paths = {key: Path(hf_hub_download(repository, name, repo_type='dataset', revision=commit,
              token=token, local_dir=str(output))) for key, name in selected.items()}
    return paths, {'repository': repository, 'commit': commit, 'paths': selected}


def prepare(paths, output, provenance, seed=781, tune_fraction=.75, layouts=None, field_map=None):
    root = Path(output)
    root.mkdir(parents=True, exist_ok=True)
    choices = {'raw_sha256': {key: sha256_file(file) for key, file in paths.items()},
        'provenance': provenance, 'seed': seed, 'tune_fraction': tune_fraction,
        'layouts': layouts or {}, 'field_map': field_map or {}}
    if not 0 < tune_fraction < 1 or set(paths) != set(FILES):
        raise ValueError('Supply all six official query/candidate files and a tune fraction in (0,1)')
    if (root / 'prepared_manifest.json').is_file():
        saved = verify_prepared(root)
        if saved['choices'] != choices:
            raise ValueError('Data preparation choices changed; use a new output folder')
        return saved
    db = sqlite3.connect(root / 'audit.sqlite3')
    db.execute('DROP TABLE IF EXISTS cases')
    db.execute('CREATE TABLE cases (split TEXT, kind TEXT, id TEXT, content_hash TEXT, group_hash TEXT, PRIMARY KEY(split,kind,id))')
    audit = {'splits': {}, 'test_labels_used': False, 'all_query_and_candidate_rows_retained': True,
''')


In [71]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''             'gold_sets_truncated_or_padded': False, 'IDs_zero_padded_or_rewritten': False}
    artifacts, ids_by_split, dev_groups = {}, {}, {}
    try:
        for split in ('train', 'dev', 'test'):
            audit['splits'][split] = {}
            ids_by_split[split] = {}
            for kind in ('candidates', 'queries'):
                key = split + '_' + kind
                file = root / (key + '.jsonl')
                temporary = file.with_suffix('.jsonl.partial')
                ids, empty, blank_labels = [], 0, 0
                with temporary.open('w', encoding='utf-8', newline='\n') as stream:
                    for row in stream_rows(paths[key], kind == 'queries', split != 'test',
                            (layouts or {}).get(key), (field_map or {}).get(key)):
                        text = row['text']
                        raw_hash = hashlib.sha256(text.encode()).hexdigest()
                        group_hash = hashlib.sha256(' '.join(unicodedata.normalize('NFKC', text).casefold().split()).encode()).hexdigest()
                        try:
                            db.execute('INSERT INTO cases VALUES (?,?,?,?,?)', (split, kind, row['id'], raw_hash, group_hash))
                        except sqlite3.IntegrityError as error:
                            raise ValueError(f'Duplicate original ID in {key}: {row["id"]}') from error
                        row['content_sha256'] = raw_hash
                        blank_labels += row.pop('blank_or_nonstring_labels_ignored', 0)
                        empty += int(not text.strip())
                        if split == 'dev' and kind == 'queries':
                            dev_groups[row['id']] = group_hash
                        ids.append(row['id'])
                        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
                temporary.replace(file)
                if not ids or (kind == 'candidates' and len(ids) < 7):
                    raise ValueError(f'{key}: need nonempty queries and at least seven candidate cases')
                ids_by_split[split][kind] = ids
                audit['splits'][split][kind] = {'rows': len(ids), 'empty_text_rows_retained': empty,
                                               'blank_gold_values_ignored': blank_labels}
                if kind == 'candidates':
                    atomic_json(root / (split + '_candidate_ids.json'), ids)
            pool = set(ids_by_split[split]['candidates'])
            if split != 'test':
                truth = {row['id']: row['relevant_candidates'] for row in records(root / (split + '_queries.jsonl'))}
                missing = {key: [value for value in labels if value not in pool] for key, labels in truth.items()}
''')


In [72]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''                missing = {key: value for key, value in missing.items() if value}
                audit['splits'][split].update(gold_total=sum(map(len, truth.values())),
                    empty_gold_queries=sum(not value for value in truth.values()), unreachable_gold=missing,
                    reachable_gold=sum(sum(value in pool for value in labels) for labels in truth.values()))
                max_hits = sum(min(7, sum(value in pool for value in labels)) for labels in truth.values())
                audit['splits'][split]['theoretical_micro_f1_ceiling_at7'] = 2 * max_hits / max(1, 7 * len(truth) + sum(map(len, truth.values())))
                # Complete official gold sets remain in local evaluation, even if unreachable.
                atomic_json(root / (split + '_truth.json'), truth)
        groups = sorted(set(dev_groups.values()), key=lambda value: hashlib.sha256((str(seed) + value).encode()).hexdigest())
        cut = min(len(groups) - 1, max(1, round(len(groups) * tune_fraction))) if len(groups) >= 4 else len(groups)
        tune_groups = set(groups[:cut])
        tune_ids = [key for key, group in dev_groups.items() if group in tune_groups]
        confirm_ids = [key for key, group in dev_groups.items() if group not in tune_groups]
        atomic_json(root / 'selection_split.json', {'tune_ids': tune_ids, 'confirm_ids': confirm_ids,
                    'official_dev_all_ids_retained': True, 'confirmation_available': bool(confirm_ids)})
        overlap = db.execute("SELECT COUNT(DISTINCT d.id) FROM cases d JOIN cases t ON d.group_hash=t.group_hash WHERE d.split='dev' AND d.kind='queries' AND t.split='train' AND t.kind='queries'").fetchone()[0]
        audit['dev_query_texts_seen_in_training'] = overlap
        audit['candidate_pool_ID_overlap'] = {a + '_' + b: len(set(ids_by_split[a]['candidates']) & set(ids_by_split[b]['candidates']))
            for a, b in (('train', 'dev'), ('train', 'test'), ('dev', 'test'))}
        audit['tune_queries'], audit['confirmation_queries'] = len(tune_ids), len(confirm_ids)
        atomic_json(root / 'data_audit.json', audit)
        names = [f'{split}_{kind}.jsonl' for split in ('train', 'dev', 'test') for kind in ('queries', 'candidates')]
        names += [f'{split}_candidate_ids.json' for split in ('train', 'dev', 'test')]
        names += ['train_truth.json', 'dev_truth.json', 'selection_split.json', 'data_audit.json']
        for name in names:
            artifacts[name] = {'bytes': (root / name).stat().st_size, 'sha256': sha256_file(root / name)}
        body = {'version': 'pcr-data-v1', 'status': 'complete', 'choices': choices, 'artifacts': artifacts}
        manifest = {**body, 'identity': identity(body)}
        atomic_json(root / 'prepared_manifest.json', manifest)
        return manifest
    finally:
        db.close()


def verify_prepared(root):
    root = Path(root)
    value = read_json(root / 'prepared_manifest.json')
    if value.get('version') != 'pcr-data-v1' or value.get('status') != 'complete' or identity({k: v for k, v in value.items() if k != 'identity'}) != value['identity']:
        raise ValueError('Attach completed P7_00 prepared data, not data from another phase')
    for name, item in value['artifacts'].items():
''')


In [73]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''        if Path(name).name != name or (root / name).stat().st_size != item['bytes'] or sha256_file(root / name) != item['sha256']:
            raise ValueError('Missing/changed prepared artifact: ' + name)
    return value


def prepared_input(cfg):
    paths = [Path(cfg.prepared_dir)] if cfg.prepared_dir else [file.parent for file in Path(cfg.input_root).rglob('prepared_manifest.json')
        if read_json(file).get('version') == 'pcr-data-v1']
    archives = [file for file in paths if file.is_file() and file.suffix == '.zip']
    paths = [file for file in paths if file not in archives]
    if not paths and not archives:
        archives = list(Path(cfg.input_root).rglob('P7_Prepared_Data.zip'))
    for file in archives:
        with zipfile.ZipFile(file) as archive:
            names = [name for name in archive.namelist() if name.endswith('prepared_manifest.json')]
            if len(names) != 1:
                raise ValueError('Prepared ZIP must contain exactly one manifest')
            prefix = names[0][:-len('prepared_manifest.json')]
            manifest = json.loads(archive.read(names[0]))
            if manifest.get('version') != 'pcr-data-v1' or identity({k: v for k, v in manifest.items() if k != 'identity'}) != manifest['identity']:
                raise ValueError('Prepared ZIP manifest identity is invalid')
            destination = Path(cfg.run_dir) / 'prepared_input'
            destination.mkdir(parents=True, exist_ok=True)
            for name in [*manifest['artifacts'], 'prepared_manifest.json']:
                if Path(name).name != name:
                    raise ValueError('Unsafe prepared ZIP path')
                if archive.getinfo(prefix + name).file_size > 2 << 30:
                    raise ValueError('Prepared ZIP member exceeds expected size')
                with archive.open(prefix + name) as incoming, (destination / name).open('wb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing, 1 << 20)
            paths.append(destination)
    if not paths:
        raise FileNotFoundError('Attach complete P7_00 output or P7_Prepared_Data.zip')
    choices = [(path, verify_prepared(path)) for path in paths]
    if len({manifest['identity'] for _, manifest in choices}) != 1:
        raise ValueError('Several Phase 7 data versions differ; set PREPARED_DIR')
    return choices[0]
''')


[Markdown Cell]

`ranking.py` — Full-pool scoring, Micro-F1@7, tune-only fusion and strict prediction ZIP validation.

In [74]:
# [Code Cell]
module_parts.setdefault('ranking.py', []).append(r'''"""Full-pool vector scoring, tune-only lexical/neural fusion and official top-seven ZIPs."""
from pathlib import Path
import json
import zipfile
import numpy as np
import torch
import torch.distributed as dist
from tqdm.auto import tqdm
from .io_utils import atomic_json, read_json, sha256_file, identity
from .lexical import scores_for, stable_order
from .official_evaluator import validate, score


@torch.no_grad()
def embed(model, store, split, kind, cfg, device, rank=0, world=1):
    reference = store.catalog['references'][split][kind]
    documents = reference['document_indices']
    vectors = torch.zeros(len(documents), cfg.embedding_dim, device=device, dtype=torch.float32)
    owned = list(range(rank, len(documents), world))
    role = 0 if kind == 'queries' else 1
    model.eval()
    for first in tqdm(range(0, len(owned), cfg.encode_batch_size), desc=f'{split} {kind}, GPU 0',
                      disable=rank != 0, unit='batches'):
        indices = owned[first:first + cfg.encode_batch_size]
        batch = store.batch([documents[index] for index in indices], [role] * len(indices), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            vectors[indices] = model(**batch).float()
    if world > 1:
        dist.all_reduce(vectors, op=dist.ReduceOp.SUM)
    if not torch.isfinite(vectors).all():
        raise FloatingPointError('Nonfinite retrieval vectors')
    return vectors


def dense_scores(query, candidates, destination=None):
    shape = (len(query), len(candidates))
    matrix = np.memmap(destination, np.float32, 'w+', shape=shape) if destination else np.empty(shape, np.float32)
    for first in range(0, len(query), 128):
        matrix[first:first + 128] = (query[first:first + 128].float() @ candidates.float().T).cpu().numpy()
    if hasattr(matrix, 'flush'):
''')


In [75]:
# [Code Cell]
module_parts.setdefault('ranking.py', []).append(r'''        matrix.flush()
    return matrix


def inverse_rank(values):
    order = stable_order(values)
    ranks = np.empty(len(order), np.int64)
    ranks[order] = np.arange(1, len(order) + 1)
    return ranks


def predictions(dense, lexical, query_ids, candidate_ids, neural_weight=1., rank_constant=60):
    if dense.shape != lexical.shape or dense.shape != (len(query_ids), len(candidate_ids)) or len(candidate_ids) < 7:
        raise ValueError('Ranking matrices/IDs do not describe this split candidate pool')
    result = {}
    for index, query in enumerate(query_ids):
        if neural_weight == 1.:
            values = dense[index]
        elif neural_weight == 0.:
            values = lexical[index]
        else:
            values = neural_weight / (rank_constant + inverse_rank(dense[index])) + (1 - neural_weight) / (rank_constant + inverse_rank(lexical[index]))
        result[query] = [candidate_ids[int(position)] for position in stable_order(values)[:7]]
    return result


def metrics(truth, prediction, candidate_ids, subset=None):
    keys = set(subset) if subset is not None else set(truth)
    if not keys:
        return None
    gold = {key: value for key, value in truth.items() if key in keys}
    rows = {key: prediction[key] for key in gold}
    errors = validate(rows, gold, set(candidate_ids))
    if errors:
        raise ValueError('Official retrieval validation failed: ' + '; '.join(errors[:10]))
    values = score(gold, rows)
    return {'micro_f1_at7': values['primary_metric_1'], 'micro_precision_at7': values['secondary_metric_1'],
            'micro_recall_at7': values['secondary_metric_2'], 'tp': values['tp'], 'fp': values['fp'], 'fn': values['fn'],
            'queries': len(gold)}

''')


In [76]:
# [Code Cell]
module_parts.setdefault('ranking.py', []).append(r'''
def select_fusion(dense, lexical, query_ids, candidate_ids, truth, selection):
    trials = []
    # Alpha=0 is a separate diagnostic lexical baseline. Required method outputs
    # keep a real nonzero learned-model contribution, including when BM25 is strong.
    for alpha in (.1, .25, .5, .75, 1.):
        for constant in ((20, 60) if alpha < 1 else (60,)):
            rows = predictions(dense, lexical, query_ids, candidate_ids, alpha, constant)
            evaluated = metrics(truth, rows, candidate_ids, selection['tune_ids'])
            trials.append({'neural_weight': alpha, 'rank_constant': constant, 'tune': evaluated})
    winner = max(trials, key=lambda row: (row['tune']['micro_f1_at7'], row['neural_weight']))
    baseline = predictions(dense, lexical, query_ids, candidate_ids, 0.)
    chosen = predictions(dense, lexical, query_ids, candidate_ids, winner['neural_weight'], winner['rank_constant'])
    return {'rule': {key: winner[key] for key in ('neural_weight', 'rank_constant')}, 'tune': winner['tune'],
        'confirmation': metrics(truth, chosen, candidate_ids, selection['confirm_ids']),
        'official_dev': metrics(truth, chosen, candidate_ids),
        'bm25_baseline_tune': metrics(truth, baseline, candidate_ids, selection['tune_ids']),
        'trials': trials, 'selected_on': 'official_dev_tune_partition', 'confirmation_used_for_selection': False}


def package(root, prediction, test_ids, candidate_ids, name):
    errors = validate(prediction, {key: [] for key in test_ids}, set(candidate_ids))
    if errors:
        raise ValueError('Submission rejected by official validation: ' + '; '.join(errors[:10]))
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    target = root / 'prediction.json'
    atomic_json(target, prediction)
    archive = root / ('submission_' + name + '.zip')
    temporary = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as stream:
        stream.write(target, 'prediction.json')
    temporary.replace(archive)
    with zipfile.ZipFile(archive) as stream:
        if stream.testzip() is not None or stream.namelist() != ['prediction.json'] or stream.read('prediction.json') != target.read_bytes():
            raise ValueError('Submission archive failed integrity/layout checks')
    return {'zip_path': str(archive), 'prediction_path': str(target), 'zip_sha256': sha256_file(archive),
            'prediction_sha256': sha256_file(target), 'queries': len(prediction), 'exactly_seven_unique_valid_IDs': True}
''')


[Markdown Cell]

`source_identity.py` — Reviewed source, configuration and ownership verification.

In [77]:
# [Code Cell]
module_parts.setdefault('source_identity.py', []).append(r'''"""Portable structural source identity, independent of ast.dump display changes."""
import ast
import hashlib
import json

ALGORITHM = 'python-canonical-ast-v1'


def canonical_tree(value):
    """Keep meaningful node fields; omit absent/default optional fields consistently.

    Python 3.12 added empty type_params; Python 3.13 changed ast.dump's default
    omission of empty lists. We never hash that debugging representation. Real
    nonempty fields, including generic type parameters, remain in the identity.
    Source locations, whitespace and comments do not affect model semantics.
    """
    if isinstance(value, ast.AST):
        fields = {}
        for name, item in ast.iter_fields(value):
            if item is None or (isinstance(item, list) and not item):
                continue
            fields[name] = canonical_tree(item)
        return {'node': type(value).__name__, 'fields': fields}
    if isinstance(value, list):
        return [canonical_tree(item) for item in value]
    if isinstance(value, bytes):
        return {'constant_type': 'bytes', 'hex': value.hex()}
    if value is Ellipsis:
        return {'constant_type': 'ellipsis'}
    if isinstance(value, complex):
        return {'constant_type': 'complex', 'real': value.real, 'imag': value.imag}
    return value


def source_fingerprint(source: str) -> str:
    """Hash a deterministic JSON representation, not Python-version-specific text."""
    payload = {'algorithm': ALGORITHM, 'tree': canonical_tree(ast.parse(source))}
    serialized = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(serialized.encode('ascii')).hexdigest()
''')


[Markdown Cell]

`state.py` — Reviewed source, configuration and ownership verification.

In [78]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
''')


In [79]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
''')


In [80]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible summarization checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_score'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
''')


In [81]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''            raise ValueError('Incomplete/incompatible summarization checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        # Optimizer states exist only for the active parameters in these PEFT
        # runs. Count the actual payload instead of reserving Adam moments for
        # every frozen foundation weight at each checkpoint write.
        seen=set()
        def tensor_bytes(value):
            if isinstance(value,torch.Tensor):
                if id(value) in seen:
                    return 0
                seen.add(id(value))
                return value.numel()*value.element_size()
            if isinstance(value,dict):
                return sum(tensor_bytes(part) for part in value.values())
            if isinstance(value,(list,tuple)):
                return sum(tensor_bytes(part) for part in value)
            return 0
        if temporary.is_file():
            temporary.unlink()  # only this uncommitted slot's known temporary file
        estimate=tensor_bytes(payload)+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_score':payload['cursor']['best_score']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
''')


In [82]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''            atomic_json(self.directory/'best.pt.json',receipt)
        return path,receipt
''')


[Markdown Cell]

`train.py` — Two NCCL spawn workers, AMP and exact optimizer/scaler/scheduler/cursor/RNG checkpoint recovery.

In [83]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''"""Two T4 NCCL spawn workers, exhaustive queries and exact full-state recovery."""
import argparse
from datetime import timedelta
import math
import os
from pathlib import Path
import signal
import socket
import time
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from tqdm.auto import tqdm
from .config import PCRConfig
from .features import FeatureStore
from .batches import RetrievalBatches, epoch_order, relevance_loss
from .model import build_model
from .io_utils import atomic_json, read_json, run_lock
from .state import Checkpoints, WarmupCosine, capture_rng, restore_rng, seed_everything
from .ranking import embed, dense_scores, metrics
from .lexical import scores_for


def initial_cursor():
    return {'epoch': 0, 'next_batch': 0, 'global_step': 0, 'optimizer_steps': 0, 'complete': False,
        'best_score': -1., 'bad_epochs': 0, 'history': [], 'queries': 0, 'loss_sum': 0.,
        'overflows': 0, 'consecutive_overflows': 0, 'epoch_update_seconds': 0.}


def optimizer_for(model, cfg):
    foundation, heads = [], []
    for name, value in model.named_parameters():
        if value.requires_grad:
            (foundation if name.startswith('backbone.') or name == 'query_prompts' else heads).append(value)
    groups = [{'params': foundation, 'lr': cfg.learning_rate}, {'params': heads, 'lr': cfg.head_learning_rate}]
    groups = [group for group in groups if group['params']]
    fused = all(value.is_cuda for group in groups for value in group['params'])
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay, fused=fused)

''')


In [84]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''
def save_all(rank, manager, model, optimizer, scaler, scheduler, cursor, cfg, device, best=False, paused=False):
    states = [None] * cfg.world_size
    dist.all_gather_object(states, capture_rng(device))
    failure = [None]
    if rank == 0:
        try:
            actual = model.module if hasattr(model, 'module') else model
            payload = {'version': 1, 'fingerprint': manager.fingerprint, 'config': cfg.__dict__,
                'model': actual.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'scheduler': scheduler.state_dict(), 'epoch': cursor['epoch'], 'step': cursor['global_step'],
                'cursor': cursor, 'rng_by_rank': states}
            manager.commit(payload, best=best)
            atomic_json(Path(cfg.run_dir) / 'training_summary.json', {'status': 'complete' if cursor['complete'] else 'paused_recoverable' if paused else 'training', **cursor})
        except Exception as error:
            failure[0] = str(error)
    dist.broadcast_object_list(failure, src=0)
    if failure[0]:
        raise RuntimeError('Checkpoint commit failed: ' + failure[0])
    dist.barrier()


@torch.no_grad()
def validate_dev(rank, model, store, job, cfg, device):
    candidates = embed(model, store, 'dev', 'candidates', cfg, device, rank, cfg.world_size)
    queries = embed(model, store, 'dev', 'queries', cfg, device, rank, cfg.world_size)
    result = [None]
    if rank == 0:
        from .ranking import predictions
        references = store.catalog['references']['dev']
        rows = predictions(dense_scores(queries, candidates), scores_for(job['lexical'], 'dev'),
            references['queries']['ids'], references['candidates']['ids'])
        truth = read_json(Path(job['prepared_dir']) / 'dev_truth.json')
        selection = read_json(Path(job['prepared_dir']) / 'selection_split.json')
        result[0] = metrics(truth, rows, references['candidates']['ids'], selection['tune_ids'])
    dist.broadcast_object_list(result, src=0)
    return result[0]


def worker(rank, job, port, event):
''')


In [85]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''    cfg = PCRConfig(**job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank, world_size=2, timeout=timedelta(hours=2))
    try:
        seed_everything(cfg.seed)
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest = manager.latest()
        payload = manager.load(latest) if latest else None
        model = build_model(job, cfg, device, load_parent=payload is None)
        if payload:
            model.load_state_dict(payload['model'], strict=True)
        ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, find_unused_parameters=False, gradient_as_bucket_view=True)
        optimizer = optimizer_for(model, cfg)
        store = FeatureStore(job['features'])
        dataset = RetrievalBatches(job, cfg, store)
        slots = cfg.world_size * cfg.query_batch_per_rank
        steps = math.ceil(len(dataset) / slots)
        scheduler = WarmupCosine(optimizer, max(1, steps * cfg.max_epochs), cfg.warmup_fraction, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        cursor = initial_cursor()
        if payload:
            optimizer.load_state_dict(payload['optimizer'])
            scaler.load_state_dict(payload['scaler'])
            for name in ('total_steps', 'warmup_steps', 'minimum', 'base_lrs'):
                setattr(scheduler, name, payload['scheduler'][name])
            scheduler.load_state_dict(payload['scheduler'])
            cursor = payload['cursor']
            restore_rng(payload['rng_by_rank'][rank], device)
            del payload
            if cursor['global_step'] != cursor['epoch'] * steps + cursor['next_batch']:
                raise ValueError('Saved cursor differs from this exhaustive query plan')
        else:
            seed_everything(cfg.seed + rank)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
        if cursor['complete'] and job['extension_requested']:
            cursor.update(complete=False, bad_epochs=0)
        if cursor['complete']:
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
''')


In [86]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''            return
        started = time.monotonic()
        for epoch in range(cursor['epoch'], job['target_epochs']):
            ddp.train()
            plan = epoch_order(len(dataset), cfg.seed, epoch, slots)
            progress = tqdm(total=steps, initial=cursor['next_batch'], disable=rank != 0,
                desc=f'{cfg.experiment_id} epoch {epoch + 1}/{job["target_epochs"]}')
            for step in range(cursor['next_batch'], steps):
                update_started = time.monotonic()
                stop = torch.tensor(int(event.is_set() or time.monotonic() - started >= cfg.max_run_seconds or
                    time.time() >= float(os.environ.get('CS781_P7_DEADLINE', 'inf'))), device=device)
                dist.all_reduce(stop, op=dist.ReduceOp.MAX)
                if stop.item():
                    progress.close()
                    save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device, paused=True)
                    return
                global_indices = plan[step].tolist()
                selected = global_indices[rank * cfg.query_batch_per_rank:(rank + 1) * cfg.query_batch_per_rank]
                count = sum(index >= 0 for index in global_indices)
                inputs, positives, allowed, weights, teacher = dataset.build(selected, epoch, step, rank, device)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.float16):
                    embeddings = ddp(**inputs)
                    loss, _ = relevance_loss(embeddings, len(selected), positives, allowed, weights, cfg, teacher)
                scaler.scale(loss * cfg.world_size / count).backward()
                totals = loss.detach().double()
                dist.all_reduce(totals)
                actual_loss = totals.item()
                if not math.isfinite(actual_loss):
                    raise FloatingPointError('Nonfinite retrieval objective; recover the last committed checkpoint')
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                previous = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous:
                    scheduler.advance()
                    cursor['optimizer_steps'] += 1
                    cursor['consecutive_overflows'] = 0
                else:
''')


In [87]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''                    cursor['overflows'] += 1
                    cursor['consecutive_overflows'] += 1
                if cursor['consecutive_overflows'] > cfg.max_consecutive_overflows:
                    raise FloatingPointError('Repeated FP16 overflows; keep checkpoints and inspect the failure report')
                cursor.update(next_batch=step + 1, global_step=cursor['global_step'] + 1,
                    queries=cursor['queries'] + count, loss_sum=cursor['loss_sum'] + actual_loss,
                    epoch_update_seconds=cursor['epoch_update_seconds'] + time.monotonic() - update_started)
                optimizer.zero_grad(set_to_none=True)
                if rank == 0:
                    progress.update()
                    if cursor['global_step'] % cfg.log_every_steps == 0 or step + 1 == steps:
                        progress.set_postfix(loss=cursor['loss_sum'] / max(1, cursor['queries']), lr=optimizer.param_groups[0]['lr'],
                            step=cursor['global_step'], amp_skips=cursor['overflows'],
                            seconds_per_update=cursor['epoch_update_seconds'] / (step + 1))
                if cursor['global_step'] % 500 == 0:
                    save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
            progress.close()
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
            evaluated = validate_dev(rank, model, store, job, cfg, device)
            if cursor['queries'] != len(dataset):
                raise ValueError('A complete epoch must visit every training query once')
            score = evaluated['micro_f1_at7']
            improved = score > cursor['best_score']
            cursor['best_score'] = max(cursor['best_score'], score)
            cursor['bad_epochs'] = 0 if improved else cursor['bad_epochs'] + 1
            peak = torch.tensor(torch.cuda.max_memory_allocated(device), device=device, dtype=torch.int64)
            dist.all_reduce(peak, op=dist.ReduceOp.MAX)
            cursor['history'].append({'epoch': epoch + 1, 'train_loss': cursor['loss_sum'] / len(dataset),
                'all_training_queries': cursor['queries'], 'dev_tune_dense': evaluated,
                'training_update_seconds': cursor['epoch_update_seconds'], 'max_peak_allocated_bytes_both_GPUs': peak.item()})
            cursor.update(epoch=epoch + 1, next_batch=0, queries=0, loss_sum=0., epoch_update_seconds=0.)
            cursor['complete'] = cursor['epoch'] >= job['target_epochs'] or (cursor['epoch'] >= cfg.minimum_epochs and cursor['bad_epochs'] >= cfg.patience)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device, best=improved)
            if rank == 0:
                print({'epoch': cursor['epoch'], 'tune_micro_f1_at7': score, 'complete': cursor['complete']}, flush=True)
            if cursor['complete']:
                return
    finally:
        dist.destroy_process_group()

''')


In [88]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    job = read_json(args.job)
    from .jobs import hardware
    hardware()
    cfg = PCRConfig(**job['config'])
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    with run_lock(Path(cfg.run_dir)):
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        if manager.best():
            manager.adopt(manager.best(), best=True)
        latest = manager.latest()
        if latest:
            manager.adopt(latest)
            print('Resume exact saved step', latest[1]['step'], 'epoch', latest[1]['epoch'], flush=True)
        context = mp.get_context('spawn')
        event = context.Event()
        signal.signal(signal.SIGINT, lambda *_: event.set())
        signal.signal(signal.SIGTERM, lambda *_: event.set())
        with socket.socket() as sock:
            sock.bind(('127.0.0.1', 0))
            port = sock.getsockname()[1]
        mp.spawn(worker, args=(job, port, event), nprocs=2, join=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`workflow.py` — Input checks, cache/training/export stages and a shared launch deadline.

In [89]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''"""Checked data/foundation -> resumable cache -> DDP training -> exact top-seven ZIP."""
import argparse
from dataclasses import asdict
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import time
import traceback
import zipfile
import torch
import tokenizers
from .config import PCRConfig
from .features import select_cache, verify_cache
from .io_utils import atomic_json, identity, read_json
from .jobs import prepare_job, hardware, sanity
from .state import Checkpoints


def child(command):
    process = subprocess.Popen(command)
    old = {}
    def stop(signum, frame):
        if process.poll() is None:
            process.send_signal(signum)
    try:
        for signum in (signal.SIGINT, signal.SIGTERM):
            old[signum] = signal.signal(signum, stop)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        for signum, callback in old.items():
            signal.signal(signum, callback)


def observations(root):
    root = Path(root)
''')


In [90]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''    archive = root / 'observations.zip'
    with zipfile.ZipFile(archive.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as stream:
        for file in root.rglob('*'):
            if file.is_file() and file.suffix in ('.json', '.txt', '.log') and file.stat().st_size < 8 << 20:
                stream.write(file, file.relative_to(root).as_posix())
    archive.with_suffix('.zip.partial').replace(archive)


def stage_reports(job):
    root = Path(job['config']['run_dir'])
    for previous in job['resume_roots']:
        old = Path(previous)
        if old.resolve() == root.resolve():
            continue
        for name in ('model_report.json', 'training_summary.json'):
            if not (root / name).is_file() and (old / name).is_file():
                shutil.copyfile(old / name, root / name)


def run_component(cfg, catalog, teacher=None, parent_override=None, teacher_only=False, export_only=False):
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    job = prepare_job(cfg, catalog, teacher, parent_override)
    stage_reports(job)
    features = select_cache(job, cfg)
    if features.get('status') == 'needs_build':
        child([sys.executable, '-u', '-m', 'pcr.features', '--cache-job', str(Path(features['directory']) / 'cache_job.json')])
        if not (Path(features['directory']) / 'feature_manifest.json').is_file():
            atomic_json(root / 'final_report.json', {'status': 'paused_recoverable', 'stage': 'frozen_feature_cache',
                'next_action': 'Save COMPLETE output and resume this same notebook; committed case rows are retained.'})
            return None
        features = verify_cache(features['directory'], features['identity'])
    job['features'] = features
    atomic_json(root / 'job.json', job)
    if not export_only:
        if time.time() >= float(os.environ.get('CS781_P7_DEADLINE', 'inf')):
            atomic_json(root / 'final_report.json', {'status': 'paused_recoverable', 'stage': 'before_training'})
            return None
        sanity(job)
        child([sys.executable, '-u', '-m', 'pcr.train', '--job', str(root / 'job.json')])
''')


In [91]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''        if read_json(root / 'training_summary.json')['status'] != 'complete':
            atomic_json(root / 'final_report.json', {'status': 'paused_recoverable', 'stage': 'training',
                'next_action': 'Save COMPLETE output and resume this same notebook; exact optimizer/cursor states are kept.'})
            return None
    else:
        manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest, best = manager.latest(), manager.best()
        if not latest or not best or not manager.load(latest)['cursor']['complete']:
            raise ValueError('Select a COMPLETE own Phase 7 task for export')
        manager.adopt(best, best=True)
        manager.adopt(latest)
    command = [sys.executable, '-u', '-m', 'pcr.export', '--job', str(root / 'job.json')]
    if teacher_only:
        command.append('--teacher-only')
    child(command)
    return job


def bind_teacher(job, cfg, teacher):
    # No student updates take place before automatic teacher construction finishes.
    # Rebind the initial preparation fingerprint once; later full-state recovery
    # requires this exact score snapshot.
    root = Path(cfg.run_dir)
    if job.get('teacher') and job['teacher']['sha256'] != teacher['sha256']:
        raise ValueError('KD teacher changed after student preparation')
    if not job.get('teacher'):
        manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        if manager.latest():
            raise ValueError('Cannot replace the teacher after any student optimizer state exists')
        job['teacher'] = teacher
        # Match jobs.prepare_job's fingerprint formula for every future resume.
        from . import VERSION
        from .parents import source_hashes
        job['fingerprint'] = identity({'version': VERSION, 'choices': cfg.choices(), 'data': job['prepared_identity'],
            'foundation': job['parent']['checkpoint_sha256'], 'tokenizer': job['parent']['tokenizer_sha256'],
            'sources': source_hashes(Path(__file__).parent), 'torch': str(torch.__version__),
            'tokenizers': tokenizers.__version__, 'teacher': teacher['sha256']})
        atomic_json(root / 'job.json', job)
    return job

''')


In [92]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''
def execute(plan_path, catalog_path):
    plan, catalog = read_json(plan_path), read_json(catalog_path)
    cfg = PCRConfig(**plan['config'])
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('CS781_P7_DEADLINE', str(time.time() + cfg.max_run_seconds))
    try:
        hardware()
        if plan['mode'] == 'train' and cfg.method == 'kd':
            from .distillation import auto_teacher, teacher_snapshot
            # If a teacher snapshot is already saved, supply its hash before
            # recovery validation, so an interrupted student never resets.
            local = root / 'job.json'
            paths = [local] if local.is_file() else [Path(cfg.resume_run_dir) / 'job.json'] if cfg.resume_run_dir else [file for file in Path(cfg.input_root).rglob('job.json')
                if read_json(file).get('config', {}).get('experiment_id') == cfg.experiment_id]
            old = read_json(paths[0]) if paths and paths[0].is_file() else None
            saved_teacher = old.get('teacher') if old else None
            # Restage a read-only prior score snapshot before verifying its bytes.
            if saved_teacher:
                for origin in [root, *([paths[0].parent] if paths else [])]:
                    if (origin / 'teacher_export_receipt.json').is_file():
                        from .io_utils import publish_copy
                        for name in ('teacher_scores.bin', 'teacher_export_receipt.json'):
                            publish_copy(origin / name, root / name)
                        saved_teacher = {**saved_teacher, 'scores_path': str(root / 'teacher_scores.bin')}
                        break
            job = prepare_job(cfg, catalog, saved_teacher)
            # A cache/teacher pause may precede the first student checkpoint.
            # Carry the internal teacher forward with the complete saved output.
            for previous in job['resume_roots']:
                old_teacher = Path(previous) / 'auto_teacher'
                if old_teacher.is_dir() and old_teacher.resolve() != (root / 'auto_teacher').resolve() and not (root / 'auto_teacher').exists():
                    shutil.copytree(old_teacher, root / 'auto_teacher')
            teacher = auto_teacher(job, cfg, catalog, run_component)
            if teacher is None:
                atomic_json(root / 'final_report.json', {'status': 'paused_recoverable', 'stage': 'automatic_retrieval_teacher',
                    'next_action': 'Save COMPLETE output and resume notebook 07; no separate export notebook is needed.'})
                return read_json(root / 'final_report.json')
            bind_teacher(job, cfg, teacher)
''')


In [93]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''            from .parents import parent_class
            from .model import CaseRetriever
            Config, Model = parent_class(job['parent_source_dir'], 'b', job['parent_contract'])
            with torch.device('meta'):
                base_cfg = Config(**job['parent']['config'])
                student = CaseRetriever(Model(base_cfg), cfg, base_cfg)
            student_parameters = sum(value.numel() for value in student.parameters())
            if teacher['teacher_parameters'] <= student_parameters:
                raise ValueError('Choose a larger own teacher. Automatic Phase 6 A/B/C/D recovery normally supplies a larger full retriever than this compact B student.')
            atomic_json(root / 'distillation_report.json', {'teacher_parameters': teacher['teacher_parameters'],
                'student_parameters': student_parameters, 'student_parameter_fraction': student_parameters / teacher['teacher_parameters'],
                'student_initialized_from_own_phase1_B': True, 'teacher_scores_cover_entire_training_candidate_pool': True,
                'separate_export_notebook_required': False})
            del student
            trained = run_component(cfg, catalog, teacher)
        else:
            trained = run_component(cfg, catalog, teacher_only=plan['mode'] == 'teacher', export_only=plan['mode'] == 'teacher')
        if trained is None:
            return read_json(root / 'final_report.json')
        return read_json(root / ('teacher_export_receipt.json' if plan['mode'] == 'teacher' else 'final_report.json'))
    except Exception as error:
        atomic_json(root / 'failure_report.json', {'status': 'failed', 'type': type(error).__name__, 'message': str(error),
            'traceback': traceback.format_exc(), 'next_action': 'Keep checkpoints and cache journals. Correct the reported input/settings; no silent restart or fake score is produced.'})
        raise
    finally:
        observations(root)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    args = parser.parse_args()
    from .io_utils import run_lock
    root = Path(read_json(args.plan)['config']['run_dir'])
    with run_lock(root / 'workflow_ownership'):
        print(json.dumps(execute(args.plan, args.catalog), indent=2), flush=True)


if __name__ == '__main__':
''')


In [94]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''    main()
''')


In [95]:
# [Code Cell]
EXPECTED_SOURCE_SHA256 = {'__init__.py': '9487973c4beb606ee21c6bcac58a19b36dbcd818b3af174db5ace2a79c4f516a', 'adaptation.py': '7645ff4cb0f8ff1438e9c14d087704b7a1979a2c06a627cd8953797f1540e7dd', 'batches.py': 'cd6c59fc23099894a6c71112f067e06843d43c11a6d02c07815b101ad5d44c52', 'compare.py': '6cc8b6bc2436acd3fe983e3872dbbc2c49dc9675d7eb20b40d4e30af9a053699', 'config.py': '036e856d4ea7935b557d0eb30bd303a295e7b0d4311d442590044f573670080f', 'distillation.py': '566223aa83e1fde5472ea5b585f26710af1757e94e1c599296e3e68b2ab1f916', 'export.py': '484328658f6c4c1c885dee5836279d6267035aad6c4f9ff5eebdde04b0c20ffa', 'features.py': '335279ff4500465b4f037fa2871daf05597b365b173b6d2ba25019c8c17f3e70', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'jobs.py': '625d1e9ddb5b65fbeb6fe8c8757ca0d1068b3d8c1e10845cdc607f890f62ab8f', 'lexical.py': '63d912da0786105b20e84e835e245d00ee5b7b4464f5126ed65547dbd49d5f18', 'model.py': 'cf35cfac47a745283ec43dfdaf7a51899a9c042b9479d0f20adfbcaa51b7c77e', 'native_contracts.py': 'f76cae1b17467620d65bcdd85d0aba6d15b5bf8cf7f8e4a8b30580b7951e368c', 'native_parents.py': '1c77968d9f566a65e4508d3cb590fe08fba058c7490de2e13baf3165095170ca', 'official_evaluator.py': '1d8b311201f6938a0e93721fd62c527cd61f2816b6d76c9e115b1084c750ca93', 'p6_contract.py': '006a2a0f7c74d6829d25d334f3a174a95517a3e36c97a4a9852c2d9d445dc3fa', 'parents.py': '46cc4c00d24b3b62955a76ec71d14782ee47a2874876daca285c722d44f5c6d5', 'preparation.py': 'c668a659668e9d892b1d4b6701e7ac6b806405f7b57aad5464b0eb8638edfcaf', 'ranking.py': 'f093cc8e6560c041abd15d0fc57c2073c62082054b0ce7658fd1dbb496890663', 'source_identity.py': 'bff746c41ac050d03b1b02aee9d6e709c334697ad9ff6e69af4edacd0923690c', 'state.py': '53affe1a87e329eb0929431690e8127988457baf2504cfab5f5fa35e77f10a17', 'train.py': '662f79283a3667ca5b0fc9b89d1149cd1ffb32e557a72730766d72d67eb0c335', 'workflow.py': '78dea590f441fc95f95f3b8d5d23272534f6f33b17f23dfe8f397022235e61f2'}
import hashlib
import importlib
code_directory = Path(OUTPUT_ROOT) / 'code'
package_directory = code_directory / 'pcr'
package_directory.mkdir(parents=True, exist_ok=True)
for name, pieces in module_parts.items():
    source = ''.join(pieces)
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED_SOURCE_SHA256[name], name
    compile(source, name, 'exec')
    target = package_directory / name
    if (Path(OUTPUT_ROOT) / 'job.json').is_file() and target.is_file() and target.read_bytes() != source.encode():
        raise ValueError('Saved Phase 7 source changed. Resume with its exact notebook.')
    target.write_bytes(source.encode())
for name in list(sys.modules):
    if name == 'pcr' or name.startswith('pcr.'):
        del sys.modules[name]
sys.path.insert(0, str(code_directory))
importlib.invalidate_caches()
print('Verified Phase 7 runtime modules:', len(EXPECTED_SOURCE_SHA256))


Verified Phase 7 runtime modules: 23


[Markdown Cell]

### Numeric checkpoint compatibility
Known NumPy numeric metadata remains supported with `weights_only=True`, including in every spawned worker.

In [96]:
# [Code Cell]
COMPATIBILITY_PARTS = {}


In [97]:
# [Code Cell]
COMPATIBILITY_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''"""Read existing scalar-metadata checkpoints without changing the LSI runtime.

Only NumPy's numeric scalar constructor, dtype constructor and a fixed list of
numeric dtype classes are registered. Tensor/state loading stays weights_only.
No checkpoint is rewritten, and no globals are inferred from checkpoint text.
"""
from importlib import import_module
import io
import os

import numpy as np
import torch

VERSION = 'cs781-numpy-scalar-resume-v1'
DTYPE_NAMES = (
    'bool', 'int8', 'int16', 'int32', 'int64',
    'uint8', 'uint16', 'uint32', 'uint64',
    'float16', 'float32', 'float64',
)


def register_numpy_scalars() -> list[str]:
    """Allow the known numeric metadata types in this interpreter, idempotently."""
    namespace = 'numpy._core.multiarray' if int(np.__version__.split('.')[0]) >= 2 else 'numpy.core.multiarray'
    scalar_constructor = import_module(namespace).scalar
    accepted = [scalar_constructor, np.dtype, *[type(np.dtype(name)) for name in DTYPE_NAMES]]
    existing = torch.serialization.get_safe_globals()
    torch.serialization.add_safe_globals([value for value in accepted if value not in existing])
    os.environ['CS781_NUMPY_RESUME_PID'] = str(os.getpid())
    return [value.__module__ + '.' + value.__name__ for value in accepted]


def self_test() -> dict:
    """Exercise scalar loading on CPU without advancing any training RNG."""
    metadata = {name: np.dtype(name).type(1) for name in DTYPE_NAMES}
    stream = io.BytesIO()
    torch.save({'metadata': metadata, 'weights': torch.tensor([1., 2.])}, stream)
    stream.seek(0)
    loaded = torch.load(stream, map_location='cpu', weights_only=True)
    for name, original in metadata.items():
''')


In [98]:
# [Code Cell]
COMPATIBILITY_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''        assert type(loaded['metadata'][name]) is type(original), name
        assert loaded['metadata'][name] == original, name
    assert torch.equal(loaded['weights'], torch.tensor([1., 2.]))
    return {'status': 'passed', 'compatibility_version': VERSION,
            'torch': str(torch.__version__), 'numpy': np.__version__,
            'weights_only': True, 'numeric_dtype_types': len(DTYPE_NAMES)}
''')


In [99]:
# [Code Cell]
COMPATIBILITY_PARTS.setdefault('sitecustomize.py', []).append(r'''"""Run-local startup hook inherited by subprocesses and mp.spawn children.

Keep a pre-existing environment startup hook. This directory is inserted only
in the launch environment of this notebook, never installed into site-packages.
"""
import importlib.machinery
import importlib.util
import os
from pathlib import Path
import sys
import traceback


def _original_startup_hook() -> None:
    own_directory = Path(__file__).resolve().parent
    other_paths = [entry for entry in sys.path
                   if Path(entry or os.getcwd()).resolve() != own_directory]
    spec = importlib.machinery.PathFinder.find_spec('sitecustomize', other_paths)
    if spec is None or spec.loader is None:
        return
    if spec.origin and Path(spec.origin).resolve() == Path(__file__).resolve():
        return
    original = importlib.util.module_from_spec(spec)
    current = sys.modules.get('sitecustomize')
    sys.modules['sitecustomize'] = original
    try:
        spec.loader.exec_module(original)
    finally:
        if current is not None:
            sys.modules['sitecustomize'] = current
        else:
            sys.modules.pop('sitecustomize', None)


try:
    _original_startup_hook()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        from cs781_numpy_resume import register_numpy_scalars
        register_numpy_scalars()
except Exception:
''')


In [100]:
# [Code Cell]
COMPATIBILITY_PARTS.setdefault('sitecustomize.py', []).append(r'''    traceback.print_exc()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        sys.stderr.write('Checkpoint compatibility startup failed; training was not started.\n')
        sys.stderr.flush()
        os._exit(78)
''')


In [101]:
# [Code Cell]
compat_directory = Path(OUTPUT_ROOT) / 'checkpoint_compat'
compat_directory.mkdir(exist_ok=True)
for name, parts in COMPATIBILITY_PARTS.items():
    (compat_directory / name).write_text(''.join(parts), encoding='utf-8')
sys.path.insert(0, str(compat_directory))
import cs781_numpy_resume
cs781_numpy_resume.register_numpy_scalars()
print(cs781_numpy_resume.self_test())


{'status': 'passed', 'compatibility_version': 'cs781-numpy-scalar-resume-v1', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'weights_only': True, 'numeric_dtype_types': 12}


[Markdown Cell]

### T4 x2 execution optimization
The extra modules pack source chunks across cases, balance new cache work between the two GPUs,
overlap input preparation/transfers with training, measure larger encoder microbatches, and avoid
activation recomputation when a measured safe case count fits in memory. No model parameters,
global query batch, negatives, LR schedule, epochs or source tokens are removed.

Existing full caches are reused; incomplete original caches retain all committed rows. New packed
caches have a new layout and must be resumed with this optimized suite. Existing training checkpoints
retain model/optimizer/scaler/scheduler/RNG/cursor state. Different kernels and dropout batching can
change later numerical results; this is not a bitwise-trajectory guarantee.

In [102]:
# [Code Cell]
T4_PARTS = {}


[Markdown Cell]

`p7_t4/__init__.py` — checked execution optimization.

In [103]:
# [Code Cell]
T4_PARTS.setdefault('__init__.py', []).append(r'''"""Execution-only T4 optimizations for the reviewed Phase 7 retrievers."""
VERSION = 'p7-t4-execution-v1'
''')


[Markdown Cell]

`p7_t4/batches.py` — checked execution optimization.

In [104]:
# [Code Cell]
T4_PARTS.setdefault('batches.py', []).append(r'''"""Identical candidates/labels with cached negatives and asynchronous input staging."""
import atexit
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from pcr.batches import RetrievalBatches as OriginalBatches, epoch_order
from pcr.lexical import stable_order


class FastBatches(OriginalBatches):
    """Preserve the original query order and candidate union on every rank/update."""
    options = None

    def __init__(self, job, cfg, store):
        super().__init__(job, cfg, store)
        self.hard, self.teacher_hard = [], []
        for query, positive in enumerate(self.positives):
            gold = set(positive)
            aliases = {self.candidate_docs[index] for index in gold}
            self.hard.append([int(index) for index in stable_order(self.lexical[query])
                              if int(index) not in gold and self.candidate_docs[int(index)] not in aliases][:cfg.hard_negatives])
            if self.teacher is not None:
                self.teacher_hard.append([int(index) for index in stable_order(self.teacher[query])
                                          if int(index) not in gold][:cfg.hard_negatives])
        self.executor, self.future, self.future_key, self.plan = None, None, None, None
        self.plan_epoch = None
        self.stream = None
        self.keep_alive = None

    def cpu_batch(self, query_indices, epoch):
        chosen = set()
        for query in query_indices:
            if query < 0:
                continue
            positives = set(self.positives[query])
            aliases = {self.candidate_docs[index] for index in positives}
            chosen.update(positives)
            chosen.update(self.hard[query])
            if self.teacher is not None:
                chosen.update(self.teacher_hard[query])
''')


In [105]:
# [Code Cell]
T4_PARTS.setdefault('batches.py', []).append(r'''            rng = np.random.default_rng(self.cfg.seed + epoch * 1_000_003 + query * 101 + 17)
            taken = 0
            for index in rng.permutation(len(self.candidate_ids)):
                if int(index) not in positives and self.candidate_docs[int(index)] not in aliases:
                    chosen.add(int(index))
                    taken += 1
                    if taken == self.cfg.random_negatives:
                        break
        candidates = sorted(chosen or {0})
        documents = [self.query_docs[max(0, query)] for query in query_indices] + [self.candidate_docs[index] for index in candidates]
        inputs = self.store.batch(documents, [0] * len(query_indices) + [1] * len(candidates), torch.device('cpu'))
        positive = np.zeros((len(query_indices), len(candidates)), np.bool_)
        allowed = np.ones_like(positive)
        candidate_docs = np.asarray([self.candidate_docs[index] for index in candidates])
        for row, query in enumerate(query_indices):
            if query < 0:
                continue
            gold = self.positives[query]
            positive[row] = np.isin(candidates, gold)
            allowed[row] = positive[row] | ~np.isin(candidate_docs, [self.candidate_docs[index] for index in gold])
        weights = np.asarray([query >= 0 for query in query_indices], np.float32)
        teacher = None
        if self.teacher is not None:
            teacher = torch.from_numpy(np.asarray(self.teacher[np.ix_([max(0, q) for q in query_indices], candidates)]).copy())
        return inputs, torch.from_numpy(positive), torch.from_numpy(allowed), torch.from_numpy(weights), teacher

    def stage(self, query_indices, epoch, device):
        batch = self.cpu_batch(query_indices, epoch)
        if device.type != 'cuda':
            return batch, None, None
        with torch.cuda.device(device):
            def pin(value):
                if isinstance(value, torch.Tensor):
                    return value.pin_memory()
                if isinstance(value, dict):
                    return {key: pin(part) for key, part in value.items()}
                if isinstance(value, tuple):
                    return tuple(pin(part) for part in value)
                return value
            pinned = pin(batch)
''')


In [106]:
# [Code Cell]
T4_PARTS.setdefault('batches.py', []).append(r'''            def move(value):
                if isinstance(value, torch.Tensor):
                    return value.to(device, non_blocking=True)
                if isinstance(value, dict):
                    return {key: move(part) for key, part in value.items()}
                if isinstance(value, tuple):
                    return tuple(move(part) for part in value)
                return value
            with torch.cuda.stream(self.stream):
                result = move(pinned)
                event = torch.cuda.Event()
                event.record(self.stream)
            return result, event, pinned

    def build(self, query_indices, epoch, step, rank, device):
        if device.type != 'cuda' or not self.options['prefetch']:
            result, event, pinned = self.stage(query_indices, epoch, device)
            self.keep_alive = pinned
            return result
        if self.executor is None:
            self.executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix='p7-input')
            self.stream = torch.cuda.Stream(device=device)
            atexit.register(self.executor.shutdown, wait=True, cancel_futures=True)
        if self.plan_epoch != epoch:
            self.plan = epoch_order(len(self), self.cfg.seed, epoch, self.cfg.world_size * self.cfg.query_batch_per_rank)
            self.plan_epoch = epoch
        key = (epoch, step, rank, tuple(query_indices))
        if self.future_key == key:
            result, event, pinned = self.future.result()
        else:
            if self.future is not None:
                self.future.result()
            result, event, pinned = self.stage(query_indices, epoch, device)
        current = torch.cuda.current_stream(device)
        current.wait_event(event)
        def record(value):
            if isinstance(value, torch.Tensor):
                value.record_stream(current)
            elif isinstance(value, dict):
                for part in value.values():
''')


In [107]:
# [Code Cell]
T4_PARTS.setdefault('batches.py', []).append(r'''                    record(part)
            elif isinstance(value, tuple):
                for part in value:
                    record(part)
        record(result)
        self.keep_alive = pinned
        self.future, self.future_key = None, None
        if step + 1 < len(self.plan):
            first = rank * self.cfg.query_batch_per_rank
            selected = self.plan[step + 1, first:first + self.cfg.query_batch_per_rank].tolist()
            self.future_key = (epoch, step + 1, rank, tuple(selected))
            self.future = self.executor.submit(self.stage, selected, epoch, device)
        return result
''')


[Markdown Cell]

`p7_t4/cache.py` — checked execution optimization.

In [108]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''"""Balanced two-GPU, multi-document frozen-cache packing with legacy-row recovery."""
import argparse
from datetime import timedelta
import hashlib
import json
import math
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn import functional as F
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from pcr.config import PCRConfig
from pcr.io_utils import atomic_json, ensure_space, identity, read_json, records, sha256_file
from pcr.model import build_model
from pcr.features import recover_journal as original_recover
from pcr.parents import source_hashes


def prepare_tokens(job, event=None):
    root = Path(job['directory'])
    signature = identity({'documents': job['catalog']['documents_sha256'],
        'tokenizer': sha256_file(job['model_job']['tokenizer_path']), 'policy': 'full-text-plus-EOS-uint16-v1'})
    manifest_file, binary, journal = root / 'token_manifest.json', root / 'tokens.bin', root / 'token_progress.jsonl'
    if manifest_file.is_file():
        value = read_json(manifest_file)
        if value['identity'] != signature or sha256_file(binary) != value['sha256'] or value['rows'] != list(records(journal)):
            raise ValueError('Packed-cache token identity/checksum changed')
        return value
    previous, end = [], 0
    valid_bytes = 0
    if journal.is_file():
        with journal.open('rb') as stream:
            while True:
''')


In [109]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''                line = stream.readline()
                if not line or not line.endswith(b'\n'):
                    break
                item = json.loads(line)
                if item['index'] != len(previous) or item['offset'] != end or item['length'] < 1:
                    raise ValueError('Invalid full-token journal')
                previous.append(item)
                end += item['length']
                valid_bytes = stream.tell()
        with journal.open('r+b') as stream:
            stream.truncate(valid_bytes)
    binary.touch(exist_ok=True)
    if binary.stat().st_size < end * 2:
        raise ValueError('Committed token cache is truncated')
    with binary.open('r+b') as stream:
        stream.truncate(end * 2)
    codec = Tokenizer.from_file(job['model_job']['tokenizer_path'])
    progress = tqdm(total=job['catalog']['unique_documents'], initial=len(previous), desc='Tokenize all cases once', unit='cases')
    with binary.open('ab') as output, journal.open('a', encoding='utf-8', newline='\n') as log:
        for row in records(root / 'documents.jsonl'):
            if row['index'] < len(previous):
                continue
            if (event is not None and event.is_set()) or time.time() >= float(os.environ.get('CS781_P7_DEADLINE', 'inf')):
                progress.close()
                return None
            tokens = codec.encode(row['text'], add_special_tokens=False).ids + [2]
            if min(tokens) < 0 or max(tokens) >= 32768:
                raise ValueError('Own tokenizer IDs must fit the verified 32768 vocabulary')
            ensure_space(root, len(tokens) * 2 + 4096, job['config']['min_free_bytes'])
            output.write(np.asarray(tokens, np.uint16).tobytes())
            output.flush()
            os.fsync(output.fileno())
            item = {'index': row['index'], 'offset': end, 'length': len(tokens), 'content_sha256': row['sha256']}
            log.write(json.dumps(item) + '\n')
            log.flush()
            os.fsync(log.fileno())
            previous.append(item)
            end += len(tokens)
            progress.update()
    progress.close()
''')


In [110]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''    value = {'identity': signature, 'rows': previous, 'sha256': sha256_file(binary), 'all_source_tokens_with_EOS': end}
    atomic_json(manifest_file, value)
    return value


def make_plan(job, tokens):
    root = Path(job['directory'])
    path = root / 't4_cache_plan.json'
    if path.is_file():
        plan = read_json(path)
        if plan['identity'] != job['identity'] or plan['token_sha256'] != tokens['sha256']:
            raise ValueError('Saved two-GPU work plan changed')
        return plan
    owners, committed = [], set()
    for rank in range(2):
        old, _ = original_recover(root / f'rank_{rank}.jsonl', root / f'rank_{rank}.bin', job['width'], rank)
        indices = [item['index'] for item in old]
        owners.append(indices)
        committed.update(indices)
    chunk = job['config']['chunk_tokens']
    costs = [math.ceil(row['length'] / chunk) for row in tokens['rows']]
    remaining = sorted((index for index in range(len(costs)) if index not in committed), key=lambda index: (-costs[index], index))
    loads = [0, 0]
    for index in remaining:
        rank = min(range(2), key=lambda value: (loads[value], value))
        owners[rank].append(index)
        loads[rank] += costs[index]
    if sorted(owners[0] + owners[1]) != list(range(len(costs))):
        raise ValueError('GPU plan must cover every document exactly once')
    plan = {'version': 'pcr-t4-plan-v1', 'identity': job['identity'], 'token_sha256': tokens['sha256'],
            'owners': owners, 'remaining_chunk_loads': loads,
            'old_parity_remaining_chunk_loads': [sum(costs[i] for i in remaining if i % 2 == rank) for rank in range(2)],
            'previously_committed_cases_preserved': len(committed)}
    atomic_json(path, plan)
    return plan


def recover(root, rank, plan, width):
    path, binary = root / f'rank_{rank}.jsonl', root / f'rank_{rank}.bin'
    rows, end, valid_bytes = [], 0, 0
''')


In [111]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''    if path.is_file():
        with path.open('rb') as stream:
            while True:
                line = stream.readline()
                if not line or not line.endswith(b'\n'):
                    break
                row = json.loads(line)
                if len(rows) >= len(plan['owners'][rank]) or row['index'] != plan['owners'][rank][len(rows)] or row['offset'] != end or row['length'] < 1:
                    raise ValueError('Packed-cache committed rows do not match the saved GPU plan')
                end += row['length']
                rows.append(row)
                valid_bytes = stream.tell()
        with path.open('r+b') as stream:
            stream.truncate(valid_bytes)
    binary.touch(exist_ok=True)
    if binary.stat().st_size < end * width * 2:
        raise ValueError('Committed packed features are truncated')
    with binary.open('r+b') as stream:
        stream.truncate(end * width * 2)
    return rows, end


def packs(token_array, token_rows, owned, cfg, batch_size):
    pending, tags = [], []
    for index in owned:
        row = token_rows[index]
        for chunk_index, first in enumerate(range(0, row['length'], cfg.chunk_tokens)):
            part = token_array[row['offset'] + first:row['offset'] + min(first + cfg.chunk_tokens, row['length'])]
            pending.append(part)
            tags.append((index, chunk_index, len(part), first + len(part) == row['length']))
            if len(pending) == batch_size:
                yield pending, tags
                pending, tags = [], []
    if pending:
        yield pending, tags


def packed_features(model, parts, cfg, device):
    """One GPU call for chunks from many cases; all padding/tail lengths respected."""
    ids = torch.zeros(len(parts), cfg.chunk_tokens, dtype=torch.long)
''')


In [112]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''    mask = torch.zeros_like(ids, dtype=torch.bool)
    weights = torch.zeros(len(parts), cfg.tokens_per_chunk, cfg.chunk_tokens, dtype=torch.float32)
    lengths = []
    for row, part in enumerate(parts):
        length = len(part)
        ids[row, :length] = torch.from_numpy(np.asarray(part, np.int64).copy())
        mask[row, :length] = True
        pooled = min(length, cfg.tokens_per_chunk)
        lengths.append(pooled)
        for slot in range(pooled):
            first, stop = slot * length // pooled, math.ceil((slot + 1) * length / pooled)
            weights[row, slot, first:stop] = 1. / (stop - first)
    def move(value):
        return value.pin_memory().to(device, non_blocking=True) if device.type == 'cuda' else value.to(device)
    with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
        hidden = model.frozen_prefix(move(ids), move(mask), all(len(part) == cfg.chunk_tokens for part in parts))
    with torch.autocast(device_type=device.type, enabled=False):
        pooled = torch.bmm(move(weights), hidden.float()).cpu()
    if not torch.isfinite(pooled).all():
        raise FloatingPointError('Nonfinite frozen prefix features')
    return [pooled[row, :length] for row, length in enumerate(lengths)]


def encode_owned(model, job, tokens, plan, rank, device, batch_size, event=None):
    cfg, root = PCRConfig(**job['config']), Path(job['directory'])
    done, offset = recover(root, rank, plan, job['width'])
    token_array = np.memmap(root / 'tokens.bin', np.uint16, 'r')
    owned = plan['owners'][rank][len(done):]
    partial, elapsed = {}, {}
    progress = tqdm(total=len(plan['owners'][rank]), initial=len(done), disable=rank != 0, desc='Packed frozen cache, GPU 0', unit='cases')
    started = time.monotonic()
    with (root / f'rank_{rank}.bin').open('ab') as output, (root / f'rank_{rank}.jsonl').open('a', encoding='utf-8', newline='\n') as log:
        for parts, tags in packs(token_array, tokens['rows'], owned, cfg, batch_size):
            if (event is not None and event.is_set()) or time.time() >= float(os.environ.get('CS781_P7_DEADLINE', 'inf')) or time.monotonic() - started >= cfg.cache_run_seconds:
                break
            first = 0
            while first < len(parts):
                amount = min(batch_size, len(parts) - first)
                begin = time.monotonic()
                try:
''')


In [113]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''                    values = packed_features(model, parts[first:first + amount], cfg, device)
                except torch.cuda.OutOfMemoryError:
                    if amount == 1:
                        raise
                    batch_size = max(1, amount // 2)
                    torch.cuda.empty_cache()
                    continue
                per_chunk = (time.monotonic() - begin) / amount
                for value, (index, chunk_index, length, last) in zip(values, tags[first:first + amount]):
                    partial.setdefault(index, []).append(value)
                    elapsed[index] = elapsed.get(index, 0.) + per_chunk
                    if last:
                        memory = torch.cat(partial.pop(index))
                        memory = F.adaptive_avg_pool1d(memory.T[None], min(len(memory), cfg.memory_slots))[0].T.half().numpy()
                        output.write(memory.tobytes())
                        output.flush()
                        os.fsync(output.fileno())
                        row = {'index': index, 'offset': offset, 'length': len(memory),
                            'source_tokens': tokens['rows'][index]['length'], 'content_sha256': tokens['rows'][index]['content_sha256'],
                            'encode_seconds': elapsed.pop(index)}
                        log.write(json.dumps(row) + '\n')
                        log.flush()
                        os.fsync(log.fileno())
                        offset += len(memory)
                        done.append(row)
                        progress.update()
                first += amount
    progress.close()
    del token_array
    return len(done) == len(plan['owners'][rank])


def verify(root, expected, original):
    root = Path(root)
    value = read_json(root / 'feature_manifest.json')
    if value.get('version') != 'pcr-features-v2-t4':
        return original(root, expected)
    if value.get('status') != 'complete' or value['identity'] != expected:
        raise ValueError('Packed cache is incomplete or from another frozen prefix')
    if value.get('execution_source_sha256') != source_hashes(Path(__file__).parent):
''')


In [114]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''        raise ValueError('Packed cache execution source changed; use its exact optimized suite')
    for name, digest in value['files'].items():
        if sha256_file(root / name) != digest:
            raise ValueError('Packed cache checksum failed: ' + name)
    plan, tokens = read_json(root / 't4_cache_plan.json'), read_json(root / 'token_manifest.json')
    if plan['identity'] != expected or plan['token_sha256'] != tokens['sha256']:
        raise ValueError('Packed GPU plan/token identity changed')
    index = {}
    for rank in range(2):
        rows = list(records(root / f'rank_{rank}.jsonl'))
        if [row['index'] for row in rows] != plan['owners'][rank]:
            raise ValueError('Packed cache rows differ from their balanced GPU plan')
        offset = 0
        for row in rows:
            source = tokens['rows'][row['index']]
            if row['offset'] != offset or row['length'] < 1 or row['source_tokens'] != source['length'] or row['content_sha256'] != source['content_sha256']:
                raise ValueError('Packed cache row/source coverage failed')
            offset += row['length']
            index[str(row['index'])] = {'rank': rank, **row}
        if (root / f'rank_{rank}.bin').stat().st_size != offset * value['width'] * 2:
            raise ValueError('Packed cache feature bytes are truncated')
    if index != value['index'] or sorted(map(int, index)) != list(range(value['catalog']['unique_documents'])):
        raise ValueError('Packed cache did not retain every unique case')
    if value['catalog'] != read_json(root / 'document_catalog.json'):
        raise ValueError('Packed document catalog changed')
    return {**value, 'directory': str(root)}


def finish(job, tokens, plan):
    root = Path(job['directory'])
    index = {str(row['index']): {'rank': rank, **row} for rank in range(2) for row in records(root / f'rank_{rank}.jsonl')}
    names = ['rank_0.bin', 'rank_1.bin', 'rank_0.jsonl', 'rank_1.jsonl', 'document_catalog.json', 'documents.jsonl',
             'tokens.bin', 'token_progress.jsonl', 'token_manifest.json', 't4_cache_plan.json']
    value = {'version': 'pcr-features-v2-t4', 'status': 'complete', 'identity': job['identity'],
        'execution_source_sha256': source_hashes(Path(__file__).parent),
        'width': job['width'], 'index': index, 'catalog': job['catalog'],
        'files': {name: sha256_file(root / name) for name in names}, 'all_source_tokens_encoded': True,
        'unique_document_source_tokens_with_EOS': tokens['all_source_tokens_with_EOS'], 'frozen_outputs_only': True,
        'GPU_remaining_chunk_loads': plan['remaining_chunk_loads'], 'legacy_rows_preserved': plan['previously_committed_cases_preserved']}
    atomic_json(root / 'feature_manifest.json', value)
''')


In [115]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''    return value


def worker(rank, job, tokens, plan, port, event):
    from .control import current_profile
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank, world_size=2, timeout=timedelta(hours=2))
    try:
        cfg = PCRConfig(**job['config'])
        model = build_model(job['model_job'], cfg, device)
        model.eval()
        complete = encode_owned(model, job, tokens, plan, rank, device, current_profile()['options']['cache_batch'], event)
        status = torch.tensor(int(complete), device=device)
        dist.all_reduce(status, op=dist.ReduceOp.MIN)
        if rank == 0:
            if status.item():
                finish(job, tokens, plan)
            else:
                atomic_json(Path(job['directory']) / 'cache_pause_report.json', {'status': 'paused_recoverable',
                    'next_action': 'Save COMPLETE output and resume the same T4-optimized notebook; committed rows are preserved.'})
    finally:
        dist.destroy_process_group()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--cache-job', required=True)
    args = parser.parse_args()
    job = read_json(args.cache_job)
    context = mp.get_context('spawn')
    event = context.Event()
    signal.signal(signal.SIGINT, lambda *_: event.set())
    signal.signal(signal.SIGTERM, lambda *_: event.set())
    tokens = prepare_tokens(job, event)
    if tokens is None:
        return
    plan = make_plan(job, tokens)
    with socket.socket() as sock:
''')


In [116]:
# [Code Cell]
T4_PARTS.setdefault('cache.py', []).append(r'''        sock.bind(('127.0.0.1', 0))
        port = sock.getsockname()[1]
    mp.spawn(worker, args=(job, tokens, plan, port, event), nprocs=2, join=True)
''')


[Markdown Cell]

`p7_t4/control.py` — checked execution optimization.

In [117]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''"""Versioned execution policy without modifying the saved learning-runtime identity."""
import json
import os
from pathlib import Path
import shutil
import sys
import torch
import torch.distributed as dist
from pcr.io_utils import atomic_json, identity, read_json
from pcr.parents import source_hashes
from . import VERSION

_PROFILE = None


def current_profile():
    return _PROFILE


def bind(plan_path, options_path):
    plan, options = read_json(plan_path), read_json(options_path)
    cfg = plan['config']
    required = {'cache_batch', 'max_microbatch', 'inference_batch', 'memory_fraction', 'tuning_seconds', 'prefetch'}
    if set(options) != required or min(options['cache_batch'], options['max_microbatch'], options['inference_batch'], options['tuning_seconds']) < 1 or not .35 <= options['memory_fraction'] <= .8:
        raise ValueError('Invalid T4 execution options')
    body = {'version': VERSION, 'source_sha256': source_hashes(Path(__file__).parent), 'options': options,
            'learning_update_plan_changed': False, 'dropout_kernel_rounding_may_differ': True}
    profile = {**body, 'signature': identity(body)}
    root = Path(cfg['run_dir'])
    paths = [root]
    if cfg.get('resume_run_dir'):
        paths.append(Path(cfg['resume_run_dir']))
    else:
        paths += [file.parent for file in Path(cfg['input_root']).rglob('job.json')
                  if read_json(file).get('version') == 'cs781-phase7-pcr-v1' and
                  read_json(file).get('config', {}).get('experiment_id') == cfg['experiment_id']]
    for path in paths:
        if (path / 't4_execution_profile.json').is_file():
            if read_json(path / 't4_execution_profile.json')['signature'] != profile['signature']:
                raise ValueError('Saved T4 execution policy/source changed. Resume its exact optimized notebook/settings.')
''')


In [118]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''    atomic_json(root / 't4_execution_profile.json', profile)
    os.environ['CS781_P7_T4_PROFILE'] = str(root / 't4_execution_profile.json')
    return profile


@torch.no_grad()
def embed(model, store, split, kind, cfg, device, rank=0, world=1):
    from tqdm.auto import tqdm
    reference = store.catalog['references'][split][kind]
    documents = reference['document_indices']
    vectors = torch.zeros(len(documents), cfg.embedding_dim, device=device, dtype=torch.float32)
    owned = list(range(rank, len(documents), world))
    batch_size = current_profile()['options']['inference_batch']
    role = 0 if kind == 'queries' else 1
    model.eval()
    model._p7_execution = {**getattr(model, '_p7_execution', {}), 'inference_batch': batch_size}
    first = 0
    progress = tqdm(total=len(owned), disable=rank != 0, desc=f'{split} {kind}, GPU 0', unit='cases')
    while first < len(owned):
        indices = owned[first:first + batch_size]
        try:
            batch = store.batch([documents[index] for index in indices], [role] * len(indices), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                vectors[indices] = model(**batch).float()
            first += len(indices)
            progress.update(len(indices))
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            model._p7_execution['inference_batch'] = batch_size
            torch.cuda.empty_cache()
    progress.close()
    if world > 1:
        dist.all_reduce(vectors, op=dist.ReduceOp.SUM)
    if not torch.isfinite(vectors).all():
        raise FloatingPointError('Nonfinite retrieval vectors')
    return vectors


''')


In [119]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''def install(profile):
    global _PROFILE
    if profile['source_sha256'] != source_hashes(Path(__file__).parent):
        raise ValueError('T4 execution source differs from its verified snapshot')
    if _PROFILE is not None:
        if _PROFILE['signature'] != profile['signature']:
            raise ValueError('Another T4 execution profile is already installed in this worker')
        return
    _PROFILE = profile
    from . import kernels, cache
    from .batches import FastBatches
    from pcr import model, adaptation, features, jobs, workflow, train, ranking, export
    model.CaseRetriever.forward = kernels.forward
    adaptation.prefix_attend = kernels.prefix_attend
    FastBatches.options = profile['options']
    train.RetrievalBatches = FastBatches
    original_verify = features.verify_cache
    verify = lambda root, expected: cache.verify(root, expected, original_verify)
    features.verify_cache = workflow.verify_cache = verify
    original_prepare = jobs.prepare_job
    def prepare(cfg, catalog, teacher=None, parent_override=None):
        job = original_prepare(cfg, catalog, teacher, parent_override)
        root = Path(cfg.run_dir)
        previous = root / 't4_execution_profile.json'
        if previous.is_file() and read_json(previous)['signature'] != profile['signature']:
            raise ValueError('This task was saved with another T4 execution policy')
        destination = root / 'code/p7_t4'
        destination.mkdir(parents=True, exist_ok=True)
        for file in Path(__file__).parent.glob('*.py'):
            if file.resolve() != (destination / file.name).resolve():
                shutil.copyfile(file, destination / file.name)
        atomic_json(previous, profile)
        return job
    jobs.prepare_job = workflow.prepare_job = prepare
    original_ddp = train.DDP
    def ddp(model, *args, **kwargs):
        # Probe BEFORE DDP installs reducer hooks. The two ranks may test different
        # shapes/counts after an OOM and must not enter unmatched collectives.
        cfg = model.cfg
        device = next(model.parameters()).device
''')


In [120]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''        rank = dist.get_rank() if dist.is_initialized() else 0
        if rank == 0:
            print('Calibrating safe encoder batches on the actual GPU; no optimizer updates are performed.', flush=True)
        report = kernels.tune(model, cfg, device, profile['options'])
        atomic_json(Path(cfg.run_dir) / f't4_tuning_gpu{rank}.json', {**report, 'rank': rank, 'profile_signature': profile['signature']})
        if rank == 0:
            print('Selected T4 execution:', getattr(model, '_p7_execution', {}), flush=True)
        return original_ddp(model, *args, **kwargs)
    train.DDP = ddp
    ranking.embed = train.embed = export.embed = embed
    original_child = workflow.child
    def child(command):
        if len(command) >= 4 and command[2] == '-m':
            action = {'pcr.features': 'cache', 'pcr.train': 'train', 'pcr.export': 'export'}.get(command[3])
            if action:
                command = [sys.executable, '-u', '-m', 'p7_t4.launch', action, *command[4:]]
        return original_child(command)
    workflow.child = child
    from pcr.state import Checkpoints
    original_commit, original_load = Checkpoints.commit, Checkpoints.load
    def commit(manager, payload, best=False):
        payload['t4_execution_profile'] = profile
        return original_commit(manager, payload, best)
    def load(manager, candidate):
        payload = original_load(manager, candidate)
        old = payload.get('t4_execution_profile')
        if old and old['signature'] != profile['signature']:
            raise ValueError('Checkpoint belongs to a different T4 execution policy')
        return payload
    Checkpoints.commit, Checkpoints.load = commit, load

''')


[Markdown Cell]

`p7_t4/kernels.py` — checked execution optimization.

In [121]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''"""Memory-tested microbatches, optional recomputation and exact cached prefix masks."""
import importlib
import time
import torch
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from pcr.state import capture_rng, restore_rng


def forward(self, memory, valid, role):
    execution = getattr(self, '_p7_execution', {})
    block = execution.get('microbatch', 32) if self.training else execution.get('inference_batch', 128)
    free_limit = execution.get('no_checkpoint_cases', 0)
    recompute = self.training and len(memory) > free_limit
    values = []
    for first in range(0, len(memory), block):
        args = (memory[first:first + block], valid[first:first + block], role[first:first + block])
        values.append(checkpoint(self._encode, *args, use_reentrant=False) if recompute else self._encode(*args))
    return torch.cat(values)


def prefix_attend(self, query, kv, positions, key_valid=None):
    """Original real sparse edges + learned KV prefixes, reusing only a Boolean mask."""
    length = kv[0].shape[2]
    signature = (query.device, length, len(positions))
    cache = getattr(self, '_p7_prefix_masks', {})
    mask = cache.get(signature)
    if mask is None:
        route = importlib.import_module(type(self).__module__).sparse_key_indices
        indices, legal = route(positions, length, self.local_window, self.dilation, self.dilated_links, self.global_stride)
        counts = torch.zeros(len(positions), length, dtype=torch.int32, device=query.device)
        counts.scatter_add_(1, indices, legal.to(torch.int32))
        mask = (counts > 0)[None, None]
        if len(cache) >= 2:
            cache.clear()
        cache[signature] = mask
        self._p7_prefix_masks = cache
    allowed = mask.expand(query.shape[0], 1, -1, -1)
    if key_valid is not None:
        allowed = allowed & key_valid[:, None, None, :]
''')


In [122]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''    prefix = self.prefix_key.shape[1]
    allowed = torch.cat((torch.ones(query.shape[0], 1, len(positions), prefix,
                        dtype=torch.bool, device=query.device), allowed), dim=-1)
    key = torch.cat((self.prefix_key[None].expand(query.shape[0], -1, -1, -1).to(kv[0].dtype), kv[0]), dim=2)
    value = torch.cat((self.prefix_value[None].expand(query.shape[0], -1, -1, -1).to(kv[1].dtype), kv[1]), dim=2)
    output = F.scaled_dot_product_attention(query, key, value, attn_mask=allowed,
        dropout_p=self.dropout if self.training else 0., is_causal=False)
    return self.out(output.transpose(1, 2).contiguous().flatten(2))


def tune(model, cfg, device, options):
    """Measure real forward/backward peak memory and rate; never update weights.

    Test with worst-case memory length and both document roles. Preserve RNG and
    mode, clear probe gradients, and reserve space for optimizer moments/input
    prefetch. Checkpoint-free execution is only used up to a tested case count.
    """
    if device.type != 'cuda':
        model._p7_execution = {'microbatch': cfg.encode_batch_size, 'no_checkpoint_cases': 0,
                               'inference_batch': options['inference_batch']}
        return {'status': 'CPU_fixture', **model._p7_execution}
    saved_rng, mode = capture_rng(device), model.training
    started = time.monotonic()
    results = []
    cap = int(options['max_microbatch'])
    batches = [value for value in (16, 32, 64, 128, 256) if value <= cap]
    if not batches:
        batches = [cap]
    free, total = torch.cuda.mem_get_info(device)
    initial = torch.cuda.memory_allocated(device)
    active = sum(value.numel() * value.element_size() for value in model.parameters() if value.requires_grad)
    budget = min(total * options['memory_fraction'], initial + max(0, free - 3 * active - (384 << 20)))
    try:
        model.train()
        for recompute in (False, True):
            for batch in batches:
                if results and time.monotonic() - started >= options['tuning_seconds']:
                    break
                model._p7_execution = {'microbatch': batch, 'no_checkpoint_cases': 0 if recompute else batch,
                                      'inference_batch': options['inference_batch']}
''')


In [123]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''                inputs = None
                try:
                    inputs = (torch.randn(batch, cfg.memory_slots, model.base_cfg.d_model, device=device, dtype=torch.float16) * .1,
                              torch.ones(batch, cfg.memory_slots, device=device, dtype=torch.bool),
                              torch.arange(batch, device=device) % 2)
                    # Warm up cuDNN/SDPA before measuring this actual shape.
                    model.zero_grad(set_to_none=True)
                    with torch.autocast('cuda', dtype=torch.float16):
                        model(*inputs).sum().backward()
                    torch.cuda.synchronize(device)
                    model.zero_grad(set_to_none=True)
                    torch.cuda.reset_peak_memory_stats(device)
                    begin = time.perf_counter()
                    with torch.autocast('cuda', dtype=torch.float16):
                        model(*inputs).sum().backward()
                    torch.cuda.synchronize(device)
                    elapsed = time.perf_counter() - begin
                    peak = torch.cuda.max_memory_allocated(device)
                    results.append({'batch': batch, 'checkpointed': recompute, 'seconds': elapsed,
                                    'cases_per_second': batch / max(elapsed, 1e-9), 'peak_bytes': peak,
                                    'fits_safety_budget': peak <= budget})
                    if peak > budget:
                        break
                except torch.cuda.OutOfMemoryError:
                    results.append({'batch': batch, 'checkpointed': recompute, 'OOM': True, 'fits_safety_budget': False})
                    break
                finally:
                    model.zero_grad(set_to_none=True)
                    del inputs
                    torch.cuda.empty_cache()
        valid = [row for row in results if row.get('fits_safety_budget')]
        if not valid:
            raise RuntimeError('No tested microbatch fits the safety budget. Close other GPU sessions or reduce MAX_MICROBATCH; keep checkpoints.')
        winner = max(valid, key=lambda row: row['cases_per_second'])
        no_checkpoint = max([row['batch'] for row in valid if not row['checkpointed']] or [0])
        # A faster measured recomputed kernel can still be preferable on a small
        # device; avoid disabling checkpoints just because some small batch fits.
        if winner['checkpointed']:
            no_checkpoint = 0
        model._p7_execution = {'microbatch': winner['batch'], 'no_checkpoint_cases': no_checkpoint,
''')


In [124]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''                               'inference_batch': options['inference_batch']}
        return {'status': 'measured', 'device': torch.cuda.get_device_name(device), 'total_bytes': total,
                'safety_budget_bytes': int(budget), 'elapsed_seconds': time.monotonic() - started,
                'execution': model._p7_execution, 'trials': results, 'weights_updated': False, 'RNG_restored': True}
    finally:
        model.zero_grad(set_to_none=True)
        model.train(mode)
        restore_rng(saved_rng, device)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
''')


[Markdown Cell]

`p7_t4/launch.py` — checked execution optimization.

In [125]:
# [Code Cell]
T4_PARTS.setdefault('launch.py', []).append(r'''"""Install the same execution policy in parent processes and NCCL spawn children."""
import argparse
import os
import sys
from pcr.io_utils import read_json
from .control import bind, install

if os.environ.get('CS781_P7_T4_PROFILE'):
    install(read_json(os.environ['CS781_P7_T4_PROFILE']))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('workflow', 'cache', 'train', 'export'))
    parser.add_argument('--options')
    args, remaining = parser.parse_known_args()
    if args.action == 'workflow':
        if not args.options or '--plan' not in remaining:
            parser.error('workflow requires --options and --plan')
        plan = remaining[remaining.index('--plan') + 1]
        install(bind(plan, args.options))
    elif not os.environ.get('CS781_P7_T4_PROFILE'):
        parser.error('Use the checked T4-optimized workflow launcher')
    from pcr import workflow, train, export
    from . import cache
    sys.argv = ['p7_t4.' + args.action, *remaining]
    {'workflow': workflow.main, 'cache': cache.main, 'train': train.main, 'export': export.main}[args.action]()


if __name__ == '__main__':
    main()
''')


In [126]:
# [Code Cell]
T4_SOURCE_SHA256 = {'__init__.py': 'b137646b725a1d25903d7974fe5083987f32109e5637a1421d277f8f08a6570b', 'batches.py': 'e50753d2d46c82c650f1550a7264b710a55f6ea08c3e7623c2a1cbc48e9bf546', 'cache.py': 'c89cbb2bfb4fee0c0740b360723a72859363cd3b70ebd50bf72fd67ed5991daa', 'control.py': 'c9f95c77830a6e47f9bf44c6b8005f157f4be9799cebd929c9da62859d886b99', 'kernels.py': '88c54536b9b3f7a5c78c8d80089c157c0234450c4548553f36c4233c53d92553', 'launch.py': '8fc470d4cd8a2295ca6eed45a18d66c80483b2845f0616495b3daa8c8af226b3'}
t4_directory = code_directory / 'p7_t4'
t4_directory.mkdir(exist_ok=True)
for name, pieces in T4_PARTS.items():
    text = ''.join(pieces)
    assert hashlib.sha256(text.encode()).hexdigest() == T4_SOURCE_SHA256[name], name
    compile(text, name, 'exec')
    target = t4_directory / name
    if (Path(OUTPUT_ROOT) / 't4_execution_profile.json').is_file() and target.is_file() and target.read_bytes() != text.encode():
        raise ValueError('Saved optimized source changed. Use its exact optimized notebook.')
    target.write_bytes(text.encode())
t4_options_path = Path(OUTPUT_ROOT) / 't4_options.json'
t4_options_path.write_text(json.dumps({'cache_batch': T4_CACHE_BATCH,
    'max_microbatch': T4_MAX_MICROBATCH, 'inference_batch': T4_INFERENCE_BATCH,
    'memory_fraction': T4_MEMORY_FRACTION, 'tuning_seconds': T4_TUNING_SECONDS,
    'prefetch': T4_PREFETCH}, indent=2), encoding='utf-8')
print('T4 x2 execution modules verified. Original learning/source identity is preserved.')


T4 x2 execution modules verified. Original learning/source identity is preserved.


[Markdown Cell]

### Workflow reporting import fix
The checked launcher supplies the missing `json` import inside the workflow process. Existing learning/runtime and optimized execution identities are preserved.

In [127]:
# [Code Cell]
workflow_entry_source = '"""Supply the missing reporting import without changing saved runtime identities.\n\nThe original workflow resolves json.dumps before it calls execute(). Keeping this\ncompatibility import in the entry module preserves existing model, optimizer,\nsource fingerprint, packed cache and T4 execution-profile recovery identities.\n"""\nimport json\nfrom pcr import workflow\n\nworkflow.json = json\n\n\ndef main():\n    from p7_t4.launch import main as launch\n    launch()\n\n\nif __name__ == \'__main__\':\n    main()\n'
assert hashlib.sha256(workflow_entry_source.encode()).hexdigest() == '7e66b05da73255304dbe679fb60fd2f10713ab636ed3a4f9502b936db9a3bf69'
workflow_entry_path = code_directory / 'p7_workflow_entry.py'
workflow_entry_path.write_text(workflow_entry_source, encoding='utf-8')



496

[Markdown Cell]

## 3. Validate settings and inherit exact saved recovery choices

In [128]:
# [Code Cell]
PARENT_CATALOG = {'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}


In [129]:
# [Code Cell]
from dataclasses import asdict
from pcr.config import PCRConfig
from pcr.io_utils import atomic_json, read_json
requested = dict(experiment_id=EXPERIMENT_ID, family=FAMILY, method=METHOD, run_dir=OUTPUT_ROOT,
    input_root=INPUT_ROOT, prepared_dir=PREPARED_DIR, parent_run_dir=PARENT_RUN_DIR,
    tokenizer_path=TOKENIZER_PATH, metadata_path=METADATA_PATH, resume_run_dir=RESUME_RUN_DIR,
    feature_cache_dir=FEATURE_CACHE_DIR, teacher_run_dir=TEACHER_RUN_DIR,
    max_epochs=MAX_EPOCHS, target_total_epochs=TARGET_TOTAL_EPOCHS,
    max_run_seconds=int(TRAINING_MINUTES * 60), cache_run_seconds=int(CACHE_MINUTES * 60),
    chunk_tokens=CHUNK_TOKENS, tokens_per_chunk=TOKENS_PER_CHUNK, memory_slots=MEMORY_SLOTS,
    cache_chunk_batch=CACHE_CHUNK_BATCH, train_last_units=TRAIN_LAST_UNITS,
    query_batch_per_rank=QUERY_BATCH_PER_RANK, encode_batch_size=ENCODE_BATCH_SIZE,
    learning_rate=LEARNING_RATE, head_learning_rate=HEAD_LEARNING_RATE)
local = Path(OUTPUT_ROOT) / 'job.json'
paths = [local] if local.is_file() else [Path(RESUME_RUN_DIR) / 'job.json'] if RESUME_RUN_DIR else list(Path(INPUT_ROOT).rglob('job.json'))
prior = [(file, read_json(file)) for file in paths if file.is_file()]
prior = [(file, value) for file, value in prior if value.get('version') == 'cs781-phase7-pcr-v1'
         and value.get('config', {}).get('experiment_id') == EXPERIMENT_ID]
if prior:
    if len({value['fingerprint'] for _, value in prior}) != 1:
        raise ValueError('Conflicting task versions. Set RESUME_RUN_DIR explicitly.')
    settings = prior[0][1]['config'].copy()
    transport = {'run_dir','input_root','prepared_dir','parent_run_dir','tokenizer_path','metadata_path',
        'resume_run_dir','feature_cache_dir','teacher_run_dir','max_run_seconds','cache_run_seconds',
        'cache_chunk_batch','target_total_epochs'}
    settings.update({key: value for key, value in requested.items() if key in transport and value is not None})
    print('Inherited the saved model, update plan and learning settings.')
else:
    if RESUME_RUN_DIR or REQUIRE_EXISTING_CHECKPOINT:
        raise ValueError('The SAME saved task was not found; a fresh restart was not allowed.')
    settings = asdict(PCRConfig(**requested))
config = PCRConfig(**settings)
config.validate()
plan_path = Path(OUTPUT_ROOT) / 'pipeline_plan.json'
catalog_path = code_directory / 'parent_catalog.json'
atomic_json(plan_path, {'mode': MODE, 'config': asdict(config)})
atomic_json(catalog_path, PARENT_CATALOG)
print(json.dumps(asdict(config), indent=2))


Inherited the saved model, update plan and learning settings.
{
  "experiment_id": "P7_05_D_Prefix",
  "family": "d",
  "method": "prefix",
  "run_dir": "/kaggle/working/cs781_phase7_score/P7_05_D_Prefix",
  "input_root": "/kaggle/input",
  "prepared_dir": null,
  "parent_run_dir": null,
  "tokenizer_path": null,
  "metadata_path": null,
  "resume_run_dir": null,
  "feature_cache_dir": null,
  "teacher_run_dir": null,
  "parent_checkpoint": "best",
  "world_size": 2,
  "seed": 781,
  "max_epochs": 12,
  "target_total_epochs": null,
  "minimum_epochs": 3,
  "patience": 3,
  "max_run_seconds": 2100,
  "cache_run_seconds": 2100,
  "min_free_bytes": 1073741824,
  "chunk_tokens": 256,
  "tokens_per_chunk": 4,
  "memory_slots": 128,
  "cache_chunk_batch": 16,
  "train_last_units": 2,
  "rank": 16,
  "alpha": 16.0,
  "prompt_tokens": 32,
  "prefix_tokens": 16,
  "embedding_dim": 256,
  "query_batch_per_rank": 4,
  "encode_batch_size": 16,
  "hard_negatives": 12,
  "random_negatives": 4,
  "le

[Markdown Cell]

### Importable supervised ranking modules
These small cells write checked public-text features, the from-scratch auxiliary learner, and own-neural fusion. No external model is loaded.

In [130]:
# [Code Cell]
SCORE_PARTS = {}


[Markdown Cell]

`p7_rank/__init__.py`

In [131]:
# [Code Cell]
SCORE_PARTS.setdefault('__init__.py', []).append(r'''"""Official-text learning-to-rank after the six own Phase-1 adaptation methods."""
VERSION = 'p7-official-text-ranker-v1'
''')


[Markdown Cell]

`p7_rank/compare.py`

In [132]:
# [Code Cell]
SCORE_PARTS.setdefault('compare.py', []).append(r'''"""Compare real enhanced submissions and tune a small optional rank ensemble."""
from pathlib import Path
import shutil
import numpy as np
from pcr.io_utils import atomic_json, read_json, sha256_file
from pcr.ranking import inverse_rank, metrics, package
from pcr.state import Checkpoints
from .learner import predictions


def run(input_root, prepared, output):
    root, data = Path(output), Path(prepared)
    root.mkdir(parents=True, exist_ok=True)
    truth, selection = read_json(data / 'dev_truth.json'), read_json(data / 'selection_split.json')
    accepted, rejected, seen = [], [], {}
    for file in Path(input_root).rglob('score_report.json'):
        report = read_json(file)
        directory, encoder = file.parent, file.parent.parent
        try:
            job = read_json(encoder / 'job.json')
            if report.get('version') != 'p7-score-enhanced-v1' or report['status'] != 'submission_verified':
                raise ValueError('Not a verified enhanced submission')
            if report['prepared_identity'] != read_json(data / 'prepared_manifest.json')['identity']:
                raise ValueError('Prepared dataset differs')
            if sha256_file(directory / Path(report['zip_path']).name) != report['zip_sha256']:
                raise ValueError('Submission ZIP changed')
            manager = Checkpoints(encoder, job['fingerprint'], 0)
            if report['encoder_checkpoint_sha256'] not in {item[1]['sha256'] for item in manager.candidates()}:
                raise ValueError('Actual own encoder checkpoint missing')
            key = report['family'], report['method']
            if key not in seen or report['tune']['micro_f1_at7'] > seen[key][1]['tune']['micro_f1_at7']:
                seen[key] = directory, report
        except (ValueError, OSError, KeyError) as error:
            rejected.append({'root': str(directory), 'reason': str(error)})
    accepted = list(seen.values())
    if not accepted:
        atomic_json(root / 'rejected_inputs.json', rejected)
        raise FileNotFoundError('Attach complete enhanced P7S model outputs including actual encoder weights and score_boost/.')
    info = accepted[0][1]['score_IDs']
    matrices = {'dev': [], 'test': []}
''')


In [133]:
# [Code Cell]
SCORE_PARTS.setdefault('compare.py', []).append(r'''    for directory, report in accepted:
        if report['score_IDs'] != info:
            raise ValueError('Ensemble pool/query ID order differs')
        for split in matrices:
            if sha256_file(directory / (split + '_hybrid.npy')) != report['score_files'][split + '_hybrid.npy']:
                raise ValueError('Saved hybrid score matrix changed')
            values = np.load(directory / (split + '_hybrid.npy'), mmap_mode='r')
            matrices[split].append(np.stack([1 / (60 + inverse_rank(row)) for row in values]))
    winner = max(range(len(accepted)), key=lambda i: accepted[i][1]['tune']['micro_f1_at7'])
    weights = np.zeros(len(accepted), np.float32)
    weights[winner] = 1
    best = accepted[winner][1]['tune']['micro_f1_at7']
    trials = []
    for index in range(len(accepted)):
        for amount in (.15, .3, .5):
            trial = weights * (1 - amount)
            trial[index] += amount
            values = sum(float(weight) * matrix for weight, matrix in zip(trial, matrices['dev']))
            rows = predictions(values, info['dev']['query_ids'], info['dev']['candidate_ids'])
            evaluated = metrics(truth, rows, info['dev']['candidate_ids'], selection['tune_ids'])
            trials.append({'added': index, 'weight': amount, 'tune': evaluated})
            if evaluated['micro_f1_at7'] > best:
                best, weights = evaluated['micro_f1_at7'], trial
    # Freeze weights before inspecting confirmation or producing test rankings.
    atomic_json(root / 'frozen_ensemble.json', {'weights': weights.tolist(), 'selected_on': 'dev_tune_only'})
    rows = {}
    for split in matrices:
        values = sum(float(weight) * matrix for weight, matrix in zip(weights, matrices[split]))
        rows[split] = predictions(values, info[split]['query_ids'], info[split]['candidate_ids'])
    receipt = package(root, rows['test'], info['test']['query_ids'], info['test']['candidate_ids'], 'P7S_Validated_Ensemble')
    shutil.copyfile(accepted[winner][0] / Path(accepted[winner][1]['zip_path']).name, root / 'best_single_submission.zip')
    required = {('a', 'lorma'), ('a', 'partial'), ('b', 'prompt'), ('c', 'ia3'), ('d', 'prefix'), ('b', 'kd')}
    report = {'status': 'submission_verified', 'models': [report['experiment_id'] for _, report in accepted],
        'methods': [list(key) for key in seen], 'missing_required': [list(key) for key in sorted(required - set(seen))],
        'weights': weights.tolist(), 'trials': trials, 'rejected': rejected,
        'tune': metrics(truth, rows['dev'], info['dev']['candidate_ids'], selection['tune_ids']),
        'confirmation': metrics(truth, rows['dev'], info['dev']['candidate_ids'], selection['confirm_ids']),
        'official_dev': metrics(truth, rows['dev'], info['dev']['candidate_ids']),
        'test_score': None, 'confirmation_used_for_selection': False, **receipt}
    atomic_json(root / 'score_comparison.json', report)
''')


In [134]:
# [Code Cell]
SCORE_PARTS.setdefault('compare.py', []).append(r'''    return report
''')


[Markdown Cell]

`p7_rank/learner.py`

In [135]:
# [Code Cell]
SCORE_PARTS.setdefault('learner.py', []).append(r'''"""Train an auxiliary ranker from scratch on all official training pairs.

This CPU tree ranker complements the already adapted own neural models; it is
not represented as an additional PEFT technique. It has no Adam/scaler state.
Warm-started trees, iteration counters and data/source identities are checkpointed.
"""
from pathlib import Path
import os
import time
import joblib
import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from pcr.io_utils import atomic_json, identity, read_json, sha256_file
from pcr.ranking import metrics
from .text import NAMES


def training_view(values, root: Path, reserve_bytes=1 << 30):
    """Disk-backed F64/Fortran input avoids sklearn's large implicit dtype/order copies."""
    shape = (values.shape[0] * values.shape[1], values.shape[2])
    path, receipt = root / 'training_pairs_f64.bin', root / 'training_pairs_f64.json'
    import hashlib
    filename = getattr(values, 'filename', None)
    digest = sha256_file(Path(filename)) if filename else hashlib.sha256(values.tobytes()).hexdigest()
    expected = {'shape': list(shape), 'source_shape': list(values.shape), 'dtype': 'float64', 'order': 'F', 'source_sha256': digest}
    cached = read_json(receipt) if receipt.is_file() else {}
    if not (path.is_file() and {k: v for k, v in cached.items() if k != 'sha256'} == expected
            and cached.get('sha256') == sha256_file(path)
            and path.stat().st_size == shape[0] * shape[1] * 8):
        from pcr.io_utils import ensure_space
        ensure_space(root, shape[0] * shape[1] * 8, reserve_bytes)
        output = np.memmap(path, np.float64, 'w+', shape=shape, order='F')
        flat = values.reshape(shape)
        for column in range(shape[1]):
            for first in range(0, shape[0], 131072):
                output[first:first + 131072, column] = flat[first:first + 131072, column]
        output.flush()
        del output
        atomic_json(receipt, {**expected, 'sha256': sha256_file(path)})
''')


In [136]:
# [Code Cell]
SCORE_PARTS.setdefault('learner.py', []).append(r'''    return np.memmap(path, np.float64, 'r', shape=shape, order='F')


def predict_matrix(model, values: np.ndarray) -> np.ndarray:
    values = values[:, :, :model.n_features_in_]
    shape = values.shape[:2]
    result = np.empty(shape, np.float32)
    for first in range(0, len(values), 32):
        part = values[first:first + 32]
        result[first:first + len(part)] = model.decision_function(part.reshape(-1, part.shape[-1])).reshape(part.shape[:2])
    return result


def predictions(values: np.ndarray, query_ids: list[str], candidate_ids: list[str]) -> dict:
    return {key: [candidate_ids[int(index)] for index in np.argsort(-values[row], kind='stable')[:7]]
            for row, key in enumerate(query_ids)}


def dump(model, path: Path):
    temporary = path.with_suffix('.partial')
    joblib.dump(model, temporary, compress=1)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
    temporary.replace(path)


def train_variant(prepared: str | Path, features: dict, output: str | Path, budget_seconds=2400,
                  feature_count=10, balance='query', storage_root=None) -> dict:
    """Every Q x C pair is used; dev tune selects trees, confirmation never fits/selects."""
    data, root, source = Path(prepared), Path(output), Path(features['directory'])
    root.mkdir(parents=True, exist_ok=True)
    settings = {'trees': [20, 40, 80, 120, 200, 320], 'learning_rate': .08, 'leaves': 15, 'depth': 4,
                'minimum_leaf': 80, 'l2': 10., 'seed': 781, 'feature_count': feature_count, 'balance': balance}
    signature = identity({'features': features['identity'], 'train_truth': sha256_file(data / 'train_truth.json'),
        'tune_partition': sha256_file(data / 'selection_split.json'), 'settings': settings,
        'source': sha256_file(Path(__file__)), 'sklearn': sklearn.__version__})
    receipt_path, latest, best_path = root / 'ranker_progress.json', root / 'ranker_latest.joblib', root / 'ranker_best.joblib'
    if receipt_path.is_file():
        progress = read_json(receipt_path)
        if progress['identity'] != signature or progress['latest_sha256'] != sha256_file(latest):
''')


In [137]:
# [Code Cell]
SCORE_PARTS.setdefault('learner.py', []).append(r'''            raise ValueError('Ranker source/data/checkpoint changed; resume its exact output')
        if progress['complete']:
            completed = read_json(root / 'ranker_report.json')
            if completed['best_model_sha256'] != sha256_file(best_path):
                raise ValueError('Saved auxiliary best model is corrupted')
            for name, digest in completed['score_files'].items():
                if sha256_file(root / name) != digest:
                    raise ValueError('Saved auxiliary score cache is corrupted: ' + name)
            return completed
        model = joblib.load(latest)  # own verified saved output, never external pretrained objects
    else:
        progress = {'identity': signature, 'iterations': 0, 'best_tune': -1., 'history': [], 'complete': False}
        model = HistGradientBoostingClassifier(max_iter=20, warm_start=True, early_stopping=False,
            learning_rate=.08, max_leaf_nodes=15, max_depth=4, min_samples_leaf=80,
            l2_regularization=10., random_state=781)
    x = np.load(source / 'train_text.npy', mmap_mode='r')
    xd = np.load(source / 'dev_text.npy', mmap_mode='r')
    train_info, dev = features['splits']['train'], features['splits']['dev']
    truth = read_json(data / 'train_truth.json')
    by_id = {key: index for index, key in enumerate(train_info['candidate_ids'])}
    target = np.zeros(x.shape[:2], np.bool_)
    for row, key in enumerate(train_info['query_ids']):
        for candidate in truth[key]:
            if candidate in by_id:
                target[row, by_id[candidate]] = True
    counts = target.sum(1)
    weights = np.where(target, 1 / np.maximum(1, counts[:, None]),
                       1 / np.maximum(1, x.shape[1] - counts[:, None])).astype(np.float32)
    if balance == 'global':
        weights = np.where(target, target.size / max(1, int(target.sum())),
                            target.size / max(1, int((~target).sum()))).astype(np.float32)
    weights *= target.size / weights.sum()
    dev_truth, selection = read_json(data / 'dev_truth.json'), read_json(data / 'selection_split.json')
    started = time.monotonic()
    training = training_view(x, Path(storage_root) if storage_root else root)[:, :feature_count]
    for count in settings['trees']:
        if count <= progress['iterations']:
            continue
        if time.monotonic() - started >= budget_seconds:
            atomic_json(root / 'pause_report.json', {'status': 'paused_recoverable', 'next_action': 'Attach complete 00 output and resume 00.'})
''')


In [138]:
# [Code Cell]
SCORE_PARTS.setdefault('learner.py', []).append(r'''            return {'status': 'paused_recoverable'}
        begin = time.monotonic()
        print(f'Train auxiliary ranker to {count} trees on ALL {target.size:,} pairs / {len(target)} queries.', flush=True)
        model.set_params(max_iter=count)
        model.fit(training, target.ravel(), sample_weight=weights.ravel())
        scores = predict_matrix(model, xd)
        rows = predictions(scores, dev['query_ids'], dev['candidate_ids'])
        evaluated = metrics(dev_truth, rows, dev['candidate_ids'], selection['tune_ids'])
        dump(model, latest)
        if evaluated['micro_f1_at7'] > progress['best_tune']:
            dump(model, best_path)
            progress['best_tune'], progress['best_trees'] = evaluated['micro_f1_at7'], count
        progress['history'].append({'trees': count, 'seconds': time.monotonic() - begin, 'tune': evaluated})
        progress.update(iterations=count, latest_sha256=sha256_file(latest), complete=False)
        atomic_json(receipt_path, progress)
        print({'trees': count, 'tune_F1': evaluated['micro_f1_at7']}, flush=True)
    best = joblib.load(best_path)
    score_files = {}
    for split in ('dev', 'test'):
        values = np.load(source / (split + '_text.npy'), mmap_mode='r')
        result = predict_matrix(best, values)
        path = root / (split + '_learned_text.npy')
        np.save(path, result)
        score_files[path.name] = sha256_file(path)
    progress['complete'] = True
    report = {'status': 'complete', 'identity': signature, 'feature_identity': features['identity'],
        'best_trees': progress['best_trees'], 'best_tune': progress['best_tune'], 'history': progress['history'],
        'training_queries': len(target), 'all_training_query_candidate_pairs': int(target.size),
        'positive_training_pairs': int(target.sum()), 'fit_labels_from_official_train_only': True,
        'feature_count': feature_count, 'balance': balance,
        'confirmation_used_for_selection': False, 'test_labels_used': False,
        'auxiliary_ranker_is_not_a_separate_PEFT_method': True,
        'score_files': score_files, 'best_model_sha256': sha256_file(best_path), 'sklearn': sklearn.__version__}
    atomic_json(root / 'ranker_report.json', report)
    atomic_json(receipt_path, progress)
    return report


def train(prepared: str | Path, features: dict, output: str | Path, budget_seconds=2400) -> dict:
    """Small declared ablation: core/expanded features and query/global class balance.
''')


In [139]:
# [Code Cell]
SCORE_PARTS.setdefault('learner.py', []).append(r'''
    All variants visit the SAME complete training pairs. Only the existing tune
    partition chooses the final variant/tree count; no confirmation/test labels
    enter this procedure. A single disk-backed training input is shared.
    """
    import shutil
    root = Path(output)
    root.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    variants = [('core_query', 7, 'query'), ('expanded_query', 10, 'query'), ('expanded_global', 10, 'global')]
    reports = []
    for name, count, balance in variants:
        remaining = max(0., budget_seconds - (time.monotonic() - started))
        value = train_variant(prepared, features, root / name, budget_seconds=remaining,
                              feature_count=count, balance=balance, storage_root=root)
        if value.get('status') != 'complete':
            atomic_json(root / 'ranker_progress.json', {'complete': False, 'paused_variant': name,
                'next_action': 'Save complete 00 output and resume 00; completed variants/tree blocks are retained.'})
            return {'status': 'paused_recoverable'}
        reports.append((name, value))
    name, selected = max(reports, key=lambda item: (item[1]['best_tune'], -item[1]['feature_count']))
    for file in ('ranker_best.joblib', 'ranker_latest.joblib', 'dev_learned_text.npy', 'test_learned_text.npy'):
        shutil.copyfile(root / name / file, root / file)
    result = {**selected, 'selected_variant': name, 'variant_trials': [
        {'variant': key, 'tune': row['best_tune'], 'trees': row['best_trees'],
         'features': row['feature_count'], 'balance': row['balance']} for key, row in reports]}
    atomic_json(root / 'ranker_report.json', result)
    atomic_json(root / 'ranker_progress.json', {'complete': True, 'selected_variant': name,
        'best_model_sha256': result['best_model_sha256'], 'all_variants_complete': True})
    return result
''')


[Markdown Cell]

`p7_rank/text.py`

In [140]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''"""Complete-document lexical/section/passage and public reporter-reference features.

No relevance labels, candidate IDs as predictors, pretrained weights, external
corpora or future CITATOR INFO annotations are used by this feature constructor.
Every query/document remains in its original split and candidate order.
"""
from pathlib import Path
import math
import re
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tqdm.auto import tqdm
from pcr.io_utils import atomic_json, identity, read_json, records, sha256_file
from pcr.preparation import verify_prepared

NAMES = ['word_cosine', 'phrase_cosine', 'bm25_idf2', 'direct_reporter_reference',
         'shared_reporter_cosine', 'headnote_cosine', 'max_passage_cosine',
         'query_passage_max_word_cosine', 'query_headnote_cosine', 'both_headnote_cosine']


def clean(text: str) -> str:
    """Remove placeholders and future citation-index annotations, retaining body text."""
    text = re.sub(r'CITATOR\s+INFO\s*:.*?(?=\bACT\s*:|\bHEADNOTE\s*:|\bJUDGMENT\b|\bJUDGEMENT\b|$)', ' ', text, flags=re.I | re.S)
    return re.sub(r'<[^>]+>', ' ', text)


def citations(text: str) -> set[str]:
    """Canonicalize reporter/year/volume/page strings, never IndianKanoon case IDs."""
    value = re.sub(r'[.\[\](),:;]', ' ', text.upper())
    for spaced, compact in ((r'S\s+C\s+R', 'SCR'), (r'S\s+C\s+C', 'SCC'),
                            (r'A\s+I\s+R', 'AIR'), (r'I\s+T\s+R', 'ITR'), (r'S\s+C', 'SC')):
        value = re.sub(r'\b' + spaced + r'\b', compact, value)
    value = re.sub(r'(?<=[A-Z])(?=\d)|(?<=\d)(?=[A-Z])', ' ', value)
    year, found = r'(?:18|19|20)\d{2}', set()
    for m in re.finditer(rf'\b({year})\s+AIR\s+(?:SC\s*)?(\d+)\b|\bAIR\s+({year})\s+(?:SC\s*)?(\d+)\b', value):
        y, page = (m[1], m[2]) if m[1] else (m[3], m[4])
        found.add('AIR:' + y + ':' + str(int(page)))
    for m in re.finditer(rf'\b({year})\s+(\d+)\s+(SCR|SCC|ITR|STC|LLJ|MLJ)\s+(\d+)\b|\b({year})\s+(SCR|SCC|ITR|STC|LLJ|MLJ)\s+(\d+)\s+(\d+)\b', value):
        y, vol, reporter, page = (m[1], m[2], m[3], m[4]) if m[1] else (m[5], m[7], m[6], m[8])
''')


In [141]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''        found.add(reporter + ':' + y + ':' + str(int(vol)) + ':' + str(int(page)))
    for m in re.finditer(rf'\b({year})\s+(SCR|SCC|ITR|STC)\s+(\d+)\b', value):
        # Some citations omit a volume. Keep an explicit volume-free alias;
        # avoid mistaking the first number of a full volume/page pair for a page.
        tail = value[m.end():]
        if not re.match(r'\s+\d+\b', tail):
            found.add(m[2] + ':' + m[1] + ':1:' + str(int(m[3])))
    return found


def primary(text: str) -> set[str]:
    m = re.search(r'\bCITATIONS?\s*:\s*(.*?)(?=CITATOR\s+INFO|\bACT\s*:|\bHEADNOTE\s*:|\bJUDGMENT\b|$)', text, flags=re.I | re.S)
    return citations(m[1]) if m else set()


def citation_features(queries: list[dict], cases: list[dict]) -> tuple[np.ndarray, np.ndarray]:
    vocabulary: dict[str, int] = {}
    def encode(sets):
        rows, columns = [], []
        for row, values in enumerate(sets):
            for key in sorted(values):
                rows.append(row)
                columns.append(vocabulary.setdefault(key, len(vocabulary)))
        return rows, columns
    a = encode([citations(clean(row['text'])) for row in queries])
    b = encode([primary(row['text']) for row in cases])
    c = encode([citations(clean(row['text'])) for row in cases])
    def matrix(values, length):
        return sparse.csr_matrix((np.ones(len(values[0]), np.float32), values), shape=(length, len(vocabulary)))
    query, own, full = matrix(a, len(queries)), matrix(b, len(cases)), matrix(c, len(cases))
    direct = (query @ own.T).toarray()
    denominator = np.sqrt(np.maximum(1, query.getnnz(1)))[:, None] * np.sqrt(np.maximum(1, full.getnnz(1)))[None]
    shared = (query @ full.T).toarray() / denominator
    return np.log1p(direct).astype(np.float32), shared.astype(np.float32)


def headnote(text: str) -> str:
    m = re.search(r'\bHEADNOTE\s*:\s*(.*?)(?=\bJUDGMENT\b|\bJUDGEMENT\b|$)', text, flags=re.I | re.S)
    return m[1] if m and m[1].strip() else text

''')


In [142]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''
def query_section_features(qtext, ctext, word=None, cword=None):
    """Other direction of local matching plus explicit query legal-summary channels."""
    if word is None:
        word = TfidfVectorizer(lowercase=True, sublinear_tf=True, stop_words='english',
            min_df=1, max_df=.95 if len(ctext) > 2 else 1., dtype=np.float32, token_pattern=r'(?u)\b\w\w+\b')
        cword = word.fit_transform(ctext)
    result = np.zeros((len(qtext), len(ctext), 3), np.float32)
    qhead, chead = word.transform([headnote(text) for text in qtext]), word.transform([headnote(text) for text in ctext])
    result[:, :, 1] = (qhead @ cword.T).toarray()
    result[:, :, 2] = (qhead @ chead.T).toarray()
    pending, owners = [], []
    def flush():
        if not pending:
            return
        scores = (word.transform(pending) @ cword.T).toarray()
        for index, owner in enumerate(owners):
            result[owner, :, 0] = np.maximum(result[owner, :, 0], scores[index])
        pending.clear()
        owners.clear()
    for owner, text in enumerate(qtext):
        words = text.split()
        covered = 0
        for first in range(0, len(words), 256):
            part = words[first:first + 256]
            covered += len(part)
            pending.append(' '.join(part))
            owners.append(owner)
            if len(pending) >= 128:
                flush()
        assert covered == len(words)
    flush()
    return result


def split_features(queries: list[dict], cases: list[dict], passage_words: int = 256) -> np.ndarray:
    """Build label-free Q x C x F features; all source words reach the full channels."""
    qtext, ctext = [clean(row['text']) for row in queries], [clean(row['text']) for row in cases]
    shape = (len(queries), len(cases))
    values = np.zeros((*shape, len(NAMES)), np.float32)
''')


In [143]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''    print(f'Build public-text features for {len(queries)} queries x {len(cases)} candidates. No documents are sampled.', flush=True)
    common = dict(lowercase=True, sublinear_tf=True, stop_words='english', min_df=1,
                  max_df=.95 if len(cases) > 2 else 1., dtype=np.float32, token_pattern=r'(?u)\b\w\w+\b')
    word = TfidfVectorizer(**common)
    try:
        cword = word.fit_transform(ctext)
    except ValueError as error:
        if 'After pruning, no terms remain' in str(error):
            word.set_params(max_df=1.)
            cword = word.fit_transform(ctext)
        elif 'empty vocabulary' in str(error):
            return values  # empty official texts are retained with zero lexical evidence
        else:
            raise
    qword = word.transform(qtext)
    values[:, :, 0] = (qword @ cword.T).toarray()
    phrase = TfidfVectorizer(**common, ngram_range=(1, 2), max_features=750_000)
    phrase.set_params(max_df=word.max_df)
    print('Index full-document word/phrase features; long-document indexing can take several minutes.', flush=True)
    cp = phrase.fit_transform(ctext)
    qp = phrase.transform(qtext)
    values[:, :, 1] = (qp @ cp.T).toarray()
    del cp
    counts = CountVectorizer(vocabulary=word.vocabulary_, lowercase=True,
                             token_pattern=r'(?u)\b\w\w+\b').transform(ctext).astype(np.float32)
    qcounts = CountVectorizer(vocabulary=word.vocabulary_, lowercase=True,
                              token_pattern=r'(?u)\b\w\w+\b').transform(qtext).astype(np.float32)
    frequency = np.bincount(counts.indices, minlength=counts.shape[1])
    idf = np.log1p((len(cases) - frequency + .5) / (frequency + .5)).astype(np.float32)
    length = np.asarray(counts.sum(1)).ravel()
    denominator = 1.2 * (.25 + .75 * length / max(1., float(length.mean())))
    counts.data = counts.data * 2.2 / (counts.data + np.repeat(denominator, np.diff(counts.indptr))) * idf[counts.indices]
    qcounts.data = np.log1p(qcounts.data) * idf[qcounts.indices]
    values[:, :, 2] = np.log1p((qcounts @ counts.T).toarray())
    values[:, :, 3], values[:, :, 4] = citation_features(queries, cases)
    values[:, :, 5] = (qword @ word.transform([headnote(text) for text in ctext]).T).toarray()
    # Full query -> each passage. Max over ALL candidate passages; never a first-N cutoff.
    pending, owners = [], []
    maximum = np.zeros(shape, np.float32)
    def flush():
''')


In [144]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''        if not pending:
            return
        scored = (qp @ phrase.transform(pending).T).toarray()
        for column, owner in enumerate(owners):
            maximum[:, owner] = np.maximum(maximum[:, owner], scored[:, column])
        pending.clear()
        owners.clear()
    for owner, text in enumerate(tqdm(ctext, desc='All document passages', unit='cases')):
        words = text.split()
        covered = 0
        for first in range(0, len(words), passage_words):
            part = words[first:first + passage_words]
            pending.append(' '.join(part))
            owners.append(owner)
            covered += len(part)
            if len(pending) >= 128:
                flush()
        assert covered == len(words)
    flush()
    values[:, :, 6] = maximum
    print('Match all query passages and legal summaries in the reverse direction.', flush=True)
    values[:, :, 7:] = query_section_features(qtext, ctext, word, cword)
    if not np.isfinite(values).all():
        raise FloatingPointError('Nonfinite public-text retrieval features')
    return values


def build(prepared: str | Path, output: str | Path) -> dict:
    data, root = Path(prepared), Path(output)
    root.mkdir(parents=True, exist_ok=True)
    source = {p.name: sha256_file(p) for p in (Path(__file__), Path(__file__).parent / '__init__.py')}
    signature = identity({'data': verify_prepared(data)['identity'], 'feature_source': source,
                          'names': NAMES, 'passage_words': 256, 'feature_dtype': 'float32'})
    manifest_path = root / 'text_features.json'
    if manifest_path.is_file():
        value = read_json(manifest_path)
        if value['identity'] != signature:
            raise ValueError('Saved text features differ; use a separate output for new settings/source')
        for name, digest in value['files'].items():
            if sha256_file(root / name) != digest:
''')


In [145]:
# [Code Cell]
SCORE_PARTS.setdefault('text.py', []).append(r'''                raise ValueError('Text feature checksum failed: ' + name)
        return {**value, 'directory': str(root)}
    splits, files = {}, {}
    for split in ('train', 'dev', 'test'):
        queries, cases = [list(records(data / f'{split}_{kind}.jsonl')) for kind in ('queries', 'candidates')]
        path = root / (split + '_text.npy')
        receipt = root / (split + '_text.json')
        metadata = {'query_ids': [r['id'] for r in queries], 'candidate_ids': [r['id'] for r in cases],
                    'shape': [len(queries), len(cases), len(NAMES)], 'identity': signature}
        if not (receipt.is_file() and path.is_file() and read_json(receipt).get('identity') == signature
                and read_json(receipt).get('sha256') == sha256_file(path)):
            values = split_features(queries, cases)
            temporary = path.with_suffix('.partial')
            with temporary.open('wb') as stream:
                np.save(stream, values)
            temporary.replace(path)
            atomic_json(receipt, {**metadata, 'sha256': sha256_file(path)})
            del values
        splits[split] = metadata
        files[path.name], files[receipt.name] = sha256_file(path), sha256_file(receipt)
    value = {'version': 'p7-text-features-v1', 'identity': signature, 'prepared_identity': verify_prepared(data)['identity'],
        'splits': splits, 'files': files, 'names': NAMES, 'all_source_words_processed': True,
        'candidate_IDs_used_as_features': False, 'test_labels_used': False,
        'future_CITATOR_INFO_annotations_used': False, 'feature_source': source}
    atomic_json(manifest_path, value)
    return {**value, 'directory': str(root)}
''')


[Markdown Cell]

`p7_rank/workflow.py`

In [146]:
# [Code Cell]
SCORE_PARTS.setdefault('workflow.py', []).append(r'''"""Measured own-neural + train-learned text ranking, with tune-only rule selection."""
import argparse
import os
from pathlib import Path
import zipfile
import numpy as np
from pcr.io_utils import atomic_json, read_json, sha256_file

# mp.spawn re-imports this module. Install the SAME checked execution layer in
# each scorer worker before pcr.export resolves its embedding implementation.
if os.environ.get('CS781_P7_T4_PROFILE'):
    from p7_t4.control import install
    install(read_json(os.environ['CS781_P7_T4_PROFILE']))

from pcr.ranking import inverse_rank, metrics, package
from .learner import predictions


def locate(input_root, explicit=None):
    paths = [Path(explicit)] if explicit else sorted({file.parent.parent for file in Path(input_root).rglob('text_features.json')})
    valid = [root for root in paths if (root / 'ranker/ranker_report.json').is_file()
             and read_json(root / 'ranker/ranker_report.json').get('status') == 'complete']
    signatures = {read_json(root / 'text_features/text_features.json')['identity'] for root in valid}
    if not valid or len(signatures) != 1:
        raise FileNotFoundError('Attach complete P7S_00 output (text_features/ + ranker/); set SCORE_DATA_ROOT if multiple versions exist.')
    return valid[0]


def lexical_base(root, split, style):
    features = np.load(root / 'text_features' / (split + '_text.npy'), mmap_mode='r')
    if style == 'learned':
        return np.load(root / 'ranker' / (split + '_learned_text.npy'), mmap_mode='r')
    # Predeclared literal/phrase diagnostic alternative. It uses public reporter
    # strings, never hardcoded query->case mappings or labels at inference.
    return .5 * features[:, :, 1] + .5 * features[:, :, 6] + .3 * (features[:, :, 3] > 0)


def fuse(neural, lexical, weight, constant):
    if neural.shape != lexical.shape:
        raise ValueError('Neural/text candidate shapes differ')
''')


In [147]:
# [Code Cell]
SCORE_PARTS.setdefault('workflow.py', []).append(r'''    result = np.empty(neural.shape, np.float32)
    for row in range(len(neural)):
        result[row] = weight / (constant + inverse_rank(neural[row])) + (1 - weight) / (constant + inverse_rank(lexical[row]))
    return result


def run(encoder_root, score_root, output):
    encoder, source, root = Path(encoder_root), Path(score_root), Path(output)
    root.mkdir(parents=True, exist_ok=True)
    job, baseline = read_json(encoder / 'job.json'), read_json(encoder / 'final_report.json')
    if baseline.get('status') != 'submission_verified' or read_json(encoder / 'training_summary.json').get('status') != 'complete':
        raise ValueError('The neural encoder is paused/incomplete. Resume its exact output before score enhancement.')
    features = read_json(source / 'text_features/text_features.json')
    trained = read_json(source / 'ranker/ranker_report.json')
    if features['prepared_identity'] != job['prepared_identity'] or trained['feature_identity'] != features['identity']:
        raise ValueError('Learned text ranker and encoder use different official prepared data')
    for name, digest in features['files'].items():
        if sha256_file(source / 'text_features' / name) != digest:
            raise ValueError('Text feature checksum failed: ' + name)
    for name, digest in trained['score_files'].items():
        if sha256_file(source / 'ranker' / name) != digest:
            raise ValueError('Learned text score checksum failed: ' + name)
    from pcr.state import Checkpoints
    manager = Checkpoints(encoder, job['fingerprint'], 0, job['resume_roots'])
    candidates = [item for item in manager.candidates() if item[1]['sha256'] == baseline['checkpoint_sha256']]
    if not candidates:
        raise FileNotFoundError('Selected actual neural checkpoint is missing; attach COMPLETE Kaggle model output.')
    profile = encoder / 't4_execution_profile.json'
    if profile.is_file():
        os.environ['CS781_P7_T4_PROFILE'] = str(profile)
        from p7_t4.control import install
        install(read_json(profile))
    from pcr.export import score_checkpoint
    directory = score_checkpoint(job, candidates[0], ('dev', 'test'))
    neural, info = {}, {}
    for split in ('dev', 'test'):
        receipt = read_json(directory / (split + '_scores.json'))
        expected = features['splits'][split]
        if receipt['query_ids'] != expected['query_ids'] or receipt['candidate_ids'] != expected['candidate_ids']:
            raise ValueError('Neural and full-text query/candidate ID order differs')
''')


In [148]:
# [Code Cell]
SCORE_PARTS.setdefault('workflow.py', []).append(r'''        neural[split] = np.memmap(directory / (split + '_scores.bin'), np.float32, 'r', shape=tuple(receipt['shape']))
        info[split] = receipt
    data = Path(job['prepared_dir'])
    truth, selection = read_json(data / 'dev_truth.json'), read_json(data / 'selection_split.json')
    def lexical_for(split, style):
        if style == 'baseline':
            from pcr.lexical import scores_for
            return scores_for(job['lexical'], split)
        return lexical_base(source, split, style)
    trials = []
    if 'lexical' in job and 'rule' in baseline.get('fusion', {}):
        old = baseline['fusion']['rule']
        baseline_values = fuse(neural['dev'], lexical_for('dev', 'baseline'), old['neural_weight'], old['rank_constant'])
        baseline_rows = predictions(baseline_values, info['dev']['query_ids'], info['dev']['candidate_ids'])
        baseline_tune = metrics(truth, baseline_rows, info['dev']['candidate_ids'], selection['tune_ids'])
        trials.append({'style': 'baseline', 'neural_weight': old['neural_weight'],
            'constant': old['rank_constant'], 'tune': baseline_tune})
    for style in ('learned', 'phrase_passage'):
        lexical = lexical_base(source, 'dev', style)
        for weight in (.1, .25, .5):
            for constant in (20, 60):
                values = fuse(neural['dev'], lexical, weight, constant)
                rows = predictions(values, info['dev']['query_ids'], info['dev']['candidate_ids'])
                evaluated = metrics(truth, rows, info['dev']['candidate_ids'], selection['tune_ids'])
                trials.append({'style': style, 'neural_weight': weight, 'constant': constant, 'tune': evaluated})
    winner = max(trials, key=lambda row: (row['tune']['micro_f1_at7'], row['style'] == 'baseline', row['neural_weight']))
    atomic_json(root / 'frozen_hybrid_rule.json', {'rule': winner, 'selected_on': 'existing_dev_tune_partition',
        'confirmation_used_for_selection': False, 'test_labels_used': False})
    saved = {}
    for split in ('dev', 'test'):
        values = fuse(neural[split], lexical_for(split, winner['style']), winner['neural_weight'], winner['constant'])
        np.save(root / (split + '_hybrid.npy'), values)
        saved[split] = predictions(values, info[split]['query_ids'], info[split]['candidate_ids'])
    name = 'P7S_' + job['config']['experiment_id'].removeprefix('P7_')
    receipt = package(root, saved['test'], info['test']['query_ids'], info['test']['candidate_ids'], name)
    atomic_json(root / 'dev_prediction.json', saved['dev'])
    report = {'version': 'p7-score-enhanced-v1', 'status': 'submission_verified',
        'experiment_id': name, 'family': job['config']['family'], 'method': job['config']['method'],
        'prepared_identity': job['prepared_identity'], 'feature_identity': features['identity'],
        'encoder_checkpoint_sha256': baseline['checkpoint_sha256'], 'encoder_root': str(encoder),
''')


In [149]:
# [Code Cell]
SCORE_PARTS.setdefault('workflow.py', []).append(r'''        'auxiliary_ranker_sha256': trained['best_model_sha256'], 'rule': winner, 'trials': trials,
        'tune': metrics(truth, saved['dev'], info['dev']['candidate_ids'], selection['tune_ids']),
        'confirmation': metrics(truth, saved['dev'], info['dev']['candidate_ids'], selection['confirm_ids']),
        'official_dev': metrics(truth, saved['dev'], info['dev']['candidate_ids']),
        'baseline_official_dev': baseline['fusion']['official_dev'],
        'learned_neural_contribution': winner['neural_weight'], 'encoder_weights_changed_by_reranker': False,
        'auxiliary_ranker_fit_on_all_official_train_pairs': True,
        'auxiliary_ranker_is_not_a_seventh_PEFT_technique': True,
        'baseline_retained_if_tune_not_improved': winner['style'] == 'baseline',
        'score_files': {s + '_hybrid.npy': sha256_file(root / (s + '_hybrid.npy')) for s in ('dev', 'test')},
        'test_labels_used': False, 'test_score': None, 'candidate_IDs_used_as_features': False,
        'source': {p.name: sha256_file(p) for p in Path(__file__).parent.glob('*.py')},
        'score_IDs': {s: {k: info[s][k] for k in ('query_ids', 'candidate_ids')} for s in info}, **receipt}
    atomic_json(root / 'score_report.json', report)
    with zipfile.ZipFile(root / 'score_observations.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in root.glob('*.json'):
            archive.write(path, path.name)
    return report


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--encoder-root', required=True)
    parser.add_argument('--input-root', default='/kaggle/input')
    parser.add_argument('--score-root')
    parser.add_argument('--output', required=True)
    args = parser.parse_args()
    import json
    print(json.dumps(run(args.encoder_root, locate(args.input_root, args.score_root), args.output), indent=2), flush=True)


if __name__ == '__main__':
    main()
''')


In [150]:
# [Code Cell]
SCORE_SOURCE_SHA256 = {'__init__.py': '6fa39ad70a284f1a430e47cae1690064dc499228b742c417cbc50c80f3af7b5c', 'compare.py': 'b176ef6e325a13b001012ebe76c6f803228acdd82be56373d6bc5fadfdb18fd7', 'learner.py': '33f604b73466ee913ec8df6411d34ab887a86bc1ad2be6de7e114947235cacc4', 'text.py': 'f19bd9217f8af20125830076a13f593e175e29c8af5e03e20224a3e8f6d229cc', 'workflow.py': '5f016447c9bc566c372a65b3465530e91daabb1a60566813acb9280037a32593'}
score_code = code_directory / 'p7_rank'
score_code.mkdir(exist_ok=True)
for name, pieces in SCORE_PARTS.items():
    source = ''.join(pieces)
    assert hashlib.sha256(source.encode()).hexdigest() == SCORE_SOURCE_SHA256[name], name
    compile(source, name, 'exec')
    (score_code / name).write_bytes(source.encode())
print('Verified score-improvement modules:', len(SCORE_SOURCE_SHA256))


Verified score-improvement modules: 5


[Markdown Cell]

### Check the shared ranking input before GPU work

In [151]:
# [Code Cell]
from p7_rank.workflow import locate
SCORE_DATA_ROOT = str(locate(INPUT_ROOT, SCORE_DATA_ROOT))
print('Complete shared ranking output:', SCORE_DATA_ROOT)


Complete shared ranking output: /kaggle/input/notebooks/stashit104/p7s-00-build-train-ranking-layer/cs781_phase7_score_data


[Markdown Cell]

## 4. Execute, monitor and recover safely

In [152]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = os.pathsep.join((str(compat_directory), str(code_directory), environment.get('PYTHONPATH', '')))
    environment['CS781_NUMPY_RESUME_REQUIRED'] = '1'
    process = subprocess.Popen(command, cwd=code_directory, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            try:
                ready, _, _ = select.select([process.stdout], [], [], 1.)
                if ready:
                    block = os.read(process.stdout.fileno(), 8192)
                    if block:
                        print(decoder.decode(block), end='', flush=True)
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                print('Requesting a saved stop. Keep this cell running until the child finishes.', flush=True)
                process.send_signal(signal.SIGINT)
        status = process.wait()
        print(decoder.decode(b'', final=True), end='', flush=True)
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        process.stdout.close()


In [153]:
# [Code Cell]
run_streamed([sys.executable, '-u', '-m', 'p7_workflow_entry', 'workflow', '--options', str(t4_options_path), '--plan', str(plan_path), '--catalog', str(catalog_path)])


GPU 0 Tesla T4
GPU 1 Tesla T4
GPU 0 Tesla T4
GPU 1 Tesla T4
Resume exact saved step 1248 epoch 12
Calibrating safe encoder batches on the actual GPU; no optimizer updates are performed.
Selected T4 execution: {'microbatch': 128, 'no_checkpoint_cases': 128, 'inference_batch': 128}
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
dev queries, GPU 0: 100%|██████████| 59/59 [00:00<00:00, 1126.22cases/s]
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
dev queries, GPU 0: 100%|██████████| 59/59 [00:00<00:00, 1203.99cases/s]
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: User

[Markdown Cell]

## 5. Verify the result and keep complete outputs

[Markdown Cell]

The submission ZIP contains exactly one root prediction.json with all test query IDs and seven
unique valid candidates each. Observations and notebook files are not weight checkpoints. Preserve
foundation_seed.pt too: it enables recovery if a session ends while frozen-cache building is incomplete.
For another epoch set TARGET_TOTAL_EPOCHS=E+1 after E completed epochs. Keep MAX_EPOCHS, architecture,
tokenizer and learning settings fixed on continuation. The scorer measures full gold sets at seven;
training/reference leakage, hardcoded relevance and private labels are never used.

[Markdown Cell]

### Keep the execution reports
Save `code/p7_t4/`, `t4_execution_profile.json` and both `t4_tuning_gpu*.json` files with the complete
original output. Cache plans, tokens and per-case journals also belong in the saved output.
Autotuning performs forward/backward probes without optimizer updates and restores RNG state.
Actual hardware-dependent batch/peak-memory/rate results are recorded, not invented. A larger
safe microbatch is an execution detail; the original global query/negative update plan is retained.

[Markdown Cell]

## 6. Produce the enhanced submission
Tune selects from a small predeclared neural/text fusion grid. Confirmation and test never select weights. The final score depends on actual results; 0.5 is a target, not a guaranteed private score.

In [154]:
# [Code Cell]
command = [sys.executable, '-u', '-m', 'p7_rank.workflow',
    '--encoder-root', str(Path(OUTPUT_ROOT)), '--input-root', INPUT_ROOT,
    '--output', str(Path(OUTPUT_ROOT) / 'score_boost')]
if SCORE_DATA_ROOT:
    command += ['--score-root', SCORE_DATA_ROOT]
run_streamed(command)


{
  "version": "p7-score-enhanced-v1",
  "status": "submission_verified",
  "experiment_id": "P7S_05_D_Prefix",
  "family": "d",
  "method": "prefix",
  "prepared_identity": "ba4eee96562ffb1edc67476d092774de7e6a14da2f70f5b7af323cfaf304d45c",
  "feature_identity": "c0c28230f22d2a1dee3193df229358a5c4ef235ff240e9c920edd275112fe6e7",
  "encoder_checkpoint_sha256": "c9c0c0d0dcd1e54721586b8b1b4c2af79984268da9c5802a4092212b747e3b00",
  "encoder_root": "/kaggle/working/cs781_phase7_score/P7_05_D_Prefix",
  "auxiliary_ranker_sha256": "92dc6568f91af7a8d7196837ba1522c7ae5c36afb786f24d51bf587cbfbb595b",
  "rule": {
    "style": "learned",
    "neural_weight": 0.1,
    "constant": 20,
    "tune": {
      "micro_f1_at7": 0.4895330112721417,
      "micro_precision_at7": 0.4935064935064935,
      "micro_recall_at7": 0.48562300319488816,
      "tp": 304,
      "fp": 312,
      "fn": 322,
      "queries": 88
    }
  },
  "trials": [
    {
      "style": "baseline",
      "neural_weight": 0.5,
      "con

In [155]:
# [Code Cell]
from IPython.display import FileLink, display
score_report = read_json(Path(OUTPUT_ROOT) / 'score_boost/score_report.json')
assert score_report['status'] == 'submission_verified'
print('Baseline dev:', score_report['baseline_official_dev']['micro_f1_at7'])
print('Enhanced dev:', score_report['official_dev']['micro_f1_at7'])
print('Tune:', score_report['tune']['micro_f1_at7'])
print('Confirmation:', score_report['confirmation']['micro_f1_at7'])
print('Own neural score weight:', score_report['learned_neural_contribution'])
display(FileLink(score_report['zip_path']))
display(FileLink(str(Path(OUTPUT_ROOT) / 'score_boost/score_observations.zip')))
print('Submit the P7S ZIP shown here. Save COMPLETE output including original neural weights, code, tokenizer, caches and score_boost/.')


Baseline dev: 0.2623679822123402
Enhanced dev: 0.47026125625347415
Tune: 0.4895330112721417
Confirmation: 0.4272890484739677
Own neural score weight: 0.1


/kaggle/working/cs781_phase7_score/P7_05_D_Prefix/score_boost/submission_P7S_05_D_Prefix.zip

/kaggle/working/cs781_phase7_score/P7_05_D_Prefix/score_boost/score_observations.zip

Submit the P7S ZIP shown here. Save COMPLETE output including original neural weights, code, tokenizer, caches and score_boost/.
